<a href="https://colab.research.google.com/github/Neoweikang/cvdl_10/blob/main/notebooks/prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1 · Handwriting OCR and translation

Choose **restore** or **fresh** in Cell 3, then **Run all** on a GPU runtime.

- **Restore** uses your verified completed backup. It runs setup and model-load checks, but skips training datasets, training preflight and training.
- **Fresh** starts from the original pretrained foundations, trains on compatible public + curated training lines and saves a new verified Drive backup automatically.
- Fresh mode includes the Kaggle handwriting sources and the Chinese Essays `train` split stored on Drive. Cell 14 reports the final eligible count for each addition.
- Evaluation is optional and separately controlled in Cell 3. Final test remains locked. Restoring weights is not resuming an interrupted optimizer.

The default saved path is the backup you verified on 8 November. Your original notebook and backup are not modified. New session outputs get a separate run ID. Training progress is shown per batch; cells are numbered by their actual positions.


In [1]:
# Cell 2 · Install and verify dependencies

import sys
import subprocess
import time

started = time.perf_counter()

_loaded_transformers = sys.modules.get('transformers')
if _loaded_transformers is not None and _loaded_transformers.__version__ != '5.17.0':
    raise RuntimeError('Restart the Colab session before dependency setup: a different Transformers version is already imported.')

# Retain Colab's installed PyTorch and CUDA.
# Install the Transformers version used by this notebook.
subprocess.check_call([
    sys.executable,
    "-m", "pip", "install", "--quiet",
    "transformers==5.17.0",
    "sentencepiece==0.2.1",
    "sacremoses==0.1.1",
    "datasets==4.8.5",
    "sacrebleu==2.6.0",
    "recognizers-text-number==1.0.2a2",
    "emoji==1.7.0",
    "pyspellchecker==0.8.3",
    "jieba==0.42.1",
    "opencv-python-headless==4.11.0.86",
])

# Verify the recognition APIs after installation before loading model weights.
import torch
import transformers
from transformers import (
    RobertaTokenizer,
    ViTImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    AutoModelForTextRecognition,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Selected device:", DEVICE)

# Check GPU memory and execute a small tensor operation to verify CUDA access.
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {memory_gb:.1f} GB")
    print("GPU tensor check:", torch.ones(1, device=DEVICE).item())
else:
    print("GPU unavailable; model inference and training may be slow.")

print("OCR imports verified.")
print(f"Dependency setup completed in {time.perf_counter() - started:.1f} seconds.")


Transformers: 5.17.0
PyTorch: 2.14.1+cu130
Selected device: cuda
GPU: NVIDIA L4
GPU memory: 22.0 GB
GPU tensor check: 1.0
OCR imports verified.
Dependency setup completed in 36.7 seconds.


In [2]:
# Cell 3 · Session mode and project settings
from pathlib import Path
from google.colab import drive
from datetime import datetime, timezone
from uuid import uuid4

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/handwritten_translation_project')
SESSION_MODE = 'fresh'  # @param ['restore', 'fresh']
SAVED_RUN = str(PROJECT_DIR / 'saved_training_runs' / '20261108T053607Z_9f86fd19_20261108_075632_824811') # @param {type:'string'}
AUTO_SAVE_FRESH = True
# Evaluation is separate from restoring models for the demo.
RUN_OCR_EVALUATION = False
RUN_CURATED_VALIDATION = False
RUN_FINAL_TEST = False  # Unlock only after freezing the evaluation plan.
SHOW_DEMO = True
if SESSION_MODE not in ('fresh', 'restore'):
    raise ValueError('Choose fresh or restore.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
CHECKPOINT_DIR = Path('/content/ocr_sessions') / RUN_ID
GROUP_DATA_DIR = PROJECT_DIR / 'data' / 'group'
PUBLIC_DATA_DIR = PROJECT_DIR / 'data' / 'public'
RESULTS_DIR = PROJECT_DIR / 'results'
for folder in (CHECKPOINT_DIR, GROUP_DATA_DIR, PUBLIC_DATA_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
TRAINED_READY = set()
DATA_PROVENANCE = {}
RESTORED_RECORDS = {}
MODEL_RUN_LABEL = 'fresh: ' + RUN_ID
RUN_PUBLIC_SUBSETS = SESSION_MODE == 'fresh'
RUN_CURATED_AUDIT = SESSION_MODE == 'fresh' or RUN_CURATED_VALIDATION or RUN_FINAL_TEST
RUN_OCR_COHORTS = RUN_OCR_EVALUATION
RUN_OCR_INTERVALS = RUN_OCR_EVALUATION
PUBLIC_DATASETS = {'English': 'Teklia/IAM-line', 'Chinese': 'Teklia/CASIA-HWDB2-line'}
# Clear cached predictions whenever starting either session mode.
for name in list(globals()):
    if name.startswith(('_PAGE_', '_SIMPLE_')):
        globals().pop(name, None)
print('Mode:', SESSION_MODE, '| Session:', RUN_ID)
print('Saved selection:', SAVED_RUN if SESSION_MODE == 'restore' else '(none)')


Mounted at /content/drive
Mode: fresh | Session: 20261109T033914Z_187fd23d
Saved selection: (none)


In [3]:
# Cell 4 · Shared OCR metrics and augmentation

import types
workflow = types.ModuleType('ocr_workflow')
exec(r'''
"""Shared corpus metrics and training-only augmentation."""
import argparse
import hashlib
import json
import random
import unicodedata
from pathlib import Path

METRIC_POLICY = 'ocr-v2-raw-nfkc-punctuation'
# Explicit equivalences; preserve punctuation rather than deleting it.
PUNCTUATION = str.maketrans({
    '。': '.', '，': ',', '、': ',', '；': ';', '：': ':',
    '！': '!', '？': '?', '“': '"', '”': '"', '‘': "'", '’': "'",
    '「': '"', '」': '"', '『': '"', '』': '"',
    '【': '[', '】': ']', '—': '-', '–': '-', '…': '...',
})
AUGMENTATION = dict(enabled=True, probability=0.7, rotation_degrees=1.5,
                    brightness=(0.9, 1.1), contrast=(0.85, 1.15),
                    blur_probability=0.2, blur_radius=(0.1, 0.5), seed=42)


def normalize_punctuation(text):
    """Evaluation only: NFKC then the explicit punctuation table above.

    Preserve case and whitespace; do not convert simplified/traditional Chinese.
    """
    return unicodedata.normalize('NFKC', text).translate(PUNCTUATION)


def distance(a, b):
    previous = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        current = [i]
        for j, y in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (x != y)))
        previous = current
    return previous[-1]


def error_rates(references, predictions, lang):
    references, predictions = list(references), list(predictions)
    if lang not in ('en', 'zh'):
        raise ValueError('language must be en or zh')
    if not references or len(references) != len(predictions):
        raise ValueError('Need nonempty, equally sized reference/prediction lists')
    if not all(isinstance(x, str) for x in references + predictions):
        raise TypeError('Every reference and prediction must be a string')

    def rate(refs, preds):
        total = sum(map(len, refs))
        # Undefined when all reference strings/word lists are empty.
        return sum(distance(r, p) for r, p in zip(refs, preds)) / total if total else None

    nfkc = lambda xs: [unicodedata.normalize('NFKC', x) for x in xs]
    normal = lambda xs: [normalize_punctuation(x) for x in xs]
    return dict(CER=rate(references, predictions),
                CER_NFKC=rate(nfkc(references), nfkc(predictions)),
                CER_PUNCT=rate(normal(references), normal(predictions)),
                WER=rate([x.split() for x in references],
                         [x.split() for x in predictions]) if lang == 'en' else None,
                exact_match=sum(r == p for r, p in zip(references, predictions)) / len(references))


def evaluate_rows(rows, predict, lang):
    """One scorer for all model versions and both held-out splits.

    predict(row) returns untouched decoded text. Never augment evaluation rows.
    """
    rows = list(rows)
    refs = [r['text'] for r in rows]
    preds = [predict(r) for r in rows]
    return dict(metric_policy=METRIC_POLICY, lines=len(rows),
                references=refs, predictions=preds,
                **error_rates(refs, preds, lang))


def augment_line(image, sample_key, epoch=0, config=None):
    """Mild image-only changes, deterministic by sample/epoch; labels unchanged.

    Rotation expands the canvas to avoid clipping end characters. No flips,
    random crops, or erasing. Pillow is imported lazily for a dependency-free CLI.
    """
    from PIL import Image, ImageEnhance, ImageFilter
    cfg = AUGMENTATION if config is None else config
    image = image.convert('RGB')
    seed = hashlib.sha256(f"{cfg['seed']}:{epoch}:{sample_key}".encode()).digest()
    rng = random.Random(int.from_bytes(seed[:8], 'big'))
    if not cfg['enabled'] or rng.random() >= cfg['probability']:
        return image
    image = image.rotate(rng.uniform(-cfg['rotation_degrees'], cfg['rotation_degrees']),
                         resample=Image.Resampling.BICUBIC, expand=True,
                         fillcolor=(255, 255, 255))
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(*cfg['brightness']))
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(*cfg['contrast']))
    if rng.random() < cfg['blur_probability']:
        image = image.filter(ImageFilter.GaussianBlur(rng.uniform(*cfg['blur_radius'])))
    return image



''', workflow.__dict__)
error_rates = workflow.error_rates
evaluate_rows = workflow.evaluate_rows
augment_line = workflow.augment_line
AUGMENTATION = dict(workflow.AUGMENTATION)
# No historical fine-tuned weights are referenced.
print('Shared OCR metrics and training augmentation ready.')


Shared OCR metrics and training augmentation ready.


In [4]:
# Cell 5 · OCR checkpoint loading and inference

import gc, json, time, unicodedata
from pathlib import Path
from contextlib import nullcontext

from PIL import Image

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))

import torch
import pandas as pd
from huggingface_hub import hf_hub_download, HfApi
from transformers import (
    AutoImageProcessor,
    AutoModelForTextRecognition,
    RobertaTokenizer,
    TrOCRProcessor,
    ViTImageProcessor,
    VisionEncoderDecoderModel,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Select a Colab GPU runtime before loading/training full datasets.')
BASE = {
    'en': 'microsoft/trocr-base-handwritten',
    'zh': 'PaddlePaddle/PP-OCRv5_server_rec_safetensors',
}
BASE_REVISIONS = ({lang: HfApi().model_info(repo).sha for lang, repo in BASE.items()}
                  if SESSION_MODE == 'fresh' else {})
PUBLIC = {lang: CHECKPOINT_DIR / lang / 'weights' for lang in BASE}
ADAPTED = {lang: CHECKPOINT_DIR / 'unused' / lang for lang in BASE}
MAX_NEW_TOKENS = 128

_OCR = globals().get('_OCR', None)

def release_ocr():
    global _OCR
    if _OCR is not None:
        _OCR[1].cpu()
    _OCR = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


release_ocr()


def has_weights(path):
    return (path / 'config.json').is_file() and (
        any(path.glob('*.safetensors'))
        or any(path.glob('pytorch_model*.bin'))
    )


def model_sources(lang):
    sources = {'baseline': BASE[lang]}
    if lang in TRAINED_READY and has_weights(PUBLIC[lang]):
        sources['public'] = str(PUBLIC[lang])
    return sources


def load_ocr(lang, version):
    global _OCR
    source = model_sources(lang).get(version)
    if source is None:
        raise FileNotFoundError(
            f'{lang}/{version}: no saved checkpoint is available.'
        )

    if _OCR is not None and _OCR[0] == (lang, source):
        return _OCR[1], _OCR[2]

    release_ocr()

    revision = BASE_REVISIONS[lang] if source == BASE[lang] else None
    kwargs = {'revision': revision} if revision else {}
    if lang == 'en':
        def tokenizer_file(name):
            local = Path(source) / name
            return (
                str(local) if local.is_file()
                else hf_hub_download(BASE[lang], name, revision=BASE_REVISIONS[lang])
            )

        tokenizer = RobertaTokenizer(
            vocab=tokenizer_file('vocab.json'),
            merges=tokenizer_file('merges.txt'),
            model_max_length=512,
        )
        processor = TrOCRProcessor(
            ViTImageProcessor.from_pretrained(source, **kwargs), tokenizer
        )
        model = VisionEncoderDecoderModel.from_pretrained(
            source, dtype=torch.float32, **kwargs
        )
        model.config.decoder_start_token_id = tokenizer.eos_token_id
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.vocab_size = model.config.decoder.vocab_size
    else:
        processor = AutoImageProcessor.from_pretrained(source, **kwargs)
        model = AutoModelForTextRecognition.from_pretrained(
            source, dtype=torch.float32, **kwargs
        )

    model.to(DEVICE).eval()
    _OCR = ((lang, source), model, processor)
    return model, processor


def open_line(row):
    with Image.open(row['local_path']) as image:
        return image.convert('RGB')


@torch.inference_mode()
def transcribe(image, lang, model, processor):
    model.eval()
    inputs = processor(
        images=image.convert('RGB'), return_tensors='pt'
    ).to(DEVICE)

    if lang == 'en':
        tokens = model.generate(
            inputs['pixel_values'],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            decoder_start_token_id=processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
        return processor.batch_decode(
            tokens, skip_special_tokens=True
        )[0].strip()

    output = model(**inputs)
    return processor.post_process_text_recognition(
        output
    )[0]['text'].strip()


# Use the same scorer as public/final evaluation; do not redefine it here.
if 'workflow' not in globals():
    raise RuntimeError('Run shared OCR helpers before OCR loading helpers.')
distance = workflow.distance
error_rates = workflow.error_rates


def checkpoint_status(lang, version):
    if version == 'baseline':
        return 'pretrained baseline'
    if version == 'public':
        return MODEL_RUN_LABEL + '; saved training data, not evidence of improvement'
    marker = ADAPTED[lang] / 'selected.json'
    if not marker.is_file():
        return 'unavailable'
    selected = read_json(marker)
    return ('validation-selected' if selected.get('validation_metrics')
            and selected.get('selection') == 'validation_CER' else 'UNVALIDATED')


def curated_rows(split, lang):
    if not CURATED_READY:
        raise RuntimeError('Run curated data audit successfully first.')

    return sorted(
        (
            r for r in CURATED_ROWS
            if r['split'] == split and r['language'] == lang
        ),
        key=lambda r: r['sample_id'],
    )


print(
    'Available checkpoints:',
    {lang: list(model_sources(lang)) for lang in BASE},
)


Available checkpoints: {'en': ['baseline'], 'zh': ['baseline']}


In [5]:
# Cell 6 · Saved-run validation helpers
import hashlib, json, shutil
from copy import deepcopy

def session_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def session_files(root):
    root = Path(root)
    files = {}
    for path in sorted(root.rglob('*')):
        if path.is_symlink():
            raise ValueError(f'Symlinks are not accepted in model archives: {path}')
        if path.is_file() and path.name != 'backup_manifest.json':
            files[path.relative_to(root).as_posix()] = path
    return files


def check_weight_files(folder):
    config = read_json(folder / 'config.json')
    indices = list(folder.glob('*.index.json'))
    if indices:
        for index in indices:
            shards = set(read_json(index).get('weight_map', {}).values())
            if not shards:
                raise ValueError(f'Empty weight index: {index}')
            for relative in shards:
                shard = folder / relative
                if not shard.resolve().is_relative_to(folder.resolve()) or not shard.is_file() or not shard.stat().st_size:
                    raise ValueError(f'Missing/unsafe model shard: {relative}')
    elif not any(p.is_file() and p.stat().st_size for p in [folder/'model.safetensors', folder/'pytorch_model.bin']):
        raise ValueError(f'Missing complete model weights: {folder}')
    return config


def inspect_saved_run(folder):
    folder = Path(folder)
    records = {}
    for lang in ('en', 'zh'):
        record = read_json(folder / lang / 'training.json')
        if record.get('status') != 'complete' or record.get('language') != lang:
            raise ValueError(f'{lang}: training is not recorded as complete.')
        if record.get('source') != BASE[lang] or not record.get('foundation_revision'):
            raise ValueError(f'{lang}: missing or incompatible pretrained foundation identity.')
        provenance = record.get('data_provenance', {})
        if not provenance.get('sources') or not provenance.get('curated_fingerprint'):
            raise ValueError(f'{lang}: missing saved training provenance.')
        check_weight_files(folder / lang / 'weights')
        weights = folder / lang / 'weights'
        if lang == 'en' and not (weights / 'preprocessor_config.json').is_file():
            if not read_json(weights / 'processor_config.json').get('image_processor'):
                raise ValueError('English saved image processor is missing.')
        if lang == 'zh' and not (weights / 'preprocessor_config.json').is_file() and not (weights / 'processor_config.json').is_file():
            raise ValueError('Chinese saved image processor is missing.')
        records[lang] = record
    files = session_files(folder)
    manifest_path = folder / 'backup_manifest.json'
    if manifest_path.is_file():
        manifest = read_json(manifest_path)
        if set(manifest['files']) != set(files):
            raise ValueError('Backup file list differs from its checksum manifest.')
        for relative, path in files.items():
            expected = manifest['files'][relative]
            if path.stat().st_size != expected['bytes'] or session_sha256(path) != expected['sha256']:
                raise ValueError(f'Backup checksum mismatch: {relative}')
        print('Saved checksum manifest verified.')
    else:
        print('Legacy backup: no saved checksum manifest. File structure is checked; original-save integrity cannot be re-proved.')
    return records, files


In [6]:
# Cell 7 · Restore the explicitly selected completed run
# Final weights only: this does not resume an interrupted optimizer.
def restore_completed_run():
    global PUBLIC, BASE_REVISIONS, DATA_PROVENANCE, RESTORED_RECORDS, MODEL_RUN_LABEL
    TRAINED_READY.clear()
    release_ocr()
    archive = Path(SAVED_RUN).expanduser().resolve()
    if not archive.is_dir():
        raise FileNotFoundError(f'Saved run not found: {archive}. Set SAVED_RUN in setup; no fallback will run.')
    records, files = inspect_saved_run(archive)
    destination = CHECKPOINT_DIR / 'restored'
    if destination.exists():
        raise FileExistsError('Restore was already attempted in this session. Rerun setup for a clean session ID.')
    print('Copying saved models to runtime disk:', destination, flush=True)
    shutil.copytree(archive, destination)
    for relative, original in files.items():
        copied = destination / relative
        if session_sha256(original) != session_sha256(copied):
            raise RuntimeError(f'Local model copy verification failed: {relative}')
    BASE_REVISIONS = {lang: records[lang]['foundation_revision'] for lang in ('en', 'zh')}
    DATA_PROVENANCE = {lang: deepcopy(records[lang]['data_provenance']) for lang in ('en', 'zh')}
    PUBLIC = {lang: destination / lang / 'weights' for lang in ('en', 'zh')}
    try:
        # Temporary readiness permits strict loader checks; failures clear it.
        TRAINED_READY.update(('en', 'zh'))
        for lang in ('en', 'zh'):
            model, processor = load_ocr(lang, 'public')
            smoke = transcribe(Image.new('RGB', (256, 64), 'white'), lang, model, processor)
            if not isinstance(smoke, str):
                raise TypeError('Restore inference smoke test returned non-text.')
            release_ocr()
            print(lang, 'loaded; blank-image inference smoke passed. This is not an accuracy test.', flush=True)
    except Exception:
        TRAINED_READY.clear()
        release_ocr()
        raise
    RESTORED_RECORDS = records
    MODEL_RUN_LABEL = f'restored: {archive.name}'
    for lang, provenance in DATA_PROVENANCE.items():
        print(lang, 'saved training lines:', provenance['public_used'], 'public +', provenance['curated_used'], 'curated')
    print('RESTORE COMPLETE:', archive)
    print('Dataset preparation/preflight/training skipped. Evaluation remains optional.')

if SESSION_MODE == 'restore':
    restore_completed_run()
else:
    print('Fresh mode: no saved trained weights loaded.')


Fresh mode: no saved trained weights loaded.


In [7]:
# Cell 8 · Curated data audit and local cache

import csv, hashlib, io, json, shutil
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image
from google.colab import drive



CURATED_ROOT = Path(
    '/content/drive/MyDrive/CVDL_Handwriting_Dataset/v1_writers'
)
WRITERS = {
    'W01': ('Kim', 'validation'),
    'W02': ('Reiner', 'test'),
    'W03': ('Yu Wei', 'test'),
    'W04': ('Wei Kang', 'train'),
    'W05': ('Wei En', 'test'),
}
FIELDS = (
    'sample_id', 'image_path', 'text', 'language', 'source_page_id',
    'capture_id', 'line_index', 'writer_id', 'writer_name',
)
SPLITS = ('train', 'validation', 'test')
CURATED_READY = False


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def fingerprint(rows):
    # Local paths are excluded. Adding test data does not alter the train ID.
    values = [
        {k: r[k] for k in (*FIELDS, 'split', 'sha256')}
        for r in rows
    ]
    values.sort(key=lambda r: (r['split'], r['sample_id']))
    return hashlib.sha256(
        json.dumps(
            values, ensure_ascii=False, sort_keys=True
        ).encode()
    ).hexdigest()


def capture_language_problems(cap):
    # Page language is a summary; every training crop remains en OR zh.
    cid, page_language = cap.get('capture_id'), cap.get('language')
    problems = []
    if page_language not in ('en', 'zh', 'mixed'):
        problems.append(f'Invalid capture language: {cid}, {page_language!r}')
    for line in cap.get('lines', []):
        sid, language = line.get('sample_id'), line.get('language')
        if language not in ('en', 'zh'):
            problems.append(f'Invalid line language: {sid}, {language!r}')
        elif page_language in ('en', 'zh') and language != page_language:
            problems.append(f'Capture/line mismatch: {sid}, language')
    # A mixed page may retain crops from just one language after approval.
    return problems


def prepare_curated(root):
    root = root.resolve()
    problems, rows, manifests, capture_lines = [], [], {}, {}
    indexes = {
        k: defaultdict(list)
        for k in (
            'sample_id', 'path', 'pixels',
            'writer_id', 'source_page_id', 'capture_id',
        )
    }

    meta = read_json(root / 'metadata/dataset.json')
    expected = {
        w: {'name': name, 'split': split}
        for w, (name, split) in WRITERS.items()
    }
    if (
        meta.get('writers') != expected
        or meta.get('collection_mode') != 'writer_assigned'
    ):
        problems.append(
            'metadata/dataset.json disagrees with the writer assignments.'
        )

    captures = read_json(root / 'captures.json')
    if not isinstance(captures, list):
        raise ValueError(
            'captures.json must be a list, as in the inspected dataset.'
        )

    capture_ids = set()
    for cap in captures:
        split = cap.get('split')
        if split == 'pending':
            continue

        cid, writer = cap.get('capture_id'), cap.get('writer_id')
        if cid in capture_ids or split not in SPLITS:
            problems.append(f'Duplicate capture or invalid split: {cid}')
        capture_ids.add(cid)

        if WRITERS.get(writer) != (cap.get('writer_name'), split):
            problems.append(f'Capture writer assignment mismatch: {cid}')

        try:
            record = read_json(
                root / split / 'records' / f'{cid}.json'
            )
            # Compare active fields; historical migration fields are ignored.
            for key in (
                'capture_id', 'source_page_id', 'split',
                'writer_id', 'writer_name', 'language', 'lines',
            ):
                if record.get(key) != cap.get(key):
                    problems.append(
                        f'Capture record/index mismatch: {cid}, {key}'
                    )
        except Exception as e:
            problems.append(f'Cannot read capture record {cid}: {e}')

        problems.extend(capture_language_problems(cap))

        for line in cap.get('lines', []):
            sid = line.get('sample_id')
            if sid in capture_lines:
                problems.append(
                    f'Duplicate sample in capture index: {sid}'
                )
            capture_lines[sid] = (split, line)

            for key in (
                'capture_id', 'source_page_id',
                'writer_id', 'writer_name',
            ):
                if line.get(key) != cap.get(key):
                    problems.append(
                        f'Capture/line mismatch: {sid}, {key}'
                    )

    for split in SPLITS:
        path = root / split / 'labels.csv'
        if not path.exists():
            print(f'{split}: manifest absent; split unavailable.')
            continue

        manifests[split] = path.read_bytes()
        reader = csv.DictReader(
            io.StringIO(manifests[split].decode('utf-8-sig'))
        )
        if not set(FIELDS).issubset(reader.fieldnames or []):
            problems.append(f'{split}: missing required CSV columns.')
            continue

        for line_no, r in enumerate(reader, 2):
            tag = f'{split}/labels.csv:{line_no}'
            if None in r or any(
                not isinstance(r.get(k), str) or not r[k].strip()
                for k in FIELDS
            ):
                problems.append(
                    f'{tag}: malformed row or empty field/label.'
                )
                continue

            r = {**r, 'split': split}
            rows.append(r)  # Preserve corrected text exactly.

            if WRITERS.get(r['writer_id']) != (r['writer_name'], split):
                problems.append(f'{tag}: writer assignment/name mismatch.')
            if (
                r['language'] not in ('en', 'zh')
                or not r['line_index'].isdigit()
            ):
                problems.append(f'{tag}: invalid language or line index.')

            saved = capture_lines.get(r['sample_id'])
            if (
                not saved
                or saved[0] != split
                or any(
                    str(saved[1].get(k, '')) != r[k]
                    for k in FIELDS
                )
            ):
                problems.append(f'{tag}: CSV differs from capture index.')

            for key in (
                'sample_id', 'writer_id', 'source_page_id', 'capture_id'
            ):
                indexes[key][r[key]].append(split)

            try:
                # Inspection confirmed dataset-root-relative paths.
                relative = Path(r['image_path'])
                source = (root / relative).resolve()
                expected_folder = (
                    root / split / 'images' / r['writer_id']
                )
                if (
                    relative.is_absolute()
                    or '..' in relative.parts
                    or not source.is_relative_to(expected_folder)
                ):
                    raise ValueError(
                        'image_path is outside its assigned image folder'
                    )

                raw = source.read_bytes()
                r['sha256'] = hashlib.sha256(raw).hexdigest()
                if (
                    saved
                    and saved[1].get('image_sha256') != r['sha256']
                ):
                    problems.append(
                        f'{tag}: image hash differs from saved record.'
                    )

                with Image.open(io.BytesIO(raw)) as im:
                    rgb = im.convert('RGB')
                    pixels = hashlib.sha256(
                        str(rgb.size).encode() + rgb.tobytes()
                    ).hexdigest()

                indexes['path'][str(source)].append(split)
                indexes['pixels'][pixels].append(split)

                if len(rows) % 100 == 0:
                    print(f'Checked {len(rows)} line images')

            except Exception as e:
                problems.append(
                    f'{tag}: missing/unreadable/invalid image: {e}'
                )

    if set(capture_lines) != {r['sample_id'] for r in rows}:
        problems.append(
            'Manifest sample IDs and approved capture-index sample IDs differ.'
        )

    for key, mapping in indexes.items():
        for value, splits in mapping.items():
            duplicate = (
                key in ('sample_id', 'path', 'pixels')
                and len(splits) > 1
            )
            if duplicate or len(set(splits)) > 1:
                problems.append(
                    f'Duplicate or cross-split {key}: '
                    f'{value}, {sorted(set(splits))}'
                )

    for split in SPLITS:
        subset = [r for r in rows if r['split'] == split]
        pages = len({r['source_page_id'] for r in subset})
        print(f'{split}: {len(subset)} lines, {pages} pages')
        print(
            '  Writer/language:',
            dict(Counter(
                (r['writer_id'], r['language']) for r in subset
            )),
        )

    if problems:
        print('\n'.join(problems[:50]))
        raise ValueError(
            f'{len(problems)} dataset problems. '
            'Source unchanged; training blocked.'
        )

    cache = Path('/content/cvdl_curated_cache') / fingerprint(rows)[:16]

    for split, raw in manifests.items():
        destination = cache / split / 'labels.csv'
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(raw)

    for i, r in enumerate(rows, 1):
        destination = cache / r['image_path']
        destination.parent.mkdir(parents=True, exist_ok=True)

        if (
            not destination.exists()
            or hashlib.sha256(
                destination.read_bytes()
            ).hexdigest() != r['sha256']
        ):
            shutil.copyfile(
                root / r['image_path'], destination
            )

        if hashlib.sha256(
            destination.read_bytes()
        ).hexdigest() != r['sha256']:
            raise ValueError(
                f'Copy changed during audit: {r["sample_id"]}; '
                'rerun this cell.'
            )

        r['local_path'] = str(destination)
        if i % 100 == 0:
            print(f'Cached {i}/{len(rows)} images')

    print(
        'Verified. Local cache:', cache,
        '| Pending excluded; source unchanged.'
    )
    return rows


CURATED_ROWS = []
if RUN_CURATED_AUDIT:
    CURATED_ROWS = prepare_curated(CURATED_ROOT)
    CURATED_READY = True
else:
    print('Curated audit skipped; empty curation does not block the demo.')


Checked 100 line images
Checked 200 line images
Checked 300 line images
Checked 400 line images
Checked 500 line images
Checked 600 line images
Checked 700 line images
train: 655 lines, 37 pages
  Writer/language: {('W04', 'en'): 608, ('W04', 'zh'): 47}
validation: 104 lines, 6 pages
  Writer/language: {('W01', 'zh'): 50, ('W01', 'en'): 54}
test: 32 lines, 2 pages
  Writer/language: {('W03', 'zh'): 16, ('W03', 'en'): 16}
Cached 100/791 images
Cached 200/791 images
Cached 300/791 images
Cached 400/791 images
Cached 500/791 images
Cached 600/791 images
Cached 700/791 images
Verified. Local cache: /content/cvdl_curated_cache/59cdbd50767109d9 | Pending excluded; source unchanged.


In [8]:
# Cell 9 · Public training data and audited curated lines
from datasets import load_dataset, load_from_disk
from huggingface_hub import HfApi
from torch.utils.data import Dataset, ConcatDataset
import hashlib

# None means ALL public train rows. Set an integer only for a labelled pilot.
PUBLIC_TRAIN_LIMIT = None
# Only compatible, license-reviewed line-transcription sources belong here.
# Research additions can be enabled without changing training code.
PUBLIC_SOURCES = {
    'en': [{'id': 'Teklia/IAM-line', 'split': 'train'},
           {'id': 'radhakrishnadeshpande/gnhk-pagelevel', 'split': 'train', 'adapter': 'gnhk'},
           {'id': 'CATMuS/modern', 'split': 'train', 'adapter': 'catmus'}],
    'zh': [{'id': 'Teklia/CASIA-HWDB2-line', 'split': 'train'}],
}
# Synthetic data is not part of the natural-handwriting count or default mix.
INCLUDE_SYNTHETIC_PENPAL = False
if INCLUDE_SYNTHETIC_PENPAL:
    PUBLIC_SOURCES['en'].append({'id': 'breitburg/penpal', 'split': 'train', 'adapter': 'penpal'})
# The prepared releases use the same RGB image/text interface as the Hub sources.
if SESSION_MODE == 'fresh':
    bundle_folder = 'kaggle_ocr_v1'
    manifest = read_json(PUBLIC_DATA_DIR / bundle_folder / 'sources.json')
    for entry in manifest['sources']:
        PUBLIC_SOURCES[entry['language']].append(dict(
            id=entry['id'], split='train', drive_train=f'{bundle_folder}/train',
            start=entry['first_index'], count=entry['prepared_rows'],
            revision=f"kaggle-v{entry['kaggle_version']}:{entry['selection_sha256']}"))
    PUBLIC_SOURCES['zh'].append(dict(
        id='luyihang/handwritten-data-set-of-chinese-essays', split='train',
        drive_train='chinese_essays_grid_v1/train', count=2845, revision='kaggle-v1-grid-v1'))

PUBLIC_SUBSETS = {}
# DATA_PROVENANCE is initialized in setup and populated by restore or fresh loading.

class CombinedLines(Dataset):
    def __init__(self, public, curated, language):
        self.public, self.curated, self.language = public, curated, language
    def __len__(self):
        return len(self.public) + len(self.curated)
    def __getitem__(self, index):
        if index < len(self.public):
            row = self.public[index]
            return {'image': row['image'], 'text': row['text']}
        row = self.curated[index - len(self.public)]
        return {'image': open_line(row), 'text': row['text']}

def image_digest(image):
    rgb = image.convert('RGB')
    return hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()

class GNHKLines(Dataset):
    """Original annotations only: keep complete handwritten, readable lines."""
    def __init__(self, pages):
        import json
        from collections import defaultdict, Counter
        self.pages, self.lines = pages, []
        excluded = Counter()
        for page_index in range(len(pages)):
            page = pages[page_index]
            annotations = page['annotations']
            words = json.loads(annotations) if isinstance(annotations, str) else annotations
            if not isinstance(words, list):
                raise ValueError('Unexpected GNHK annotation schema.')
            grouped = defaultdict(list)
            for word in words:
                grouped[word['line_idx']].append(word)
            for line_id, group in grouped.items():
                if any(w.get('type') != 'H' for w in group):
                    excluded['not_fully_handwritten'] += 1; continue
                if any(not isinstance(w.get('text'), str) or not w['text'].strip() or
                       '%' in w['text'] for w in group):
                    excluded['placeholder_or_blank'] += 1; continue
                def coords(word):
                    p = word['polygon']
                    return [(float(p[f'x{i}']), float(p[f'y{i}'])) for i in range(4)]
                group = sorted(group, key=lambda w: min(x for x, y in coords(w)))
                text = ' '.join(w['text'].strip() for w in group)
                points = [point for w in group for point in coords(w)]
                import math
                box = (math.floor(min(x for x,y in points)), math.floor(min(y for x,y in points)),
                       math.ceil(max(x for x,y in points)), math.ceil(max(y for x,y in points)))
                w,h = page['image'].size
                box = (max(0,box[0]),max(0,box[1]),min(w,box[2]),min(h,box[3]))
                if box[2] <= box[0] or box[3] <= box[1]:
                    raise ValueError('Invalid GNHK line box.')
                self.lines.append((page_index, box, text))
        print('GNHK training pages:', len(pages), 'usable lines:', len(self.lines), 'excluded:', dict(excluded))
    def __len__(self):
        return len(self.lines)
    def __getitem__(self, index):
        page, box, text = self.lines[index]
        return {'image': self.pages[page]['image'].crop(box), 'text': text}
    def select(self, indices):
        result = object.__new__(GNHKLines)
        result.pages = self.pages
        result.lines = [self.lines[i] for i in indices]
        return result
    def shuffle(self, seed):
        import random
        indices = list(range(len(self))); random.Random(seed).shuffle(indices)
        return self.select(indices)

def load_training_data():
    if not CURATED_READY:
        raise RuntimeError('Curated audit must pass before any training.')
    # Exact pixel overlap is checked; this does not prove writer disjointness
    # for public sources without writer IDs (notably the CASIA mirror).
    heldout = {image_digest(open_line(r)) for r in CURATED_ROWS if r['split'] != 'train'}
    curated_train = {image_digest(open_line(r)): r['text'] for r in CURATED_ROWS if r['split'] == 'train'}
    shared_seen = {}
    drive_datasets = {}
    for lang, name in [('en', 'English'), ('zh', 'Chinese')]:
        own = curated_rows('train', lang)
        if not own:
            raise ValueError(f'Missing curated {lang} training lines; no silent public-only run.')
        datasets, records, seen = [], [], shared_seen
        for spec in PUBLIC_SOURCES[lang]:
            if spec['split'] != 'train':
                raise ValueError('Only explicitly audited train splits may enter training.')
            dataset_id = spec['id']
            if 'drive_train' in spec:
                relative = spec['drive_train']
                if relative not in drive_datasets:
                    drive_datasets[relative] = load_from_disk(str(PUBLIC_DATA_DIR / relative))
                public = drive_datasets[relative]
                if 'start' in spec:
                    public = public.select(range(spec['start'], spec['start'] + spec['count']))
                    if public.unique('source') != [dataset_id] or public.unique('language') != [lang]:
                        raise ValueError(f'Prepared source membership differs for {dataset_id}.')
                if len(public) != spec['count']:
                    raise ValueError(f'Expected {spec["count"]} prepared lines for {dataset_id}, found {len(public)}.')
                revision = f"{spec['revision']}@{public._fingerprint}"
            else:
                revision = spec.get('revision') or HfApi().dataset_info(dataset_id).sha
                public = load_dataset(dataset_id, revision=revision, split=spec['split'])
            if spec.get('adapter') == 'catmus':
                public = public.filter(lambda row: row['language'] == 'eng' and row['writing_type'] == 'handwritten')
                public = public.rename_column('im', 'image')
            elif spec.get('adapter') == 'penpal':
                public = public.rename_column('preview', 'image')
            if spec.get('adapter') == 'gnhk':
                public = GNHKLines(public)
            total = len(public)
            if PUBLIC_TRAIN_LIMIT is not None:
                if PUBLIC_TRAIN_LIMIT < 1:
                    raise ValueError('PUBLIC_TRAIN_LIMIT must be positive or None.')
                public = public.shuffle(seed=42).select(range(min(PUBLIC_TRAIN_LIMIT, total)))
            keep, duplicates = [], 0
            for i, row in enumerate(public):
                if not isinstance(row['text'], str) or not row['text'].strip():
                    raise ValueError(f'{dataset_id} row {i}: missing transcription.')
                digest = image_digest(row['image'])
                if digest in heldout:
                    raise ValueError(f'{dataset_id} row {i} overlaps curated holdout; training blocked.')
                if digest in seen or digest in curated_train:
                    previous = seen.get(digest, curated_train.get(digest))
                    if previous != row['text']:
                        raise ValueError(f'{dataset_id} row {i}: duplicate pixels with conflicting labels.')
                    duplicates += 1
                else:
                    seen[digest] = row['text']; keep.append(i)
                if (i + 1) % 10000 == 0:
                    print(dataset_id, 'audited', i + 1, '/', len(public), flush=True)
            public = public.select(keep)
            datasets.append(public)
            records.append(dict(dataset=dataset_id, revision=revision, split=spec['split'],
                available_public=total, public_used=len(public), duplicate_public_excluded=duplicates,
                synthetic=spec.get('adapter') == 'penpal', drive_train=spec.get('drive_train'),
                new_source=spec.get('adapter') is not None or 'drive_train' in spec))
        public = ConcatDataset(datasets)
        if not len(public):
            raise ValueError(f'{lang}: public training set is empty.')
        PUBLIC_SUBSETS[name] = {'train': CombinedLines(public, own, lang)}
        DATA_PROVENANCE[lang] = dict(sources=records,
            public_used=len(public), curated_used=len(own), curated_fingerprint=fingerprint(own),
            curated_sample_ids=[r['sample_id'] for r in own],
            curated_writers=sorted({r['writer_id'] for r in own}),
            scope='full public train' if PUBLIC_TRAIN_LIMIT is None else 'explicit pilot subset')
        print(name, DATA_PROVENANCE[lang], flush=True)
    public_total = sum(r['public_used'] for r in DATA_PROVENANCE.values())
    own_total = sum(r['curated_used'] for r in DATA_PROVENANCE.values())
    new_natural = sum(s['public_used'] for r in DATA_PROVENANCE.values() for s in r['sources'] if s['new_source'] and not s['synthetic'])
    synthetic = sum(s['public_used'] for r in DATA_PROVENANCE.values() for s in r['sources'] if s['synthetic'])
    print(f'Genuinely new natural-source lines: {new_natural}; synthetic lines separately: {synthetic}')
    print(f'Unique training lines: {public_total} public + {own_total} curated = {public_total + own_total}')
    print('No public validation/test rows are used for training. Curated held-out images audited only.')

if SESSION_MODE == 'fresh':
    load_training_data()
else:
    print('Restore: public training datasets are not downloaded.')


README.md:   0%|          | 0.00/2.14k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  167MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/validation.parquet: reconstructing file:   0%|          |  0.00B / 24.7MB            

data/validation.parquet: downloading bytes:           |  0.00B            

data/test.parquet: reconstructing file:   0%|          |  0.00B / 73.6MB            

... (451 outputs left out)

Generating train split:   0%|          | 0/33401 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8318 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10441 [00:00<?, ? examples/s]

Teklia/CASIA-HWDB2-line audited 10000 / 33401
Teklia/CASIA-HWDB2-line audited 20000 / 33401
Teklia/CASIA-HWDB2-line audited 30000 / 33401
Chinese {'sources': [{'dataset': 'Teklia/CASIA-HWDB2-line', 'revision': '2974a15ceab0657047a28d2a71b76bd7544d4663', 'split': 'train', 'available_public': 33401, 'public_used': 33401, 'duplicate_public_excluded': 0, 'synthetic': False, 'drive_train': None, 'new_source': False}, {'dataset': 'appenlimited/chinese-handwritten-ocr-image-data', 'revision': 'kaggle-v1:2d1dcf6ec663ac9da075d91c4dbfa14845e7e5b30a59863af7a9dddb0e175454@f5c1132413f70ffe', 'split': 'train', 'available_public': 106, 'public_used': 106, 'duplicate_public_excluded': 0, 'synthetic': False, 'drive_train': 'kaggle_ocr_v1/train', 'new_source': True}, {'dataset': 'luyihang/handwritten-data-set-of-chinese-essays', 'revision': 'kaggle-v1-grid-v1@7efb3aac56f89d30', 'split': 'train', 'available_public': 2845, 'public_used': 2845, 'duplicate_public_excluded': 0, 'synthetic': False, 'drive_tra

In [9]:
# Cell 10 · Public training settings and shared helpers

import gc
import hashlib
import json
import re
import time
from contextlib import nullcontext
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset

RUN_ENGLISH_PUBLIC_TRAINING = True
RUN_CHINESE_PUBLIC_TRAINING = True
PUBLIC_TRAIN_RUN_ID = RUN_ID
PUBLIC_TRAIN_SEED = 42
PUBLIC_TRAIN_EPOCHS = 1
PUBLIC_TRAIN_BATCH_SIZE = 4
PUBLIC_ENGLISH_LR = 5e-6
PUBLIC_CHINESE_LR = 1e-6
PUBLIC_AUGMENT_LANGUAGES = {'zh'}
PUBLIC_CTC_OUTPUT_KIND = 'probabilities'  # PP-OCR output; use 'logits' only for a verified model change.


def public_training_folder(lang):
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', PUBLIC_TRAIN_RUN_ID):
        raise ValueError('Use a simple, nonempty PUBLIC_TRAIN_RUN_ID.')
    if PUBLIC_TRAIN_EPOCHS < 1 or PUBLIC_TRAIN_BATCH_SIZE < 1:
        raise ValueError('Epochs and batch size must be positive.')
    if 'PUBLIC_SUBSETS' not in globals():
        raise RuntimeError('Load the opt-in public training subsets first.')
    name = {'en': 'English', 'zh': 'Chinese'}[lang]
    if not len(PUBLIC_SUBSETS[name]['train']):
        raise ValueError(f'No public training rows for {name}.')
    folder = Path(CHECKPOINT_DIR) / lang
    folder.mkdir(parents=True, exist_ok=False)
    return folder


def public_write_record(folder, record):
    record['foundation_revision'] = BASE_REVISIONS[record['language']]
    record['data_provenance'] = DATA_PROVENANCE[record['language']]
    path = folder / 'training.json'
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def public_images(rows, lang, epoch):
    images = [row['image'].convert('RGB') for row in rows]
    if lang in PUBLIC_AUGMENT_LANGUAGES and AUGMENTATION['enabled']:
        images = [workflow.augment_line(image,
                    sample_key=hashlib.sha256(str(image.size).encode() + image.tobytes() + row['text'].encode()).hexdigest(),
                    epoch=epoch, config=AUGMENTATION)
                  for image, row in zip(images, rows)]
    return images


def public_record(lang, dataset):
    return dict(status='running', language=lang, source=BASE[lang],
                dataset=globals().get('PUBLIC_DATASETS', {}).get({'en': 'English', 'zh': 'Chinese'}[lang]),
                subset_fingerprint=getattr(dataset, '_fingerprint', None),
                available_training_lines=len(dataset), seed=PUBLIC_TRAIN_SEED,
                epochs=PUBLIC_TRAIN_EPOCHS, batch_size=PUBLIC_TRAIN_BATCH_SIZE,
                augmentation=dict(AUGMENTATION) if lang in PUBLIC_AUGMENT_LANGUAGES else {'enabled': False},
                metric_policy=workflow.METRIC_POLICY,
                selection='fixed training run; not validation-selected')


def public_save_model(lang, model, processor, folder, record):
    temporary = folder / 'weights.pending'
    destination = folder / 'weights'
    model.eval()
    model.save_pretrained(temporary)
    processor.save_pretrained(temporary)
    temporary.rename(destination)
    record.update(status='complete', checkpoint=str(destination))
    public_write_record(folder, record)
    PUBLIC[lang] = destination
    TRAINED_READY.add(lang)
    print(f"Saved {lang} public checkpoint: {destination}")
    print('Runtime weights ready; the automatic backup cell preserves this completed run.')
    print('Run the larger paired validation before comparing recognition quality.')
    return record


def public_clear_model():
    if callable(globals().get('release_routed_mt')):
        release_routed_mt()
    release_ocr()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

from tqdm.auto import tqdm


In [10]:
# Cell 11 · English public training

def prepare_english_batch(rows, processor, epoch=0):
    texts = [row['text'] for row in rows]
    if any(not isinstance(text, str) or not text for text in texts):
        raise ValueError('English labels must be nonempty strings.')
    tokenizer = processor.tokenizer
    tokens = tokenizer(texts, padding=True, truncation=False, return_tensors='pt')
    if not torch.all(tokens.input_ids[:, 0] == tokenizer.bos_token_id).item():
        raise ValueError('Expected a leading RoBERTa BOS token in every target.')
    # Decoder supplies the start token; labels retain EOS and ignore padding.
    labels = tokens.input_ids[:, 1:].clone()
    labels[tokens.attention_mask[:, 1:] == 0] = -100
    max_positions = getattr(tokenizer, 'model_max_length', 512)
    if labels.shape[1] > min(max_positions, 512):
        raise ValueError('A label exceeds the supported decoder length; do not silently truncate it.')
    pixels = processor(images=public_images(rows, 'en', epoch), return_tensors='pt')['pixel_values']
    return {'pixel_values': pixels, 'labels': labels}


def train_english_public():
    folder = public_training_folder('en')
    dataset = PUBLIC_SUBSETS['English']['train']
    record = public_record('en', dataset)
    record['learning_rate'] = PUBLIC_ENGLISH_LR
    public_write_record(folder, record)
    started = time.perf_counter()
    try:
        public_clear_model()
        torch.manual_seed(PUBLIC_TRAIN_SEED)
        model, processor = load_ocr('en', 'baseline')
        tokenizer = processor.tokenizer
        check_ids = tokenizer.encode('Hello world')
        if check_ids != [0, 31414, 232, 2]:
            raise ValueError(f'Unexpected RoBERTa tokenizer output: {check_ids}. Fix the loader first.')
        model.config.decoder_start_token_id = tokenizer.eos_token_id
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.vocab_size = model.config.decoder.vocab_size
        optimizer = torch.optim.AdamW(model.parameters(), lr=PUBLIC_ENGLISH_LR)
        use_bf16 = model.device.type == 'cuda' and torch.cuda.is_bf16_supported()
        loss_sum, steps = 0.0, 0
        for epoch in range(PUBLIC_TRAIN_EPOCHS):
            loader = DataLoader(dataset, batch_size=PUBLIC_TRAIN_BATCH_SIZE, shuffle=True,
                generator=torch.Generator().manual_seed(PUBLIC_TRAIN_SEED + epoch), num_workers=0,
                collate_fn=lambda rows: prepare_english_batch(rows, processor, epoch))
            model.train()
            for batch in tqdm(loader, desc=f'English epoch {epoch + 1}/{PUBLIC_TRAIN_EPOCHS}', unit='batch'):
                optimizer.zero_grad(set_to_none=True)
                context = torch.autocast('cuda', dtype=torch.bfloat16) if use_bf16 else nullcontext()
                with context:
                    loss = model(**{key: value.to(model.device) for key, value in batch.items()}).loss
                if not torch.isfinite(loss).item():
                    raise RuntimeError('Non-finite English loss; no checkpoint selected.')
                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(norm).item():
                    raise RuntimeError('Non-finite English gradients; no checkpoint selected.')
                optimizer.step()
                steps += 1
                loss_sum += loss.detach().float().item()
            print(f'English epoch {epoch + 1}/{PUBLIC_TRAIN_EPOCHS}: loss {loss_sum / steps:.4f}')
        record.update(completed_steps=steps, average_training_loss=loss_sum / steps,
                      precision='bf16 autocast' if use_bf16 else 'float32',
                      elapsed_seconds=round(time.perf_counter() - started, 1))
        return public_save_model('en', model, processor, folder, record)
    except Exception as error:
        record.update(status='failed', error=str(error))
        public_write_record(folder, record)
        raise
    finally:
        public_clear_model()


In [11]:
# Cell 12 · Chinese vocabulary and CTC training

def public_chinese_vocabulary(processor):
    characters = processor.character_list
    if not characters or len(characters) < 2:
        raise ValueError('The Chinese processor has no usable CTC vocabulary.')
    return {character: index for index, character in enumerate(characters) if index != 0}


def collate_chinese(rows, processor, char_to_id, epoch=0):
    texts = [row['text'] for row in rows]
    labels = [[char_to_id[character] for character in text] for text in texts]
    pixels = processor(images=public_images(rows, 'zh', epoch), return_tensors='pt')['pixel_values']
    return pixels, labels


def public_ctc_log_probabilities(scores, vocabulary_size):
    if scores.ndim != 3 or scores.shape[-1] != vocabulary_size:
        raise ValueError('Expected PP-OCR scores shaped [batch, time, vocabulary].')
    scores = scores.float()
    if not torch.isfinite(scores).all().item():
        raise RuntimeError('Non-finite CTC model output.')
    if PUBLIC_CTC_OUTPUT_KIND == 'logits':
        return scores.log_softmax(-1)
    if PUBLIC_CTC_OUTPUT_KIND != 'probabilities':
        raise ValueError('PUBLIC_CTC_OUTPUT_KIND must be probabilities or logits.')
    sums = scores.sum(-1)
    if (scores.min().item() < 0 or scores.max().item() > 1.0001 or
        not torch.allclose(sums, torch.ones_like(sums), atol=1e-3, rtol=1e-3)):
        raise ValueError('PP-OCR output is not normalized probabilities; verify the model output contract.')
    return scores.clamp_min(1e-30).log()


def public_ctc_loss(scores, labels, vocabulary_size):
    if scores.shape[0] != len(labels):
        raise ValueError('CTC batch size differs from target count.')
    log_probs = public_ctc_log_probabilities(scores, vocabulary_size)
    for ids in labels:
        if not ids or any(index < 1 or index >= vocabulary_size for index in ids):
            raise ValueError('CTC targets must be nonempty and cannot contain blank or out-of-range IDs.')
    steps = scores.shape[1]
    minimum_steps = [len(ids) + sum(a == b for a, b in zip(ids, ids[1:])) for ids in labels]
    keep = [index for index, required in enumerate(minimum_steps) if required <= steps]
    skipped = len(labels) - len(keep)
    if not keep:
        return None, skipped
    kept = [labels[index] for index in keep]
    targets = torch.tensor([token for ids in kept for token in ids], dtype=torch.long, device=scores.device)
    target_lengths = torch.tensor([len(ids) for ids in kept], dtype=torch.long)
    input_lengths = torch.full((len(keep),), steps, dtype=torch.long)
    loss = torch.nn.functional.ctc_loss(log_probs[keep].transpose(0, 1).contiguous(),
        targets, input_lengths, target_lengths, blank=0, reduction='mean', zero_infinity=False)
    if not torch.isfinite(loss).item():
        raise RuntimeError('Non-finite CTC loss after alignment validation; no checkpoint selected.')
    return loss, skipped


def train_chinese_public():
    from collections import Counter
    folder = public_training_folder('zh')
    dataset = PUBLIC_SUBSETS['Chinese']['train']
    record = public_record('zh', dataset)
    record.update(learning_rate=PUBLIC_CHINESE_LR, ctc_output_kind=PUBLIC_CTC_OUTPUT_KIND, ctc_blank_id=0)
    public_write_record(folder, record)
    started = time.perf_counter()
    try:
        public_clear_model()
        torch.manual_seed(PUBLIC_TRAIN_SEED)
        model, processor = load_ocr('zh', 'baseline')
        char_to_id = public_chinese_vocabulary(processor)
        texts = [row['text'] for row in dataset]
        if any(not isinstance(text, str) for text in texts):
            raise TypeError('Chinese labels must be strings.')
        missing = Counter(character for text in texts for character in text if character not in char_to_id)
        supported = [index for index, text in enumerate(texts) if text and all(c in char_to_id for c in text)]
        record.update(supported_training_lines=len(supported), unsupported_or_empty_lines=len(texts) - len(supported),
                      unsupported_characters=dict(missing))
        print(f'Chinese labels supported: {len(supported)}/{len(texts)}; missing characters: {missing.most_common(10)}')
        if len(supported) != len(texts):
            raise ValueError('Unsupported/empty Chinese labels: fix the data or vocabulary; no rows silently omitted.')
        optimizer = torch.optim.AdamW(model.parameters(), lr=PUBLIC_CHINESE_LR)
        bn_types = (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)
        loss_sum, steps, alignment_skipped, used_examples = 0.0, 0, 0, 0
        for epoch in range(PUBLIC_TRAIN_EPOCHS):
            loader = DataLoader(Subset(dataset, supported), batch_size=PUBLIC_TRAIN_BATCH_SIZE, shuffle=True,
                generator=torch.Generator().manual_seed(PUBLIC_TRAIN_SEED + epoch), num_workers=0,
                collate_fn=lambda rows: collate_chinese(rows, processor, char_to_id, epoch))
            model.train()
            frozen_bn = [module for module in model.modules() if isinstance(module, bn_types)]
            for module in frozen_bn:
                module.eval()  # Freeze running statistics, retain trainable affine weights.
            for pixels, labels in tqdm(loader, desc=f'Chinese epoch {epoch + 1}/{PUBLIC_TRAIN_EPOCHS}', unit='batch'):
                optimizer.zero_grad(set_to_none=True)
                scores = model(pixel_values=pixels.to(model.device)).last_hidden_state
                loss, skipped = public_ctc_loss(scores, labels, len(processor.character_list))
                if skipped:
                    raise ValueError('CTC alignment too short; no training rows silently skipped.')
                alignment_skipped += skipped
                if loss is None:
                    continue
                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(norm).item():
                    raise RuntimeError('Non-finite Chinese gradients; no checkpoint selected.')
                optimizer.step()
                steps += 1
                used_examples += len(labels) - skipped
                loss_sum += loss.detach().item()
            if steps:
                print(f'Chinese epoch {epoch + 1}/{PUBLIC_TRAIN_EPOCHS}: loss {loss_sum / steps:.4f}; alignment skips {alignment_skipped}')
        if not steps:
            raise RuntimeError('No valid CTC steps completed; no checkpoint saved.')
        record.update(completed_steps=steps, used_example_visits=used_examples,
                      alignment_skipped_example_visits=alignment_skipped,
                      frozen_batchnorm_layers=len(frozen_bn), average_training_loss=loss_sum / steps,
                      elapsed_seconds=round(time.perf_counter() - started, 1))
        return public_save_model('zh', model, processor, folder, record)
    except Exception as error:
        record.update(status='failed', error=str(error))
        public_write_record(folder, record)
        raise
    finally:
        public_clear_model()


In [12]:
# Cell 13 · Training-only Unicode space normalization
if SESSION_MODE == 'fresh':
    # Fresh training targets only; runs before the vocabulary audit. No Drive labels are modified.
    # U+2002 EN SPACE -> U+0020 SPACE only in Chinese TRAINING targets.
    from copy import deepcopy

    public_clear_model()
    try:
        _, processor = load_ocr('zh', 'baseline')
        if ' ' not in public_chinese_vocabulary(processor):
            raise ValueError('Ordinary space is unsupported too; no normalization applied.')
    finally:
        public_clear_model()

    dataset = PUBLIC_SUBSETS['Chinese']['train']
    changes = deepcopy(DATA_PROVENANCE['zh'].get('curated_target_normalization', []))
    normalized = []
    for row in dataset.curated:
        updated = dict(row)  # Never mutate CURATED_ROWS or the on-Drive CSV.
        if '\u2002' in row['text']:
            updated['text'] = row['text'].replace('\u2002', ' ')
            changes.append(dict(sample_id=row['sample_id'], original=row['text'],
                                training_target=updated['text'], rule='U+2002 -> U+0020'))
        normalized.append(updated)

    # Update both records: The vocabulary audit restores its saved pre-filter provenance.
    provenance = deepcopy(DATA_PROVENANCE['zh'])
    provenance['curated_target_normalization'] = changes
    provenance['curated_fingerprint'] = fingerprint(normalized)
    if '_ZH_VOCAB_ORIGINAL' in globals() and _ZH_VOCAB_ORIGINAL.get('run_id') == RUN_ID:
        _ZH_VOCAB_ORIGINAL['provenance']['curated_target_normalization'] = deepcopy(changes)
        _ZH_VOCAB_ORIGINAL['provenance']['curated_fingerprint'] = provenance['curated_fingerprint']
    dataset.curated = normalized
    DATA_PROVENANCE['zh'] = provenance
    globals().pop('PREFLIGHT_REPORT', None)
    TRAINED_READY.clear()
    print(f'Training-only space normalization recorded for {len(changes)} curated lines; none removed.')
    print('Original labels and validation/test unchanged. Continue sequentially.')
else:
    print('Restore: no training labels changed.')


preprocessor_config.json:   0%|          | 0.00/203k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 84.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Training-only space normalization recorded for 3 curated lines; none removed.
Original labels and validation/test unchanged. Continue sequentially.


In [13]:
# Cell 14 · Quarantine unsupported PUBLIC Chinese lines, never alter labels
# No model training happens here; source labels are preserved.
# The original pretrained PP-OCR vocabulary cannot represent some CASIA symbols.
from collections import Counter
from copy import deepcopy
from torch.utils.data import ConcatDataset, Subset


def quarantine_chinese_public_vocabulary():
    global _ZH_VOCAB_ORIGINAL
    dataset = PUBLIC_SUBSETS['Chinese']['train']
    # Keep a pristine source for safe, repeatable reruns within this Run all.
    if globals().get('_ZH_VOCAB_ORIGINAL', {}).get('run_id') != RUN_ID:
        _ZH_VOCAB_ORIGINAL = dict(run_id=RUN_ID, public=dataset.public,
            provenance=deepcopy(DATA_PROVENANCE['zh']))
    original = _ZH_VOCAB_ORIGINAL
    if not isinstance(original['public'], ConcatDataset):
        raise TypeError('Expected the fresh notebook CombinedLines/ConcatDataset structure.')
    public_clear_model()
    try:
        _, processor = load_ocr('zh', 'baseline')
        vocabulary = set(public_chinese_vocabulary(processor))
    finally:
        public_clear_model()

    # Never silently remove your own labelled training lines.
    curated_issues = []
    for row in dataset.curated:
        missing = sorted(set(row['text']) - vocabulary)
        if not row['text'] or missing:
            curated_issues.append(dict(sample_id=row['sample_id'], text=row['text'], missing=missing))
    if curated_issues:
        print('Curated labels requiring review:', curated_issues)
        raise ValueError('Unsupported curated labels found. No filtering applied; preserve and review these labels.')

    provenance = deepcopy(original['provenance'])
    if len(original['public'].datasets) != len(provenance['sources']):
        raise ValueError('Public source/provenance mismatch; no filtering applied.')
    retained, exclusions, characters = [], [], Counter()
    for public, record in zip(original['public'].datasets, provenance['sources']):
        keep = []
        for index in range(len(public)):
            text = public[index]['text']
            missing = sorted(set(text) - vocabulary)
            if not text or missing:
                characters.update(c for c in text if c not in vocabulary)
                exclusions.append(dict(dataset=record['dataset'], revision=record['revision'],
                    split='train', post_dedup_index=index, text=text, missing=missing))
            else:
                keep.append(index)
        retained.append(Subset(public, keep))
        record['public_before_vocabulary_filter'] = len(public)
        record['vocabulary_excluded_lines'] = len(public) - len(keep)
        record['public_used'] = len(keep)
    public = ConcatDataset(retained)
    if not len(public):
        raise ValueError('No supported Chinese public lines remain; no filtering applied.')
    provenance['public_used'] = len(public)
    provenance['vocabulary_excluded_lines'] = len(exclusions)
    report = dict(run_id=RUN_ID, foundation=BASE['zh'], foundation_revision=BASE_REVISIONS['zh'],
        policy='whole unsupported public lines quarantined; labels unchanged; curated/heldout unchanged',
        public_before=len(original['public']), public_retained=len(public),
        public_excluded=len(exclusions), curated_retained=len(dataset.curated),
        unsupported_character_occurrences=dict(characters), exclusions=exclusions)
    path = Path(RESULTS_DIR) / RUN_ID / 'chinese_vocabulary_quarantine.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)
    # Commit only after the complete audit/report succeeds.
    dataset.public = public
    DATA_PROVENANCE['zh'] = provenance
    globals().pop('PREFLIGHT_REPORT', None)
    TRAINED_READY.clear()
    print(f"Chinese public: {report['public_before']} -> {len(public)}; quarantined {len(exclusions)} complete lines.")
    print('Unsupported character occurrences:', dict(characters))
    print('Curated Chinese retained unchanged:', len(dataset.curated), '| Report:', path)
    total = sum(len(PUBLIC_SUBSETS[name]['train']) for name in ('English', 'Chinese'))
    added = 0
    for language in ('en', 'zh'):
        for source in DATA_PROVENANCE[language]['sources']:
            if source.get('drive_train'):
                added += source['public_used']
                print(f"{language} | {source['dataset']} | {source['public_used']} eligible lines")
    print('Eligible lines from the Kaggle additions:', added)
    print('Actual eligible training lines across both languages:', total)
    print('Validation/test untouched. Continue sequentially to preflight and training.')
    return report

if SESSION_MODE == 'fresh':
    CHINESE_VOCABULARY_REPORT = quarantine_chinese_public_vocabulary()
else:
    print('Restore: saved training exclusions are retained in provenance.')


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Chinese public: 36352 -> 36278; quarantined 74 complete lines.
Unsupported character occurrences: {'‘': 48, '○': 24, '俢': 1, '伿': 1}
Curated Chinese retained unchanged: 47 | Report: /content/drive/MyDrive/handwritten_translation_project/results/20261109T033914Z_187fd23d/chinese_vocabulary_quarantine.json
en | sushant097/english-handwritten-line-dataset | 301 eligible lines
en | rohit04102005/hes-8394 | 300 eligible lines
en | trcphm/cvl-vt-lines | 2000 eligible lines
en | aryandg13/hiertext | 500 eligible lines
en | appenlimited/handwritten-ocr-image-data-in-english | 68 eligible lines
zh | appenlimited/chinese-handwritten-ocr-image-data | 106 eligible lines
zh | luyihang/handwritten-data-set-of-chinese-essays | 2845 eligible lines
Eligible lines from the Kaggle additions: 6120
Actual eligible training lines across both languages: 52710
Validation/test untouched. Continue sequentially to preflight and training.


In [14]:
# Cell 15 · Validate both training sets before either optimizer starts
# Chinese checks replay the exact seeded batches/augmentation for each epoch.
# This adds a no-gradient pass, but catches CTC failures before costly training.
def preflight_training():
    reports = {}
    for lang, name in [('en', 'English'), ('zh', 'Chinese')]:
        dataset = PUBLIC_SUBSETS[name]['train']
        public_clear_model()
        try:
            model, processor = load_ocr(lang, 'baseline')
            model.eval()
            if lang == 'en':
                tokenizer = processor.tokenizer
                if tokenizer.encode('Hello world') != [0, 31414, 232, 2]:
                    raise ValueError('Unexpected English tokenizer; training blocked.')
                checked = 0
                for start in range(0, len(dataset), PUBLIC_TRAIN_BATCH_SIZE):
                    rows = [dataset[i] for i in range(start, min(start + PUBLIC_TRAIN_BATCH_SIZE, len(dataset)))]
                    batch = prepare_english_batch(rows, processor)
                    if start == 0:
                        with torch.inference_mode():
                            loss = model(**{k: v.to(model.device) for k,v in batch.items()}).loss
                        if not torch.isfinite(loss).item():
                            raise RuntimeError('English preflight loss is non-finite.')
                    checked += len(rows)
                reports[lang] = {'labels_and_images_checked': checked, 'forward_smoke_batches': 1}
            else:
                char_ids = public_chinese_vocabulary(processor)
                vocabulary = set(char_ids)
                unsupported = {}
                for index in range(len(dataset)):
                    text = dataset[index]['text']
                    missing = sorted(set(text) - vocabulary)
                    if not text or missing:
                        unsupported[index] = missing
                if unsupported:
                    raise ValueError(f'Chinese labels unsupported in {len(unsupported)} rows; examples: {list(unsupported.items())[:20]}')
                batches = 0
                for epoch in range(PUBLIC_TRAIN_EPOCHS):
                    loader = DataLoader(dataset, batch_size=PUBLIC_TRAIN_BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(PUBLIC_TRAIN_SEED + epoch), num_workers=0,
                        collate_fn=lambda rows: collate_chinese(rows, processor, char_ids, epoch))
                    for pixels, labels in loader:
                        with torch.inference_mode():
                            scores = model(pixel_values=pixels.to(model.device)).last_hidden_state
                            loss, skipped = public_ctc_loss(scores, labels, len(processor.character_list))
                        if skipped or loss is None:
                            raise ValueError('Chinese CTC alignment infeasible in a planned batch; neither model trained.')
                        batches += 1
                        if batches % 100 == 0:
                            print('Chinese preflight batches:', batches, flush=True)
                reports[lang] = {'labels_checked': len(dataset), 'planned_batches_checked': batches}
            print(name, 'preflight passed:', reports[lang], flush=True)
        finally:
            public_clear_model()
    print('Both preflights passed. Optimization can now begin.')
    return reports

if SESSION_MODE == 'fresh':
    PREFLIGHT_REPORT = preflight_training()
else:
    print('Restore: expensive training preflight skipped.')


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

English preflight passed: {'labels_and_images_checked': 16385, 'forward_smoke_batches': 1}


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Chinese preflight batches: 100
Chinese preflight batches: 200
Chinese preflight batches: 300
Chinese preflight batches: 400
Chinese preflight batches: 500
Chinese preflight batches: 600
Chinese preflight batches: 700
Chinese preflight batches: 800
Chinese preflight batches: 900
Chinese preflight batches: 1000
Chinese preflight batches: 1100
Chinese preflight batches: 1200
Chinese preflight batches: 1300
Chinese preflight batches: 1400
Chinese preflight batches: 1500
Chinese preflight batches: 1600
Chinese preflight batches: 1700
Chinese preflight batches: 1800
Chinese preflight batches: 1900
Chinese preflight batches: 2000
Chinese preflight batches: 2100
Chinese preflight batches: 2200
Chinese preflight batches: 2300
Chinese preflight batches: 2400
Chinese preflight batches: 2500
Chinese preflight batches: 2600
Chinese preflight batches: 2700
Chinese preflight batches: 2800
Chinese preflight batches: 2900
Chinese preflight batches: 3000
Chinese preflight batches: 3100
Chinese preflight

In [15]:
# Cell 16 · Fresh training execution
if SESSION_MODE == 'fresh':
    if DEVICE != 'cuda':
        raise RuntimeError('Select a Colab GPU runtime before full training.')
    if set(globals().get('PREFLIGHT_REPORT', {})) != {'en', 'zh'}:
        raise RuntimeError('Run both training preflights first.')
    TRAINED_READY.clear()
    for language in ('en', 'zh'):
        # A new setup session is required to retrain; this cell never resumes an existing run.
        PUBLIC_TRAIN_RUN_ID = RUN_ID
        if (CHECKPOINT_DIR / language).exists():
            raise RuntimeError('This run already started. Run all from setup for a fresh run.')
    english_training_record = train_english_public()
    chinese_training_record = train_chinese_public()
    assert TRAINED_READY == {'en', 'zh'}
    print('Both fresh models ready. Training does not itself prove improvement.')
else:
    print('Restore: optimizers skipped; completed models already loaded.')


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


English epoch 1/1:   0%|          | 0/4097 [00:00<?, ?batch/s]

English epoch 1/1: loss 0.5911


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved en public checkpoint: /content/ocr_sessions/20261109T033914Z_187fd23d/en/weights
Runtime weights ready; the automatic backup cell preserves this completed run.
Run the larger paired validation before comparing recognition quality.


... (2 outputs left out)

Chinese epoch 1/1:   0%|          | 0/9082 [00:00<?, ?batch/s]

Chinese epoch 1/1: loss 0.2744; alignment skips 0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved zh public checkpoint: /content/ocr_sessions/20261109T033914Z_187fd23d/zh/weights
Runtime weights ready; the automatic backup cell preserves this completed run.
Run the larger paired validation before comparing recognition quality.
Both fresh models ready. Training does not itself prove improvement.


In [16]:
# Cell 17 · Automatically back up a successful fresh run
# Existing archives are never overwritten. Restore mode never writes its archive.
def save_completed_run():
    if TRAINED_READY != {'en', 'zh'}:
        raise RuntimeError('Both models must finish before saving a complete run.')
    inspect_saved_run(CHECKPOINT_DIR)
    root = PROJECT_DIR / 'saved_training_runs'
    root.mkdir(parents=True, exist_ok=True)
    archive = root / RUN_ID
    if archive.exists():
        raise FileExistsError(f'Backup already exists; not overwritten: {archive}')
    staging = root / ('.incomplete_' + RUN_ID)
    if staging.exists():
        raise FileExistsError(f'Previous incomplete copy exists: {staging}. Keep it for inspection; choose a new run ID.')
    print('Saving both models and run records to Drive...', flush=True)
    shutil.copytree(CHECKPOINT_DIR, staging)
    manifest = dict(schema_version=1, run_id=RUN_ID, files={})
    for relative, source in session_files(CHECKPOINT_DIR).items():
        digest = session_sha256(source)
        if session_sha256(staging / relative) != digest:
            raise RuntimeError(f'Backup copy mismatch: {relative}; do not disconnect.')
        manifest['files'][relative] = dict(bytes=source.stat().st_size, sha256=digest)
    (staging / 'backup_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    staging.rename(archive)
    print('BACKUP VERIFIED:', archive)
    print('Use this path as SAVED_RUN next time. Model weights/processors and training records saved; not optimizer/session state.')
    return archive

if SESSION_MODE == 'fresh' and AUTO_SAVE_FRESH:
    SAVED_THIS_RUN = save_completed_run()
else:
    print('Automatic backup skipped (restore mode or AUTO_SAVE_FRESH=False).')


Legacy backup: no saved checksum manifest. File structure is checked; original-save integrity cannot be re-proved.
Saving both models and run records to Drive...
BACKUP VERIFIED: /content/drive/MyDrive/handwritten_translation_project/saved_training_runs/20261109T033914Z_187fd23d
Use this path as SAVED_RUN next time. Model weights/processors and training records saved; not optimizer/session state.


In [17]:
# Cell 18 · Fixed validation cohorts

import hashlib, json, random, time, gc, os
from pathlib import Path
from PIL import Image, ImageOps
from datasets import load_dataset, load_dataset_builder
from huggingface_hub import HfApi, hf_hub_download
import torch

OCR_SEED = 20261105
OCR_EVAL_ROOT = Path(RESULTS_DIR) / RUN_ID / 'public_validation'
OCR_CACHE_ROOT = Path(PUBLIC_DATA_DIR) / 'larger_validation_v1'
OCR_EVAL_ROOT.mkdir(parents=True, exist_ok=True)
OCR_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OCR_DATASETS = {'en': 'Teklia/IAM-line', 'zh': 'Teklia/CASIA-HWDB2-line'}
OCR_CASIA_COUNT = 1000
# Empty: use the source's current commit once, then pin it in the cohort manifest.
# To exactly repeat an existing cohort in another session, copy its recorded revision here.
OCR_DATASET_REVISIONS = {lang: next(source['revision'] for source in DATA_PROVENANCE[lang]['sources']
    if source['dataset'] == OCR_DATASETS[lang]) for lang in ('en', 'zh')}
OCR_CACHE_ROOT = OCR_CACHE_ROOT / ('_'.join(OCR_DATASET_REVISIONS[lang][:12] for lang in ('en', 'zh')))
OCR_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def ocr_json_hash(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


def ocr_file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def ocr_atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def ocr_prepare_cohort(lang):
    selection_file = OCR_CACHE_ROOT / f'{lang}_selection.json'
    if selection_file.exists():
        record = json.loads(selection_file.read_text(encoding='utf-8'))
        expected_count = record['population'] if lang == 'en' else min(OCR_CASIA_COUNT, record['population'])
        if (record['dataset'] != OCR_DATASETS[lang] or record['seed'] != OCR_SEED or
            record['selected'] != expected_count or
            (OCR_DATASET_REVISIONS[lang] and record['revision'] != OCR_DATASET_REVISIONS[lang])):
            raise ValueError('Cohort settings changed. Use a new OCR_CACHE_ROOT; do not overwrite the original cohort.')
        for row in record['rows']:
            path = OCR_CACHE_ROOT / row['image_path']
            if not path.is_file() or ocr_file_hash(path) != row['sha256']:
                raise ValueError(f'Cached image missing/changed: {path}; preserve this manifest and use a new cache folder.')
        if record['fingerprint'] != ocr_json_hash(record['rows']):
            raise ValueError('Cohort manifest changed; do not silently replace the evaluation set.')
        return record

    repo = OCR_DATASETS[lang]
    revision = OCR_DATASET_REVISIONS[lang] or HfApi().dataset_info(repo).sha
    info = load_dataset_builder(repo, revision=revision).info
    if not info.splits or 'validation' not in info.splits:
        raise RuntimeError(f'{repo} has no published validation split; do not substitute training/test data.')
    total = info.splits['validation'].num_examples
    if not isinstance(total, int) or total < 1:
        raise RuntimeError('Validation count is unavailable; inspect the dataset metadata first.')
    count = total if lang == 'en' else min(OCR_CASIA_COUNT, total)
    indices = list(range(total)) if lang == 'en' else sorted(random.Random(OCR_SEED).sample(range(total), count))
    selected = set(indices)
    destination = OCR_CACHE_ROOT / f'{lang}_{revision[:12]}'
    destination.mkdir(parents=True, exist_ok=True)
    rows = []
    # Streaming avoids preparing the training/test splits. A uniform CASIA sample
    # still requires reading across its validation stream; this costs download time, not GPU time.
    stream = load_dataset(repo, split='validation', revision=revision, streaming=True)
    for index, example in enumerate(stream):
        if index > indices[-1]:
            break
        if index not in selected:
            continue
        text = example['text'].strip()
        if not text:
            raise ValueError(f'Blank reference at {repo}/validation/{index}; inspect rather than silently dropping it.')
        path = destination / f'{index:06d}.png'
        example['image'].convert('RGB').save(path)
        rows.append(dict(id=f'{repo}@{revision}:validation:{index}', source_index=index,
                         text=text, image_path=str(path.relative_to(OCR_CACHE_ROOT)),
                         sha256=ocr_file_hash(path)))
        if len(rows) % 100 == 0:
            print(lang, 'cached', len(rows), '/', count)
    if len(rows) != count:
        raise ValueError(f'Metadata said {total} rows, but only {len(rows)} selected rows were read.')
    record = dict(dataset=repo, revision=revision, split='validation', population=total,
                  selected=count, selection='full split' if lang == 'en' else 'seeded uniform indices',
                  seed=OCR_SEED, writer_metadata='not supplied by this image/text mirror',
                  rows=rows, fingerprint=ocr_json_hash(rows))
    ocr_atomic_json(selection_file, record)
    return record


if RUN_OCR_COHORTS:
    OCR_COHORTS = {lang: ocr_prepare_cohort(lang) for lang in ('en', 'zh')}
    for lang, record in OCR_COHORTS.items():
        print(lang, record['selected'], '/', record['population'], record['selection'], record['fingerprint'][:12])
    print('These are validation results, not final-test results or proven unseen-writer evaluation.')
else:
    print('Validation download skipped. Enable RUN_OCR_COHORTS before evaluation.')


Validation download skipped. Enable RUN_OCR_COHORTS before evaluation.


In [18]:
# Cell 19 · Batched OCR evaluation and resume

# Run after fixed validation cohort setup. Default evaluates baseline vs existing public checkpoint only.
from transformers import (RobertaTokenizer, ViTImageProcessor, TrOCRProcessor,
    VisionEncoderDecoderModel, AutoImageProcessor, AutoModelForTextRecognition)
import transformers

OCR_BATCH_SIZE = 4
OCR_MAX_NEW_TOKENS = 128
OCR_PREPROCESS_MODES = ['none']  # Optional second experiment: ['none', 'autocontrast']
OCR_EVAL_MODELS = {
    'en_baseline': dict(lang='en', source=BASE['en']),
    'en_public': dict(lang='en', source=str(PUBLIC['en'])),
    'zh_baseline': dict(lang='zh', source=BASE['zh']),
    'zh_public': dict(lang='zh', source=str(PUBLIC['zh'])),
}


def ocr_preprocess(image, mode='none'):
    image = ImageOps.exif_transpose(image).convert('RGB')
    if mode == 'none':
        return image
    if mode == 'autocontrast':
        return ImageOps.autocontrast(image.convert('L'), cutoff=0).convert('RGB')
    raise ValueError(f'Unknown preprocessing: {mode}')


def ocr_load_source(lang, source, revision=None):
    local = Path(source)
    is_local = local.is_dir()
    if (source.startswith('/') or source.startswith('.')) and not is_local:
        raise FileNotFoundError(f'Checkpoint not found: {source}')
    revision = None if is_local else (revision or BASE_REVISIONS[lang])
    kwargs = {} if is_local else {'revision': revision}
    identity = {'source': source, 'revision': revision}
    if is_local:
        files = sorted(p for p in local.iterdir() if p.is_file() and p.suffix in ('.json', '.txt', '.safetensors', '.bin', '.model'))
        if not files:
            raise ValueError(f'No model files in {local}')
        identity['files'] = {p.name: ocr_file_hash(p) for p in files}
    if lang == 'en':
        def tokenizer_file(name):
            if is_local and (local / name).is_file(): return str(local / name)
            repo = BASE['en'] if is_local else source
            token_revision = BASE_REVISIONS[lang] if is_local else revision
            identity['fallback_tokenizer'] = dict(repo=repo, revision=token_revision)
            return hf_hub_download(repo, name, revision=token_revision)
        tokenizer = RobertaTokenizer(vocab=tokenizer_file('vocab.json'),
                                    merges=tokenizer_file('merges.txt'), model_max_length=512)
        processor = TrOCRProcessor(image_processor=ViTImageProcessor.from_pretrained(source, **kwargs),
                                   tokenizer=tokenizer)
        model = VisionEncoderDecoderModel.from_pretrained(source, dtype=torch.float32, **kwargs)
    else:
        processor = AutoImageProcessor.from_pretrained(source, **kwargs)
        model = AutoModelForTextRecognition.from_pretrained(source, dtype=torch.float32, **kwargs)
    return model.to(DEVICE).eval(), processor, identity


def ocr_evaluate(tag, spec, mode='none'):
    if callable(globals().get('release_routed_mt')): release_routed_mt()
    lang = spec['lang']; cohort = OCR_COHORTS[lang]; rows = cohort['rows']
    if callable(globals().get('release_ocr')): release_ocr()
    if callable(globals().get('release_mt')): release_mt()
    model, processor, identity = ocr_load_source(lang, spec['source'])
    path = OCR_EVAL_ROOT / f'{tag}_{mode}.json'
    signature = dict(model=identity, language=lang, cohort=cohort['fingerprint'],
                     preprocessing=mode, preprocessing_policy='exif-rgb-autocontrast-v1', batch_size=OCR_BATCH_SIZE, max_new_tokens=OCR_MAX_NEW_TOKENS,
                     metric_policy=workflow.METRIC_POLICY, dtype='float32',
                     torch=str(torch.__version__), transformers=transformers.__version__)
    predictions = []
    try:
        if path.exists():
            saved = json.loads(path.read_text(encoding='utf-8'))
            if saved['signature'] != signature:
                raise ValueError(f'{path} belongs to different settings/data/weights. Use a new tag or output folder.')
            predictions = saved['predictions']
            if ([(r['id'],r['reference']) for r in predictions] != [(r['id'],r['text']) for r in rows[:len(predictions)]]
                or len(predictions) > len(rows) or not all(isinstance(r['prediction'],str) for r in predictions)):
                raise ValueError('Saved predictions are not an exact cohort prefix.')
        start_at = len(predictions)
        for start in range(start_at, len(rows), OCR_BATCH_SIZE):
            batch = rows[start:start + OCR_BATCH_SIZE]
            images = []
            for row in batch:
                with Image.open(OCR_CACHE_ROOT / row['image_path']) as im:
                    images.append(ocr_preprocess(im, mode))
            with torch.inference_mode():
                inputs = processor(images=images, return_tensors='pt').to(DEVICE)
                if lang == 'en':
                    tokens = model.generate(inputs['pixel_values'], max_new_tokens=OCR_MAX_NEW_TOKENS,
                        do_sample=False, num_beams=1, decoder_start_token_id=processor.tokenizer.eos_token_id,
                        eos_token_id=processor.tokenizer.eos_token_id, pad_token_id=processor.tokenizer.pad_token_id)
                    decoded = processor.batch_decode(tokens, skip_special_tokens=True)
                    # Count generated positions up to EOS. Hitting the cap is a review flag, not proof of an error.
                    caps = []
                    for sequence in tokens.tolist():
                        generated = sequence[1:]
                        eos = processor.tokenizer.eos_token_id
                        used = generated.index(eos) + 1 if eos in generated else len(generated)
                        caps.append(used >= OCR_MAX_NEW_TOKENS)
                else:
                    decoded = [r['text'] for r in processor.post_process_text_recognition(model(**inputs))]
                    caps = [False] * len(batch)  # CTC has no decoder token cap.
            if len(decoded) != len(batch) or len(caps) != len(batch):
                raise RuntimeError('Decoder returned an unexpected batch size; no rows were silently dropped.')
            predictions.extend(dict(id=r['id'], reference=r['text'], prediction=p.strip(),
                                    output_limit_flag=cap) for r,p,cap in zip(batch,decoded,caps))
            ocr_atomic_json(path, dict(signature=signature, status='partial', predictions=predictions))
            print(tag, mode, len(predictions), '/', len(rows))
        refs = [r['reference'] for r in predictions]; hyps = [r['prediction'] for r in predictions]
        metrics = workflow.error_rates(refs, hyps, lang)
        result = dict(signature=signature, status='complete', samples=len(predictions),
                      metrics=metrics, output_limit_flags=sum(r['output_limit_flag'] for r in predictions),
                      predictions=predictions)
        ocr_atomic_json(path, result)
        return result
    finally:
        model.cpu(); del model, processor
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


if RUN_OCR_EVALUATION:
    if 'OCR_COHORTS' not in globals():
        raise RuntimeError('Prepare the fixed validation cohorts first.')
    OCR_RESULTS = {}
    for tag, spec in OCR_EVAL_MODELS.items():
        for mode in OCR_PREPROCESS_MODES:
            OCR_RESULTS[(tag, mode)] = ocr_evaluate(tag, spec, mode)
    print('Complete. Predictions are saved per batch; reruns resume compatible files.')
else:
    print('OCR inference skipped. Enable RUN_OCR_EVALUATION when ready.')


OCR inference skipped. Enable RUN_OCR_EVALUATION when ready.


In [19]:
# Cell 20 · OCR confidence intervals

# Run after batched OCR evaluation. Negative delta = fewer errors for B than A.
import numpy as np
import unicodedata
import pandas as pd

OCR_BOOTSTRAPS = 2000
OCR_CI_SEED = 42


def ocr_metric_sequences(texts, metric):
    if metric == 'CER': return list(texts)
    if metric == 'CER_NFKC': return [unicodedata.normalize('NFKC', x) for x in texts]
    if metric == 'CER_PUNCT': return [workflow.normalize_punctuation(x) for x in texts]
    if metric == 'WER': return [x.split() for x in texts]
    raise ValueError(metric)


def ocr_bootstrap_metric(references, predictions_a, predictions_b=None, metric='CER', repeats=2000, seed=42):
    refs = list(references); a = list(predictions_a)
    b = None if predictions_b is None else list(predictions_b)
    if not refs or len(refs) != len(a) or (b is not None and len(refs) != len(b)):
        raise ValueError('Nonempty, exactly aligned references and predictions required.')
    if repeats < 20: raise ValueError('Use at least 20 bootstrap repetitions.')
    r = ocr_metric_sequences(refs, metric); a = ocr_metric_sequences(a, metric)
    b = None if b is None else ocr_metric_sequences(b, metric)
    lengths = np.array([len(x) for x in r], dtype=np.int64)
    errors = np.array([workflow.distance(x,y) for x,y in zip(r,a)], dtype=np.float64)
    if b is not None:
        errors = np.array([workflow.distance(x,y) for x,y in zip(r,b)], dtype=np.float64) - errors
    total = int(lengths.sum())
    if total == 0: raise ValueError('Metric has zero reference length.')
    rng = np.random.default_rng(seed); samples = []
    for _ in range(repeats):
        indices = rng.integers(0, len(refs), size=len(refs))
        denominator = int(lengths[indices].sum())
        if denominator: samples.append(float(errors[indices].sum() / denominator))
    if not samples: raise ValueError('No valid bootstrap samples.')
    lo, hi = np.quantile(samples, [.025, .975])
    return dict(metric=metric, estimate=float(errors.sum()/total), ci95_low=float(lo), ci95_high=float(hi),
                comparison='B minus A (paired)' if b is not None else 'single model',
                resampling='line-level; writer/page IDs unavailable', samples=len(refs), repeats=len(samples))


def ocr_compare_saved(tag_a, tag_b=None, mode_a='none', mode_b='none'):
    def read(tag, mode):
        record = json.loads((OCR_EVAL_ROOT / f'{tag}_{mode}.json').read_text(encoding='utf-8'))
        if record['status'] != 'complete': raise ValueError(f'{tag} is incomplete.')
        return record
    a = read(tag_a, mode_a); b = read(tag_b, mode_b) if tag_b else None
    pairs_a = [(r['id'],r['reference']) for r in a['predictions']]
    if b and (a['signature']['cohort'] != b['signature']['cohort'] or
              pairs_a != [(r['id'],r['reference']) for r in b['predictions']]):
        raise ValueError('Cannot pair different cohorts/order/references.')
    refs = [r['reference'] for r in a['predictions']]
    pred_a = [r['prediction'] for r in a['predictions']]
    pred_b = None if b is None else [r['prediction'] for r in b['predictions']]
    metrics = ['CER','CER_NFKC','CER_PUNCT'] + (['WER'] if a['signature']['language']=='en' else [])
    return [dict(model_a=tag_a,model_b=tag_b,preprocess_a=mode_a,preprocess_b=mode_b,
                 **ocr_bootstrap_metric(refs,pred_a,pred_b,m,OCR_BOOTSTRAPS,OCR_CI_SEED)) for m in metrics]


if RUN_OCR_INTERVALS:
    intervals = []
    for tag in OCR_EVAL_MODELS:
        for mode in OCR_PREPROCESS_MODES:
            intervals += ocr_compare_saved(tag,mode_a=mode)
    for a,b in [('en_baseline','en_public'),('zh_baseline','zh_public')]:
        if a in OCR_EVAL_MODELS and b in OCR_EVAL_MODELS:
            for mode in OCR_PREPROCESS_MODES:
                intervals += ocr_compare_saved(a,b,mode_a=mode,mode_b=mode)
    interval_table = pd.DataFrame(intervals)
    # Convert ratios to percentage points for readability.
    for col in ('estimate','ci95_low','ci95_high'): interval_table[col] *= 100
    print('Single-model values are percentages. Paired differences are percentage points; negative favours B.')
    display(interval_table)
    interval_table.to_csv(OCR_EVAL_ROOT / 'intervals_percentage_points.csv', index=False)
    print('Exploratory line-level intervals only. Shared writers/pages can make these too narrow; not writer-level proof.')
else:
    print('OCR statistics skipped. Enable RUN_OCR_INTERVALS after saved predictions are ready.')


OCR statistics skipped. Enable RUN_OCR_INTERVALS after saved predictions are ready.


In [20]:
# Cell 21 · Optional dataset audit and grouped OCR intervals

# These helpers are embedded in the generated notebook; no model download required.
import hashlib,json,random
from collections import defaultdict
from pathlib import Path

def audit_external_manifest(records, require_writer_ids=True):
    """Validate a converted dataset manifest BEFORE importing it into training.

    Required fields: source, sample_id, image_path, text, language, split,
    writer_id and page_id. Missing writer IDs may be accepted explicitly for
    training-only sources, never as proof of unseen-writer evaluation.
    """
    rows=list(records); issues=[]; seen={}; writers=defaultdict(set); pages=defaultdict(set); hashes=defaultdict(set)
    required=('source','sample_id','image_path','text','language','split','writer_id','page_id')
    for index,row in enumerate(rows):
        missing=[k for k in required if k not in row]
        if missing:
            issues.append(f'row {index}: missing {missing}');continue
        if row['split'] not in ('train','validation','test') or row['language'] not in ('en','zh'):
            issues.append(f'row {index}: invalid split/language')
        if not isinstance(row['text'],str) or not row['text'].strip(): issues.append(f'row {index}: blank label')
        key=(row['source'],row['sample_id'])
        if key in seen: issues.append(f'row {index}: repeated source/sample ID')
        seen[key]=index
        if not row['writer_id']:
            if require_writer_ids or row['split'] != 'train': issues.append(f'row {index}: writer unknown for held-out claim')
        else: writers[(row['source'],row['writer_id'])].add(row['split'])
        if row['page_id']: pages[(row['source'],row['page_id'])].add(row['split'])
        p=Path(row['image_path'])
        if not p.is_file(): issues.append(f'row {index}: missing image');continue
        digest=hashlib.sha256(p.read_bytes()).hexdigest()
        hashes[digest].add(row['split'])
    for label,groups in [('writer',writers),('page',pages),('identical image',hashes)]:
        for key,splits in groups.items():
            if len(splits)>1: issues.append(f'{label} crosses splits: {key}')
    return {'rows':len(rows),'valid':not issues,'issues':issues,
            'note':'Byte hashes detect exact duplicates, not different crops of the same page. Audit corpus overlap manually.'}

def paired_cer_interval(references, predictions_a, predictions_b, groups=None, repeats=1000, seed=42):
    """Paired bootstrap of CER(B)-CER(A), using writer/page groups when available.

    Inputs must be aligned and from one language/split. With no group metadata,
    this is line-level exploratory uncertainty, NOT writer-level inference.
    """
    refs=list(references);a=list(predictions_a);b=list(predictions_b)
    if not refs or not(len(refs)==len(a)==len(b)):raise ValueError('Nonempty aligned arrays required')
    if repeats<20:raise ValueError('Use at least 20 bootstrap repetitions')
    supplied=groups is not None
    groups=list(range(len(refs))) if groups is None else list(groups)
    if len(groups)!=len(refs):raise ValueError('Group count differs from references')
    blocks=defaultdict(list)
    for i,g in enumerate(groups):blocks[g].append(i)
    units=list(blocks.values());rng=random.Random(seed)
    errors_a=[workflow.distance(r,p) for r,p in zip(refs,a)]
    errors_b=[workflow.distance(r,p) for r,p in zip(refs,b)]
    denom=sum(map(len,refs))
    if not denom:raise ValueError('References contain no characters')
    delta=(sum(errors_b)-sum(errors_a))/denom
    samples=[]
    for _ in range(repeats):
        indices=[i for _ in units for i in rng.choice(units)]
        d=sum(len(refs[i]) for i in indices)
        if d:samples.append(sum(errors_b[i]-errors_a[i] for i in indices)/d)
    samples.sort()
    if not samples:raise ValueError('No nonempty bootstrap sample')
    return {'delta_CER_B_minus_A':delta,'ci95':[samples[int(.025*(len(samples)-1))],samples[int(.975*(len(samples)-1))]],
            'resampling_unit':'provided writer/page group' if supplied else 'line (exploratory only)',
            'groups':len(units),'seed':seed,'repetitions':len(samples)}

print('Optional manifest audit and paired-CER helpers ready. No data downloaded and no model run.')

# Cheap deterministic smoke checks (no downloads or GPU inference).
assert workflow.error_rates(['Ａ。'], ['A.'], 'zh')['CER_PUNCT'] == 0
assert workflow.error_rates(['a'], ['a'], 'en')['exact_match'] == 1
_probe = paired_cer_interval(['abc', 'def'], ['abc', 'def'], ['ab', 'de'], repeats=50)
assert abs(_probe['delta_CER_B_minus_A'] - 1/3) < 1e-9
print('Dependency-light scoring/bootstrap smoke checks passed.')


Optional manifest audit and paired-CER helpers ready. No data downloaded and no model run.
Dependency-light scoring/bootstrap smoke checks passed.


In [21]:
# Cell 22 · Protect evaluation against the saved training membership

def verify_saved_curated_membership():
    if not CURATED_READY:
        raise RuntimeError('Enable the curated audit in setup before evaluation.')
    if SESSION_MODE != 'restore':
        return
    for lang in ('en', 'zh'):
        saved = DATA_PROVENANCE[lang]
        ids = set(saved['curated_sample_ids'])
        writers = set(saved['curated_writers'])
        heldout = [r for r in CURATED_ROWS if r['split'] != 'train']
        if any(r['sample_id'] in ids or r['writer_id'] in writers for r in heldout):
            raise ValueError('Current held-out data overlaps this saved run training identities/writers.')
        original = [dict(r) for r in CURATED_ROWS if r['sample_id'] in ids]
        changes = {r['sample_id']: r for r in saved.get('curated_target_normalization', [])}
        for row in original:
            if row['sample_id'] in changes:
                change = changes[row['sample_id']]
                if row['text'] != change['original']:
                    raise ValueError('A saved training label changed; historical overlap audit cannot be reconstructed.')
                row['text'] = change['training_target']
        if len(original) != len(ids) or fingerprint(original) != saved['curated_fingerprint']:
            raise ValueError('Saved training membership cannot be reconstructed from current curated data. Demo works, but evaluation is blocked pending an explicit historical audit.')
    print('Saved curated training membership reconstructed; current heldout writer/image checks apply.')


In [22]:
# Cell 23 · Held-out curated validation: never train/select on these rows
# Run optionally after labels are reviewed. Test stays locked by default.
def evaluate_fresh_curated(split='validation'):
    if split not in ('validation', 'test'):
        raise ValueError('Choose a held-out split.')
    if split == 'test' and not RUN_FINAL_TEST:
        raise RuntimeError('Freeze the evaluation plan, then set RUN_FINAL_TEST=True.')
    if TRAINED_READY != {'en', 'zh'}:
        raise RuntimeError('Train or restore both models first.')
    verify_saved_curated_membership()
    results = {}
    for lang in ('en', 'zh'):
        rows = curated_rows(split, lang)
        if not rows:
            print(lang, split, 'has no rows'); continue
        paired = {}
        for version in ('baseline', 'public'):
            model, processor = load_ocr(lang, version)
            predictions = [transcribe(open_line(r), lang, model, processor) for r in rows]
            scores = error_rates([r['text'] for r in rows], predictions, lang)
            paired[version] = predictions
            results[lang, version] = scores
            print(lang, version, len(rows), scores)
        print('Page-grouped CER interval:', paired_cer_interval([r['text'] for r in rows], paired['baseline'], paired['public'], groups=[r['source_page_id'] for r in rows]))
        release_ocr()
    return results
if RUN_CURATED_VALIDATION:
    curated_validation_results = evaluate_fresh_curated()

if RUN_FINAL_TEST:
    curated_test_results = evaluate_fresh_curated('test')


In [23]:
# Cell 24 · Translation controls

# All downloads, inference, benchmark reruns and saved-result analyses are opt-in.
RUN_MT_TYPED_CHECKS = False
RUN_NLLB_TYPED_CHECKS = False
RUN_MT_DATA_LOAD = False
RUN_MT_BASELINES = False
RUN_NLLB_BENCHMARK = False
RUN_MT_SAVED_REVIEW = False
RUN_MT_CHECK_TESTS = False
RUN_MT_ROUTE_TESTS = True  # Cheap mocks only; no weights or inference.
RUN_COMET_SETUP = False
RUN_COMET_SCORING = False
RUN_PAIRED_BOOTSTRAP = False
RUN_FLAG_ASSOCIATION = False

# Select existing completed result folders here to analyse without rerunning models.
MT_BASELINE_DIR = globals().get('MT_BASELINE_DIR', None)
MT_NLLB_DIR = globals().get('MT_NLLB_DIR', None)
COMET_RESULTS_DIR = globals().get('COMET_RESULTS_DIR', None)
MT_BASELINE_RESUME_DIR = None
MT_NLLB_RESUME_DIR = None
BOOTSTRAP_REPEATS = 200  # Exploratory; use 1,000–2,000 for more stable intervals.
BOOTSTRAP_SEED = 42


In [24]:
# Cell 25 · Marian model loading

import gc
import json
import time
from pathlib import Path

import torch
import transformers
from transformers import MarianMTModel, MarianTokenizer

MT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# The key identifies the INPUT language; the target is the other language.
MT_MODELS = {
    'en': 'Helsinki-NLP/opus-mt-en-zh',
    'zh': 'Helsinki-NLP/opus-mt-zh-en',
}
MT_ZH_TAG = '>>cmn_Hans<<'  # Request Simplified Chinese output.
MT_MAX_INPUT_TOKENS = 256
MT_MAX_NEW_TOKENS = 128
MT_NUM_BEAMS = 4

# Release a previous translation model if this setup cell is rerun.
if callable(globals().get('release_mt')):
    release_mt()
_MT = None


def release_mt():
    global _MT
    if _MT is not None:
        _MT['model'].cpu()
    _MT = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_mt(source_language, *, keep_ocr=False):
    global _MT

    if source_language not in MT_MODELS:
        raise ValueError("source_language must be 'en' or 'zh'.")

    # Release OCR only when the caller allows it.
    if not keep_ocr and callable(globals().get('release_ocr')):
        release_ocr()

    if _MT is not None and _MT['source_language'] == source_language:
        return _MT

    release_mt()
    model_id = MT_MODELS[source_language]
    started = time.perf_counter()
    print('Loading translation model:', model_id)

    tokenizer = MarianTokenizer.from_pretrained(model_id)
    if source_language == 'en' and MT_ZH_TAG not in tokenizer.get_vocab():
        raise RuntimeError(
            'The tokenizer is missing the Simplified Chinese tag.'
        )

    # Keep a consistent FP32 baseline.
    model = MarianMTModel.from_pretrained(
        model_id, dtype=torch.float32
    )
    model.to(MT_DEVICE).eval()

    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()

    _MT = dict(
        source_language=source_language,
        model_id=model_id,
        revision=getattr(model.config, '_commit_hash', None),
        model=model,
        tokenizer=tokenizer,
    )

    print(
        f'Ready on {MT_DEVICE}; '
        f'loading took {time.perf_counter() - started:.2f}s'
    )
    return _MT


In [25]:
# Cell 26 · Sentence translation API

def translate_text(text, source_language, *, keep_ocr=False):
    if source_language not in MT_MODELS:
        raise ValueError(
            "Choose 'en' for English input or 'zh' for Chinese input."
        )
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Enter a non-empty sentence.')

    text = text.strip()
    loaded = load_mt(source_language, keep_ocr=keep_ocr)
    model, tokenizer = loaded['model'], loaded['tokenizer']

    prepared = (
        f'{MT_ZH_TAG} {text}'
        if source_language == 'en'
        else text
    )

    # Time text processing and inference, excluding model loading/downloads.
    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()

    inputs = tokenizer(
        prepared, return_tensors='pt', truncation=False
    )
    input_tokens = inputs['input_ids'].shape[1]
    limit = min(
        MT_MAX_INPUT_TOKENS,
        model.config.max_position_embeddings,
    )
    if input_tokens > limit:
        raise ValueError(
            f'Input has {input_tokens} tokens; limit is {limit}. '
            'Use a shorter sentence.'
        )

    # Fixed decoding settings make later comparisons easier to reproduce.
    with torch.inference_mode():
        generated = model.generate(
            **inputs.to(MT_DEVICE),
            max_length=None,
            max_new_tokens=MT_MAX_NEW_TOKENS,
            num_beams=MT_NUM_BEAMS,
            do_sample=False,
            early_stopping=True,
        )

    translation = tokenizer.decode(
        generated[0], skip_special_tokens=True
    ).strip()

    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()
    seconds = time.perf_counter() - started

    return dict(
        source_language=source_language,
        target_language=(
            'zh-Hans' if source_language == 'en' else 'en'
        ),
        source_text=text,
        translation=translation,
        model_id=loaded['model_id'],
        revision=loaded['revision'],
        input_tokens=input_tokens,
        translation_seconds=round(seconds, 4),
        # Reaching the cap can mean incomplete output.
        output_limit_reached=(
            generated.shape[1] - 1 >= MT_MAX_NEW_TOKENS
        ),
    )


In [26]:
# Cell 27 · Timed translation decoding

import sys
import numpy as np
import pandas as pd
from IPython.display import display

MT_BENCH_BEAMS = (1, 4)
MT_BENCH_REPEATS = 3
MT_BENCH_WARMUPS = 3

# Benchmark decoding supports longer sentences than the typed checks.
MT_BENCH_INPUT_LIMIT = 512
MT_BENCH_OUTPUT_LIMIT = 256


@torch.inference_mode()
def mt_decode_loaded(text, source_language, loaded, beams):
    """Translate with an already-loaded model; never read reference answers."""
    if source_language not in ('en', 'zh') or beams not in MT_BENCH_BEAMS:
        raise ValueError('Choose en/zh and one of MT_BENCH_BEAMS.')
    if loaded['source_language'] != source_language:
        raise ValueError('The loaded model has the wrong translation direction.')
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Source text must be a non-empty string.')

    model, tokenizer = loaded['model'], loaded['tokenizer']
    device = next(model.parameters()).device

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    started = time.perf_counter()

    prepared = text.strip()
    if source_language == 'en':
        prepared = f'{MT_ZH_TAG} {prepared}'

    inputs = tokenizer(prepared, return_tensors='pt', truncation=False)
    input_tokens = inputs['input_ids'].shape[1]
    limit = min(MT_BENCH_INPUT_LIMIT, model.config.max_position_embeddings)

    if input_tokens > limit:
        raise ValueError(f'Input has {input_tokens} tokens; limit is {limit}.')

    generated = model.generate(
        **inputs.to(device),
        max_length=None,
        max_new_tokens=MT_BENCH_OUTPUT_LIMIT,
        num_beams=beams,
        num_return_sequences=1,
        do_sample=False,
        early_stopping=(beams > 1),
        length_penalty=1.0,
        use_cache=True,
    )

    token_ids = generated[0].tolist()
    prediction = tokenizer.decode(
        token_ids, skip_special_tokens=True
    ).strip()

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - started

    return dict(
        prediction=prediction,
        seconds=elapsed,
        input_tokens=input_tokens,
        output_tokens=len(token_ids) - 1,  # Exclude decoder-start token.
        hit_output_limit=(len(token_ids) - 1 >= MT_BENCH_OUTPUT_LIMIT),
    )


In [27]:
# Cell 28 · Reference-free review checks

import re
import unicodedata
from decimal import Decimal

# Flags request human review only; they never trigger retries or replace output.
MT_CHECK_VERSION = 'numeric_repetition_v3'
MT_CULTURE = {'en': 'en-us', 'zh': 'zh-cn'}
MT_MONTHS = {
    name: str(i)
    for i, name in enumerate(
        ('January February March April May June July August '
         'September October November December').split(), 1
    )
}

# Convert common length, mass and voltage units to shared units.
MT_UNIT_RULES = [
    (r'(?<![a-z])(?:kilometers?|kilometres?|km)(?![a-z])|公里|千米',
     'KM', 'm', '1000'),
    (r'(?<![a-z])(?:centimeters?|centimetres?|cm)(?![a-z])|厘米',
     'CM', 'm', '0.01'),
    (r'(?<![a-z])(?:millimeters?|millimetres?|mm)(?![a-z])|毫米',
     'MM', 'm', '0.001'),
    (r'(?<![a-z])(?:meters?|metres?|m)(?![a-z])|米',
     'M', 'm', '1'),
    (r'(?<![a-z])(?:kilograms?|kg)(?![a-z])|公斤|千克',
     'KG', 'g', '1000'),
    (r'(?<![a-z])(?:milligrams?|mg)(?![a-z])|毫克',
     'MG', 'g', '0.001'),
    (r'(?<![a-z])(?:grams?|g)(?![a-z])|克',
     'G', 'g', '1'),
    (r'(?<![a-z])(?:volts?|v)(?![a-z])|伏特|伏',
     'V', 'V', '1'),
]
MT_UNIT_VALUES = {
    code: (unit, Decimal(scale))
    for _, code, unit, scale in MT_UNIT_RULES
}

def mt_quantities(text, language):
    """Extract normalized quantities; this is not semantic validation."""
    try:
        from recognizers_number import (
            recognize_number, recognize_ordinal, recognize_percentage,
        )
    except ImportError as exc:
        raise RuntimeError(
            'Install the shared translation dependencies before using review checks.'
        ) from exc
    if language not in MT_CULTURE:
        raise ValueError("Choose language='en' or 'zh'.")
    text = unicodedata.normalize('NFKC', text).replace('−', '-')
    # A hyphen attached to the preceding word/digit is a separator, not a
    # unary negative sign. Preserve standalone negatives such as "is -12 V".
    text = re.sub(r'\b[A-Za-z]+-\d+(?:-[A-Za-z0-9]+)*\b', ' ', text)
    text = re.sub(r'(?<=\w)[-–—](?=\w)', ' ', text)
    text = re.sub(r'\bB\.C\.(?:E\.?)?', 'BCE', text, flags=re.I)

    if language == 'en':
        text = re.sub(
          r'\b(0?[1-9]|1[0-2])[.:]([0-5]\d)\s*([ap])\.?m\.?(?!\w)',
          r'\1:\2 \3m', text, flags=re.I,
        )

        # Handle common month/day forms without changing modal "may".
        months = '|'.join(name for name in MT_MONTHS if name != 'May')
        text = re.sub(
            r'\b(' + months + r')\b',
            lambda m: MT_MONTHS[m[0]], text,
        )
        text = re.sub(r'(\d+)(?:st|nd|rd|th)\b', r'\1', text)
        text = re.sub(r'\bMay(?=\s+\d)', '5', text)
        text = re.sub(r'(?<=\d)\s+May\b', ' 5', text)

    # 9,000 and 9 000 both become 9000, including before Chinese 万/亿.
    text = re.sub(
        r'(?<!\d)\d{1,3}(?:[, ]\d{3})+(?!\d)',
        lambda m: re.sub(r'[, ]', '', m.group()), text,
    )
    for pattern, code, _, _ in MT_UNIT_RULES:
        text = re.sub(pattern, f' <{code}> ', text, flags=re.I)

    culture = MT_CULTURE[language]
    matches = (
        [('percent', r) for r in recognize_percentage(text, culture)]
        + [('number', r) for r in recognize_number(text, culture)]
        + [('ordinal', r) for r in recognize_ordinal(text, culture)]
    )

    # Prefer complete expressions, avoiding counting both "50%" and "50".
    matches.sort(key=lambda item: -(item[1].end - item[1].start))
    quantities, covered = set(), set()

    for kind, result in matches:
        span = set(range(result.start, result.end + 1))
        if covered & span:
            continue
        covered.update(span)

        # Alphanumeric identifiers are outside this quantity check.
        if (
            re.search(r'[A-Za-z]$', text[:result.start])
            or re.match(r'[A-Za-z]', text[result.end + 1:])
        ):
            continue

        value = Decimal(result.resolution['value'].rstrip('%'))

        if kind == 'ordinal' and (
            value <= 0
            or re.search(r'\bper\s+$', text[:result.start], re.I)
        ):
            continue

        # Avoid treating isolated numeral characters inside words as numbers.
        if (
            language == 'zh'
            and not re.search(r'\d', result.text)
            and len(result.text) == 1
            and text.strip() != result.text
            and not re.match(
                r'\s*(?:<|[人名个位只张本条岁倍件台座种例家枚边面方时点年月日天])',
                text[result.end + 1:],
            )
        ):
            continue

        unit = ''
        if kind == 'percent':
            value /= 100
        else:
            suffix = re.match(
                r'\s*(?:a\s+)?<([A-Z]+)>',
                text[result.end + 1:],
            )
            if suffix:
                unit, scale = MT_UNIT_VALUES[suffix[1]]
                value *= scale

        # Unqualified "one"/一 often acts as an article rather than a quantity.
        if value == 1 and not unit:
            continue

        # Repeating a quantity in different wording is not itself an error.
        quantities.add((format(value.normalize(), 'f'), unit))

    return quantities

def mt_has_repetition(text):
    # Use Chinese characters and English words to find repeated phrases.
    tokens = re.findall(
        r'[\u4e00-\u9fff]|[a-z0-9]+',
        unicodedata.normalize('NFKC', text).casefold(),
    )

    for width in range(1, min(8, len(tokens) // 3) + 1):
        for start in range(len(tokens) - 3 * width + 1):
            phrase = tokens[start:start + width]
            if (
                tokens[start:start + 3 * width] == phrase * 3
                and sum(map(len, phrase)) * 3 >= 6
            ):
                return True

    return False

def mt_check_translation(source, prediction, source_language):
    """Use source text and model output only; never reference translations."""
    target_language = {'en': 'zh', 'zh': 'en'}[source_language]

    before = mt_quantities(source, source_language)
    after = mt_quantities(prediction, target_language)
    missing, added = before - after, after - before

    flags = []
    if not prediction.strip():
        flags.append('empty_output')

    if mt_has_repetition(prediction) and not mt_has_repetition(source):
        flags.append('repetition')

    if missing or added:
        flags.append('quantity_mismatch')

    return dict(
        flags=flags,
        flagged=bool(flags),
        missing_quantities=sorted(missing),
        added_quantities=sorted(added),
    )

def mt_check_decoded(source, decoded, source_language):
    """Combine reference-free text checks with decoder truncation metadata."""
    checked = dict(mt_check_translation(source, decoded["prediction"], source_language))
    flags = list(checked.get("flags", []))
    if decoded.get("hit_output_limit", False) and "output_limit" not in flags:
        flags.append("output_limit")
    checked.update(flags=flags, flagged=bool(flags))
    return checked


In [28]:
# Cell 29 · NLLB model and decoder

import gc
import json
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

NLLB_MODEL_ID = 'facebook/nllb-200-distilled-600M'
NLLB_REVISION = 'f8d333a098d19b4fd9a8b18f94170487ad3f821d'
NLLB_LANGUAGES = {'en': 'eng_Latn', 'zh': 'zho_Hans'}


def load_nllb():
    if callable(globals().get('release_routed_mt')):
        release_routed_mt()
    else:
        release_mt()
    if callable(globals().get('release_ocr')):
        release_ocr()

    tokenizer = AutoTokenizer.from_pretrained(
        NLLB_MODEL_ID, revision=NLLB_REVISION,
        src_lang='eng_Latn', legacy_behaviour=False,
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB_MODEL_ID, revision=NLLB_REVISION,
        dtype=torch.float32,
    ).to(MT_DEVICE).eval()

    return dict(model=model, tokenizer=tokenizer, model_id=NLLB_MODEL_ID, revision=NLLB_REVISION)


@torch.inference_mode()
def nllb_decode_loaded(text, source_language, loaded, beams):
    if source_language not in NLLB_LANGUAGES or beams not in MT_BENCH_BEAMS:
        raise ValueError('Choose en/zh and one of MT_BENCH_BEAMS.')
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Source text must be a non-empty string.')
    model, tokenizer = loaded['model'], loaded['tokenizer']
    device = next(model.parameters()).device
    target_language = {'en': 'zh', 'zh': 'en'}[source_language]

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    started = time.perf_counter()

    # NLLB uses language tokens instead of Marian's Chinese text prefix.
    tokenizer.src_lang = NLLB_LANGUAGES[source_language]
    inputs = tokenizer(text.strip(), return_tensors='pt', truncation=False)
    input_tokens = inputs['input_ids'].shape[1]
    limit = min(MT_BENCH_INPUT_LIMIT, model.config.max_position_embeddings)

    if input_tokens > limit:
        raise ValueError(f'Input has {input_tokens} tokens; limit is {limit}.')

    generated = model.generate(
        **inputs.to(device),
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(
            NLLB_LANGUAGES[target_language]
        ),
        max_length=None,
        max_new_tokens=MT_BENCH_OUTPUT_LIMIT,
        num_beams=beams,
        num_return_sequences=1,
        do_sample=False,
        early_stopping=(beams > 1),
        length_penalty=1.0,
        use_cache=True,
    )
    token_ids = generated[0].tolist()
    prediction = tokenizer.decode(
        token_ids, skip_special_tokens=True
    ).strip()

    if device.type == 'cuda':
        torch.cuda.synchronize(device)

    return {
        'prediction': prediction,
        'seconds': time.perf_counter() - started,
        'input_tokens': input_tokens,
        'output_tokens': len(token_ids) - 1,
        'hit_output_limit': len(token_ids) - 1 >= MT_BENCH_OUTPUT_LIMIT,
    }


In [29]:
# Cell 30 · Fixed translation routes

import time
import re

MT_ROUTE_POLICY = 'direction_fixed_review_only_v1'
MT_ROUTE_BEAMS = 4
# Preserve the cache if this definition cell is rerun.
_ROUTE_NLLB = globals().get('_ROUTE_NLLB', None)


def release_routed_mt():
    global _ROUTE_NLLB
    if _ROUTE_NLLB is not None:
        _ROUTE_NLLB['model'].cpu()
    _ROUTE_NLLB = None
    release_mt()


def _load_route(language, keep_ocr=True):
    global _ROUTE_NLLB
    if language == 'en':
        if _ROUTE_NLLB is not None:
            _ROUTE_NLLB['model'].cpu()
            _ROUTE_NLLB = None
        return load_mt('en', keep_ocr=keep_ocr)
    if language != 'zh':
        raise ValueError("Choose source_language='en' or 'zh'.")
    release_mt()
    if not keep_ocr and callable(globals().get('release_ocr')):
        release_ocr()
    if _ROUTE_NLLB is None:
        tokenizer = AutoTokenizer.from_pretrained(
            NLLB_MODEL_ID, revision=NLLB_REVISION,
            src_lang='zho_Hans', legacy_behaviour=False)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            NLLB_MODEL_ID, revision=NLLB_REVISION,
            dtype=torch.float32).to(MT_DEVICE).eval()
        _ROUTE_NLLB = dict(model=model, tokenizer=tokenizer,
                          model_id=NLLB_MODEL_ID, revision=NLLB_REVISION)
    return _ROUTE_NLLB


def mt_display_text(raw, target_language, enabled=False):
    """Optional display-only punctuation. Never add a sentence ending."""
    if not enabled or target_language != 'zh-Hans':
        return raw
    # Protect entire URL-like spans BEFORE changing any punctuation.
    # Whitespace-delimited protection is deliberately conservative: a trailing
    # URL punctuation mark is left unchanged rather than risking a broken link.
    urls = re.compile(r'(?:[A-Za-z][A-Za-z0-9+.-]*://|www\.|'
                      r'(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}/)\S*', re.I)
    def format_piece(piece, is_final=False):
        piece = re.sub(r'(?<=[\u4e00-\u9fff])[,;:!?]',
                       lambda m: {',':'，',';':'；',':':'：','!':'！','?':'？'}[m[0]], piece)
        if is_final:
            # Convert only an EXISTING terminal period; never append one.
            piece = re.sub(r'(?<=[\u4e00-\u9fff])\.(?=\s*$)', '。', piece)
        return piece
    parts=[]
    end=0
    for match in urls.finditer(raw):
        parts.extend((format_piece(raw[end:match.start()]), match[0]))
        end=match.end()
    parts.append(format_piece(raw[end:], is_final=True))
    return ''.join(parts)


def translate_routed(text, source_language, *, keep_ocr=True,
                     display_punctuation=False):
    if source_language not in ('en', 'zh'):
        raise ValueError("Choose source_language='en' or 'zh'.")
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Enter non-empty source text.')
    for name in ('load_mt', 'mt_decode_loaded', 'nllb_decode_loaded',
                 'mt_check_translation', 'mt_check_decoded'):
        if not callable(globals().get(name)):
            raise RuntimeError(f'Missing {name}: run the translation definitions first.')
    source = text.strip()
    started = time.perf_counter()
    loaded = _load_route(source_language, keep_ocr)
    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()
    load_seconds = time.perf_counter() - started
    started = time.perf_counter()
    decode = mt_decode_loaded if source_language == 'en' else nllb_decode_loaded
    decoded = decode(source, source_language, loaded, MT_ROUTE_BEAMS)
    raw = decoded['prediction']
    checked = mt_check_decoded(source, decoded, source_language)
    flags = checked['flags']
    seconds = time.perf_counter() - started
    target = 'zh-Hans' if source_language == 'en' else 'en'
    decoded = dict(decoded, seconds=seconds)
    return dict(**decoded, source_text=source, source_language=source_language,
                target_language=target, translation=raw,
                display_prediction=mt_display_text(raw, target, display_punctuation),
                model_id=loaded['model_id'], revision=loaded.get('revision'),
                translation_seconds=seconds, load_seconds=load_seconds,
                selected_beams=MT_ROUTE_BEAMS, retried=False,
                needs_review=bool(flags), checks=checked,
                output_limit_reached=bool(decoded.get('hit_output_limit', False)),
                policy=MT_ROUTE_POLICY,
                review_status=('REVIEW NEEDED: '+', '.join(flags) if flags
                               else 'No heuristic flags; correctness is not guaranteed'))


In [30]:
# Cell 31 · Route and display regression tests

from unittest.mock import patch

def test_translation_routes():
    calls = []
    def loader(lang, keep_ocr=True):
        return dict(model_id='Marian' if lang=='en' else 'NLLB', revision='test')
    def decode(text, lang, loaded, beams):
        calls.append((lang, loaded['model_id'], beams))
        return dict(prediction='中文, 3.14' if lang=='en' else 'English text.',
                    seconds=0.0, input_tokens=4, output_tokens=4,
                    hit_output_limit=(lang=='zh'))
    with patch.dict(globals(), dict(_load_route=loader, mt_decode_loaded=decode,
            nllb_decode_loaded=decode, load_mt=loader, MT_DEVICE='cpu',
            mt_check_translation=lambda *args: dict(flags=[], flagged=False))):
        a=translate_routed('English input','en',display_punctuation=True)
        b=translate_routed('中文输入','zh')
        assert calls == [('en','Marian',4),('zh','NLLB',4)]
        assert a['prediction']=='中文, 3.14' and a['display_prediction']=='中文， 3.14'
        assert a['translation']==a['prediction'] and not a['retried']
        assert b['needs_review'] and b['checks']['flags']==['output_limit']
        assert b['retried'] is False and len(calls)==2
        for text,lang in [('', 'en'), ('x','fr')]:
            try: translate_routed(text,lang)
            except ValueError: pass
            else: raise AssertionError('Invalid input accepted')
    assert mt_display_text('https://example.com/中文,foo','zh-Hans',True)=='https://example.com/中文,foo'
    assert mt_display_text('主题.','zh-Hans',True)=='主题。'
    assert mt_display_text('未完的句子','zh-Hans',True)=='未完的句子'
    assert mt_display_text('中文,','zh-Hans',False)=='中文,'
    assert mt_display_text('v1.2: https://a.b','en',True)=='v1.2: https://a.b'
    print('Route selection, no retries, truncation, raw/display and input tests passed.')

def test_route_cache():
    from types import SimpleNamespace
    events=[]
    class FakeModel:
        def cpu(self): events.append('nllb_cpu'); return self
        def to(self,device): return self
        def eval(self): return self
    class Factory:
        @staticmethod
        def from_pretrained(*args,**kwargs):
            events.append('nllb_loaded'); return FakeModel()
    fake_tokenizer=SimpleNamespace(from_pretrained=lambda *a,**k:object())
    with patch.dict(globals(), dict(_ROUTE_NLLB=None,
            release_mt=lambda:events.append('release_marian'),
            release_ocr=lambda:events.append('release_ocr'),
            load_mt=lambda lang,keep_ocr=True:dict(model_id='Marian'),
            AutoTokenizer=fake_tokenizer,AutoModelForSeq2SeqLM=Factory,
            NLLB_MODEL_ID='test',NLLB_REVISION='test',MT_DEVICE='cpu',
            torch=SimpleNamespace(float32='float32'))):
        first=_load_route('zh',True)
        assert _load_route('zh',True) is first
        _load_route('en',True)
        assert events.count('nllb_loaded')==1
        assert events.count('nllb_cpu')==1
        assert 'release_ocr' not in events
        _load_route('zh',False)
        assert events.count('nllb_loaded')==2 and events.count('release_ocr')==1
    print('NLLB cache reuse, direction switching and OCR-preservation tests passed.')

if RUN_MT_ROUTE_TESTS:
    test_translation_routes()
    test_route_cache()


Route selection, no retries, truncation, raw/display and input tests passed.
NLLB cache reuse, direction switching and OCR-preservation tests passed.


In [31]:
# Cell 32 · Review-check regression tests

def mt_test_review_checks():
    mt_check_examples = [
        ('1.5 million people', '150万人', 'en', []),
        ('0.5 kg', '500克', 'en', []),
        ('twelve volts', '十二伏特', 'en', []),
        ('12V', '12伏', 'en', []),
        ('50%', '百分之五十', 'en', []),
        ('October 7', '10月7日', 'en', []),
        ('He wrote a book.', '他写了一本书。', 'en', []),
        ('预计有9000人参加。', '900,000 people are expected.',
         'zh', ['quantity_mismatch']),
        ('Voltage is 12 V.', '电压是21伏。',
         'en', ['quantity_mismatch']),
        ('Please close the door.', '', 'en', ['empty_output']),
        ('Please close the door.', '请关门请关门请关门',
         'en', ['repetition']),
    ]

    # Equivalent time formats should pass; different times should remain flagged.
    mt_check_examples += [
        ("Meet at 10 o'clock.", "十点见。", "en", []),
        ("十点见。", "Meet at 10 o'clock.", "zh", []),
        ("上午9:30。", "At 9.30 a.m.", "zh", []),
        ("上午9:30。", "At 9.30am.", "zh", []),
        (
            "上午9:30。", "At 9.45 a.m.", "zh",
            ["quantity_mismatch"],
        ),
        (
            "Meet at 10 o'clock.", "九点见。", "en",
            ["quantity_mismatch"],
        ),
        (
            "The price is 9.30 dollars.", "价格是9.30美元。",
            "en", [],
        ),
        (
            "The price is 9.30 dollars.", "价格是9.03美元。",
            "en", ["quantity_mismatch"],
        ),
    ]

    for source, prediction, language, expected in mt_check_examples:
        actual = mt_check_translation(source, prediction, language)['flags']
        assert actual == expected, (source, prediction, actual, expected)
    print(f'{len(mt_check_examples)} review checks passed.')
    return len(mt_check_examples)

if RUN_MT_CHECK_TESTS:
    mt_test_review_checks()


In [32]:
# Cell 33 · Optional typed translation checks

MT_EXAMPLES = [
    ('en', 'Computer vision is an interesting subject.'),
    ('en', 'Please close the door.'),
    ('zh', '计算机视觉是一门有趣的学科。'),
    ('zh', '请关门。'),
]


def mt_run_typed_checks():
    mt_results = []


    try:
        for source_language, sentence in MT_EXAMPLES:
            result = translate_text(sentence, source_language)
            mt_results.append(result)

            print(
                f"\n{result['source_language']} "
                f"-> {result['target_language']}"
            )
            print('Input: ', result['source_text'])
            print('Output:', result['translation'])
            print(
                f"Translation time: {result['translation_seconds']:.3f}s "
                '(loading excluded)'
            )

            if result['output_limit_reached']:
                print(
                    'Output reached the token limit; '
                    'it may be incomplete.'
                )
    finally:
        # Leave GPU memory available for the existing final OCR interface.
        release_mt()

    # Save outputs and settings so this initial run can be reviewed later.
    record = dict(
        purpose=(
            'Part C step 1: functional translation checks, '
            'not benchmark scores'
        ),
        models=MT_MODELS,
        device=MT_DEVICE,
        hardware=(
            torch.cuda.get_device_name(0)
            if MT_DEVICE == 'cuda' else 'CPU'
        ),
        transformers_version=transformers.__version__,
        torch_version=str(torch.__version__),
        decoding=dict(
            num_beams=MT_NUM_BEAMS,
            max_new_tokens=MT_MAX_NEW_TOKENS,
            do_sample=False,
        ),
        results=mt_results,
    )

    result_path = Path(RESULTS_DIR) / f'mt_step1_{time.time_ns()}.json'
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(
        json.dumps(record, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print('\nSaved:', result_path)
    return record, result_path

if RUN_MT_TYPED_CHECKS:
    MT_TYPED_RECORD, MT_TYPED_PATH = mt_run_typed_checks()


In [33]:
# Cell 34 · Optional NLLB typed checks

def mt_run_nllb_typed_checks():
    nllb_loaded = load_nllb()
    nllb_results = []

    try:
        for source_language, sentence in MT_EXAMPLES:
            result = nllb_decode_loaded(
                sentence, source_language, nllb_loaded, beams=4
            )
            nllb_results.append({
                'direction': 'en→zh' if source_language == 'en' else 'zh→en',
                'source': sentence,
                **result,
            })

        nllb_record = {
            'purpose': 'Functional checks only; not a quality or latency benchmark',
            'model_id': NLLB_MODEL_ID,
            'revision': NLLB_REVISION,
            'language_codes': NLLB_LANGUAGES,
            'precision': str(next(nllb_loaded['model'].parameters()).dtype),
            'device': MT_DEVICE,
            'transformers_version': transformers.__version__,
            'torch_version': str(torch.__version__),
            'decoding': {
                'num_beams': 4,
                'max_input_tokens': MT_BENCH_INPUT_LIMIT,
                'max_new_tokens': MT_BENCH_OUTPUT_LIMIT,
                'do_sample': False,
            },
            'results': nllb_results,
        }
    finally:
        nllb_loaded['model'].cpu()
        del nllb_loaded
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    nllb_check_path = Path(RESULTS_DIR) / f'nllb_checks_{time.time_ns()}.json'
    nllb_check_path.write_text(
        json.dumps(nllb_record, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    with pd.option_context('display.max_colwidth', None):
        display(pd.DataFrame(nllb_results)[[
            'direction', 'source', 'prediction', 'hit_output_limit'
        ]])

    print('Saved:', nllb_check_path)
    print('Check that all four outputs are non-empty and in the target language.')

    return nllb_record, nllb_check_path

if RUN_NLLB_TYPED_CHECKS:
    NLLB_TYPED_RECORD, NLLB_TYPED_PATH = mt_run_nllb_typed_checks()


In [34]:
# Cell 35 · FLORES development data helpers

import hashlib
import random

MT_DATA_READY = False
MT_DEV = {}
MT_DATA_RECORD = None
MT_DATASET = 'openlanguagedata/flores_plus'
MT_DATA_REVISION = '1147e2dc59f37fbb5c22fd8478770de02cd399f9'
MT_DEV_SIZE = 200
MT_DATA_SEED = 42
MT_DATA_DIR = MT_DEV_CACHE = None

def mt_data_hash(pairs):
    raw = json.dumps(pairs, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()

def mt_read_reference_file(path, language, script):
    # Match by ID, never by assuming the two files have the same row order.
    rows = {}
    for line in Path(path).read_text(encoding='utf-8-sig').splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        sample_id = str(row['id'])
        text = row['text']

        if row.get('split', 'dev') != 'dev':
            raise ValueError('Unexpected split: only dev is allowed in this loader.')
        if (row['iso_639_3'], row['iso_15924']) != (language, script):
            raise ValueError(f'Unexpected language/script in {Path(path).name}.')
        if row['id'] is None or not sample_id.strip() or sample_id in rows:
            raise ValueError(f'Missing or duplicate sentence ID: {sample_id!r}')
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f'Empty reference text for sentence {sample_id}.')

        rows[sample_id] = text
    return rows

def mt_prepare_dev():
    global MT_DATA_DIR, MT_DEV_CACHE
    from getpass import getpass
    from huggingface_hub import get_token, hf_hub_download
    from huggingface_hub.errors import HfHubHTTPError

    if 'PROJECT_DIR' not in globals() or not Path(PROJECT_DIR).is_dir():
        raise RuntimeError('Prepare the project folder before loading FLORES+.')
    MT_DATA_DIR = Path(PROJECT_DIR) / 'data' / 'translation' / 'flores_plus'
    MT_DATA_DIR.mkdir(parents=True, exist_ok=True)
    MT_DEV_CACHE = MT_DATA_DIR / (
        f'dev_{MT_DEV_SIZE}_seed{MT_DATA_SEED}_{MT_DATA_REVISION[:12]}.json'
    )
    settings = dict(
        dataset=MT_DATASET, revision=MT_DATA_REVISION, split='dev',
        languages={'en': 'eng_Latn', 'zh': 'cmn_Hans'},
        subset_size=MT_DEV_SIZE, seed=MT_DATA_SEED,
    )

    if MT_DEV_CACHE.exists():
        payload = json.loads(MT_DEV_CACHE.read_text(encoding='utf-8'))
        if payload['settings'] != settings:
            raise ValueError('Saved dataset settings differ; inspect the cache.')
        print('Reusing the saved development subset.')
    else:
        # Use an existing HF login/Colab secret, or a hidden token prompt.
        # The token stays inside this function and is never saved to Drive.
        token = get_token() or getpass('Hugging Face read token (hidden): ').strip()
        if not token:
            raise ValueError('A token from an account with FLORES+ access is required.')

        try:
            english_path = hf_hub_download(
                MT_DATASET, 'dev/eng_Latn.jsonl', repo_type='dataset',
                revision=MT_DATA_REVISION, token=token,
            )
            chinese_path = hf_hub_download(
                MT_DATASET, 'dev/cmn_Hans.jsonl', repo_type='dataset',
                revision=MT_DATA_REVISION, token=token,
            )
        except HfHubHTTPError as exc:
            if getattr(exc.response, 'status_code', None) in (401, 403):
                raise RuntimeError(
                    'FLORES+ access was denied. Accept its access conditions on '
                    'Hugging Face, then use a read token from that same account. '
                    'If using the HF_TOKEN Colab secret, check its permissions.'
                ) from None
            raise
        finally:
            del token

        english = mt_read_reference_file(english_path, 'eng', 'Latn')
        chinese = mt_read_reference_file(chinese_path, 'cmn', 'Hans')

        if set(english) != set(chinese):
            raise ValueError('English and Chinese sentence IDs do not match.')
        if len(english) < MT_DEV_SIZE:
            raise ValueError(
                f'Only {len(english)} pairs exist; requested {MT_DEV_SIZE}.'
            )

        # Choose once, independently of model predictions or translation quality.
        selected_ids = random.Random(MT_DATA_SEED).sample(
            sorted(english), MT_DEV_SIZE
        )
        pairs = [
            dict(id=i, en=english[i], zh=chinese[i])
            for i in selected_ids
        ]
        payload = dict(
            settings=settings,
            available_dev_pairs=len(english),
            purpose='Development evaluation only; not model training',
            license='CC-BY-SA-4.0',
            pairs=pairs,
            sha256=mt_data_hash(pairs),
        )

        # Save the selected references so later runs reuse the same examples.
        temporary = MT_DEV_CACHE.with_suffix('.tmp')
        temporary.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        temporary.replace(MT_DEV_CACHE)

    pairs = payload['pairs']
    if mt_data_hash(pairs) != payload['sha256']:
        raise ValueError('Saved reference data changed; inspect the cache.')
    if len(pairs) != MT_DEV_SIZE or len({r['id'] for r in pairs}) != len(pairs):
        raise ValueError('Unexpected subset size or duplicate sentence IDs.')

    return payload


In [35]:
# Cell 36 · Optional FLORES data loading

def mt_load_dev():
    global MT_DATA_RECORD, MT_DEV, MT_DATA_READY
    MT_DATA_READY = False
    MT_DATA_RECORD = mt_prepare_dev()


    # The same sentence pairs support evaluation in both translation directions.
    MT_DEV = {
        source: [
            dict(id=row['id'], source=row[source], reference=row[target])
            for row in MT_DATA_RECORD['pairs']
        ]
        for source, target in (('en', 'zh'), ('zh', 'en'))
    }
    MT_DATA_READY = True

    print('Available aligned dev pairs:', MT_DATA_RECORD['available_dev_pairs'])
    print('Selected pairs:', len(MT_DATA_RECORD['pairs']))
    print('English -> Chinese cases:', len(MT_DEV['en']))
    print('Chinese -> English cases:', len(MT_DEV['zh']))
    print('Reference fingerprint:', MT_DATA_RECORD['sha256'][:16])
    print('Cache:', MT_DEV_CACHE)
    print('devtest was not downloaded or evaluated. No model training was performed.')
    return MT_DATA_RECORD

if RUN_MT_DATA_LOAD:
    mt_load_dev()


In [36]:
# Cell 37 · Resumable benchmark checkpoints

def mt_checkpoint_save(path, signature, rows):
    import os
    destination = Path(path)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    temporary.write_text(json.dumps(dict(signature=signature, rows=rows),
        ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(temporary, destination)


def mt_checkpoint_load(path, signature, expected):
    path = Path(path)
    if not path.exists():
        completed = path.with_name(path.name.replace('.partial.json', '.json'))
        if completed.exists():
            raise ValueError(f'Legacy result has no compatible partial checkpoint: {completed}. Choose a new run folder.')
        return []
    record = json.loads(path.read_text(encoding='utf-8'))
    signature = json.loads(json.dumps(signature))
    if record.get('signature') != signature:
        raise ValueError(f'Incompatible checkpoint {path}; start a new run folder.')
    rows = record.get('rows', [])
    actual = [{k:r[k] for k in ('id','source','reference')} for r in rows]
    if len(rows) > len(expected) or actual != expected[:len(rows)]:
        raise ValueError(f'Checkpoint is not an exact reference prefix: {path}')
    return rows


def mt_checkpoint_signature(loaded, language, strategy):
    model = loaded['model']
    parameter = next(model.parameters())
    device = parameter.device
    return dict(schema=1, reference_sha256=MT_DATA_RECORD['sha256'],
        dataset=MT_DATA_RECORD['settings'], language=language, strategy=strategy,
        model_id=loaded.get('model_id', getattr(model.config, '_name_or_path', '')),
        revision=loaded.get('revision', getattr(model.config, '_commit_hash', None)),
        generation=model.generation_config.to_dict(), precision=str(parameter.dtype),
        hardware=torch.cuda.get_device_name(device) if device.type=='cuda' else 'CPU',
        torch=str(torch.__version__), transformers=transformers.__version__,
        cpu_threads=torch.get_num_threads(),
        repeats=MT_BENCH_REPEATS, warmups=MT_BENCH_WARMUPS,
        input_limit=MT_BENCH_INPUT_LIMIT, output_limit=MT_BENCH_OUTPUT_LIMIT,
        checks=None, controller=None)


def mt_prepare_run_folder(resume_dir, default_dir):
    folder = Path(resume_dir or default_dir)
    if resume_dir and folder.exists() and not list(folder.glob('*.partial.json')) and list(folder.glob('*.json')):
        raise ValueError('This folder contains legacy results without resumable checkpoints. Keep them for analysis and choose a new run folder.')
    folder.mkdir(parents=True, exist_ok=True)
    return folder


In [37]:
# Cell 38 · Optional Marian benchmark

def mt_run_fixed_baselines():
    import sacrebleu
    from sacrebleu.metrics import BLEU, CHRF

    if not globals().get('MT_DATA_READY', False):
        raise RuntimeError('Load FLORES+ development data first.')
    if not callable(globals().get('load_mt')):
        raise RuntimeError('Run the Marian model definitions first.')
    if MT_DATA_RECORD['settings']['split'] != 'dev':
        raise ValueError('This development benchmark must only use dev.')
    if MT_BENCH_REPEATS < 1 or MT_BENCH_WARMUPS < 1:
        raise ValueError('Repeats and warmups must be positive.')

    # Ensure both directions still use exactly the saved references and IDs.
    if mt_data_hash(MT_DATA_RECORD['pairs']) != MT_DATA_RECORD['sha256']:
        raise ValueError('The reference data changed; reload the FLORES+ development data.')

    for source, target in (('en', 'zh'), ('zh', 'en')):
        expected = [
            dict(id=r['id'], source=r[source], reference=r[target])
            for r in MT_DATA_RECORD['pairs']
        ]
        if not expected or MT_DEV[source] != expected:
            raise ValueError('Benchmark rows differ from the saved subset.')

    folder = mt_prepare_run_folder(globals().get('MT_BASELINE_RESUME_DIR'), Path(RESULTS_DIR) / 'mt_baselines' / str(time.time_ns()))
    folder.mkdir(parents=True, exist_ok=True)
    print('Results folder:', folder)

    summaries = []
    if callable(globals().get('release_routed_mt')):
        release_routed_mt()
    else:
        release_mt()

    try:
        for language in ('en', 'zh'):
            loaded = load_mt(language)
            model = loaded['model']
            device = next(model.parameters()).device
            rows = MT_DEV[language]

            # Fail clearly on overlong inputs; never truncate or drop rows.
            limit = min(
                MT_BENCH_INPUT_LIMIT, model.config.max_position_embeddings
            )
            for row in rows:
                prepared = row['source'].strip()
                if language == 'en':
                    prepared = f'{MT_ZH_TAG} {prepared}'

                count = len(
                    loaded['tokenizer'](
                        prepared, truncation=False
                    )['input_ids']
                )
                if count > limit:
                    raise ValueError(
                        f"{language}/{row['id']}: "
                        f"{count} tokens exceeds {limit}."
                    )

            for beams in MT_BENCH_BEAMS:
                for i in range(MT_BENCH_WARMUPS):
                    mt_decode_loaded(
                        rows[i % len(rows)]['source'],
                        language, loaded, beams
                    )

                if device.type == 'cuda':
                    torch.cuda.reset_peak_memory_stats(device)

                checkpoint = folder / f'{language}_beam{beams}.partial.json'
                signature = mt_checkpoint_signature(loaded, language, f'beam{beams}')
                predictions = mt_checkpoint_load(checkpoint, signature, rows)
                resumed_count = len(predictions)
                timings = [t for r in predictions for t in r['latency_seconds']]

                for i, row in enumerate(rows, 1):
                    if i <= resumed_count:
                        continue
                    trials = [
                        mt_decode_loaded(
                            row['source'], language, loaded, beams
                        )
                        for _ in range(MT_BENCH_REPEATS)
                    ]
                    first = trials[0]
                    times = [trial['seconds'] for trial in trials]
                    timings.extend(times)

                    predictions.append(dict(
                        id=row['id'],
                        source=row['source'],
                        reference=row['reference'],
                        prediction=first['prediction'],
                        latency_seconds=times,
                        input_tokens=first['input_tokens'],
                        output_tokens=first['output_tokens'],
                        hit_output_limit=any(
                            t['hit_output_limit'] for t in trials
                        ),
                        repeat_output_changed=any(
                            t['prediction'] != first['prediction']
                            for t in trials[1:]
                        ),
                    ))

                    mt_checkpoint_save(checkpoint, signature, predictions)

                    if i % 25 == 0 or i == len(rows):
                        print(
                            f'{language} | beam {beams} | '
                            f'{i}/{len(rows)} sentences'
                        )

                # Score each sentence once; repeated calls are only for timing.
                refs = [r['reference'] for r in predictions]
                hyps = [r['prediction'] for r in predictions]

                bleu = BLEU(
                    tokenize='zh' if language == 'en' else '13a'
                )
                chrf = CHRF(char_order=6, word_order=0, beta=2)

                summary = dict(
                    direction='en→zh' if language == 'en' else 'zh→en',
                    beams=beams,
                    samples=len(rows),
                    resumed_samples=resumed_count,
                    latency_scope=('multiple sessions; descriptive only' if resumed_count else 'single session'),
                    BLEU=bleu.corpus_score(hyps, [refs]).score,
                    chrF=chrf.corpus_score(hyps, [refs]).score,
                    median_seconds=float(np.median(timings)),
                    p95_seconds=float(np.percentile(timings, 95)),
                    timing_calls=len(timings),
                    empty_outputs=sum(not h for h in hyps),
                    output_limit_hits=sum(
                        r['hit_output_limit'] for r in predictions
                    ),
                    changed_on_repeat=sum(
                        r['repeat_output_changed'] for r in predictions
                    ),
                )

                record = dict(
                    schema_version=1,
                    dataset=MT_DATA_RECORD['settings'],
                    reference_sha256=MT_DATA_RECORD['sha256'],
                    model_id=loaded['model_id'],
                    revision=loaded['revision'],
                    precision=str(next(model.parameters()).dtype),
                    device=str(device),
                    hardware=(
                        torch.cuda.get_device_name(device)
                        if device.type == 'cuda' else 'CPU'
                    ),
                    versions=dict(
                        python=sys.version.split()[0],
                        torch=str(torch.__version__),
                        transformers=transformers.__version__,
                        sacrebleu=sacrebleu.__version__,
                    ),
                    cpu_threads=torch.get_num_threads(),
                    generation_defaults=model.generation_config.to_dict(),
                    decoding=dict(
                        num_beams=beams,
                        num_return_sequences=1,
                        max_new_tokens=MT_BENCH_OUTPUT_LIMIT,
                        max_length=None,
                        do_sample=False,
                        early_stopping=(beams > 1),
                        length_penalty=1.0,
                        use_cache=True,
                    ),
                    effective_input_limit=limit,
                    chinese_target_tag=MT_ZH_TAG,
                    timing=dict(
                        repeats=MT_BENCH_REPEATS,
                        warmups=MT_BENCH_WARMUPS,
                        batch_size=1,
                        includes=(
                            'tokenization, device transfer, '
                            'generation, decoding'
                        ),
                        excludes=(
                            'model loading, warmup, metrics and file saving'
                        ),
                    ),
                    peak_gpu_allocated_mb=(
                        torch.cuda.max_memory_allocated(device) / 1024**2
                        if device.type == 'cuda' else None
                    ),
                    metric_signatures=dict(
                        BLEU=str(bleu.get_signature()),
                        chrF=str(chrf.get_signature()),
                    ),
                    COMET=None,
                    COMET_status='Pending separate scoring step',
                    summary=summary,
                    predictions=predictions,
                )

                destination = folder / f'{language}_beam{beams}.json'
                destination.write_text(
                    json.dumps(record, ensure_ascii=False, indent=2),
                    encoding='utf-8',
                )
                summaries.append(summary)
                print('Saved:', destination.name)

    finally:
        release_mt()

    table = pd.DataFrame(summaries)
    table.to_csv(folder / 'summary.csv', index=False)
    display(table.round(3))

    print('Development benchmark complete. Final-test data was not used.')
    return table, folder

if RUN_MT_BASELINES:
    MT_BASELINE_TABLE, MT_BASELINE_DIR = mt_run_fixed_baselines()


In [38]:
# Cell 39 · Saved beam comparison

def mt_compare_saved_beams(baseline_dir):
    from sacrebleu.metrics import CHRF
    if not baseline_dir:
        raise ValueError('Select a completed Marian benchmark folder.')
    mt_pair_rows = []
    mt_sentence_chrf = CHRF(char_order=6, word_order=0, beta=2)

    for language in ('en', 'zh'):
        records = [
            json.loads(
                (Path(baseline_dir) / f'{language}_beam{beams}.json')
                .read_text(encoding='utf-8')
            )
            for beams in (1, 4)
        ]
        fast, slow = records

        # Pair the same sentences; keep this analysis restricted to dev.
        assert fast['dataset'] == slow['dataset']
        assert fast['dataset']['split'] == 'dev'
        if fast['reference_sha256'] != slow['reference_sha256']:
            raise ValueError('Reference fingerprints differ.')
        for run in records:
            if not run['predictions'] or len(run['predictions']) != run['summary']['samples']:
                raise ValueError('Use complete saved result files.')

        for first, second in zip(
            fast['predictions'], slow['predictions'], strict=True
        ):
            assert all(
                first[k] == second[k]
                for k in ('id', 'source', 'reference')
            )

            # References are used for offline analysis only.
            scores = [
                mt_sentence_chrf.sentence_score(
                    row['prediction'], [row['reference']]
                ).score
                for row in (first, second)
            ]

            mt_pair_rows.append(dict(
                direction='en→zh' if language == 'en' else 'zh→en',
                id=first['id'],
                beam1_chrF=scores[0],
                beam4_chrF=scores[1],
                chrF_change=scores[1] - scores[0],
                same_output=first['prediction'] == second['prediction'],
                source=first['source'],
                reference=first['reference'],
                beam1_output_limit=bool(first.get('hit_output_limit', False)),
                beam4_output_limit=bool(second.get('hit_output_limit', False)),
                beam1=first['prediction'],
                beam4=second['prediction'],
            ))

    MT_PAIR_TABLE = pd.DataFrame(mt_pair_rows)

    MT_PAIR_SUMMARY = MT_PAIR_TABLE.groupby('direction', sort=False).agg(
        samples=('id', 'size'),
        higher_chrF=('chrF_change', lambda x: int((x > 1e-6).sum())),
        lower_chrF=('chrF_change', lambda x: int((x < -1e-6).sum())),
        tied_chrF=('chrF_change', lambda x: int((x.abs() <= 1e-6).sum())),
        identical_outputs=('same_output', 'sum'),
    )
    display(MT_PAIR_SUMMARY)

    review_columns = [
        'id', 'chrF_change', 'source', 'beam1', 'beam4', 'reference'
    ]

    for direction, group in MT_PAIR_TABLE.groupby('direction', sort=False):
        print(f'\n{direction}: two largest chrF increases and decreases')
        examples = pd.concat([
            group.nlargest(2, 'chrF_change'),
            group.nsmallest(2, 'chrF_change'),
        ])
        with pd.option_context('display.max_colwidth', 100):
            display(examples[review_columns].round(3))

    # Save full text so later inspection does not require generating it again.
    mt_pair_path = Path(baseline_dir) / 'sentence_comparison.csv'
    MT_PAIR_TABLE.to_csv(mt_pair_path, index=False, encoding='utf-8-sig')

    print('\nSaved:', mt_pair_path)
    print('Higher chrF means closer to the reference, not guaranteed correctness.')

    return MT_PAIR_TABLE, MT_PAIR_SUMMARY


In [39]:
# Cell 40 · Optional saved-output review

def mt_review_saved_flags(pair_table, baseline_dir):
    mt_check_rows = []

    for row in pair_table.to_dict('records'):
        language = row['direction'].split('→')[0]

        for beams in (1, 4):
            prediction = row[f'beam{beams}']

            # The check receives no reference text or chrF score.
            result = mt_check_decoded(
                row['source'], dict(prediction=prediction,
                    hit_output_limit=row.get(f'beam{beams}_output_limit', False)),
                language,
            )
            mt_check_rows.append(dict(
                direction=row['direction'],
                id=row['id'],
                beams=beams,
                source=row['source'],
                prediction=prediction,
                **result,
            ))

    MT_CHECK_TABLE = pd.DataFrame(mt_check_rows)

    MT_CHECK_SUMMARY = MT_CHECK_TABLE.groupby(['direction', 'beams']).agg(
        samples=('id', 'size'),
        flagged=('flagged', 'sum'),
        empty=('flags', lambda x: sum('empty_output' in f for f in x)),
        repetition=('flags', lambda x: sum('repetition' in f for f in x)),
        quantities=('flags', lambda x: sum('quantity_mismatch' in f for f in x)),
        output_limit=('flags', lambda x: sum('output_limit' in f for f in x)),
    )
    display(MT_CHECK_SUMMARY)

    # Inspect flagged outputs without changing predictions.
    for direction, group in MT_CHECK_TABLE.groupby('direction', sort=False):
        print(f'\n{direction}: flagged beam-1 examples')

        examples = group[
            group['flagged'] & group['beams'].eq(1)
        ].head(5)

        with pd.option_context('display.max_colwidth', 100):
            display(examples[[
                'id', 'source', 'prediction', 'flags',
                'missing_quantities', 'added_quantities',
            ]])

    mt_check_path = Path(baseline_dir) / 'review_only_quality_checks.json'
    mt_check_path.write_text(
        json.dumps(
            dict(check_version=MT_CHECK_VERSION, records=mt_check_rows),
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )

    print('\nSaved:', mt_check_path)
    print('Flags are suspected problems, not measured translation error rates.')
    return MT_CHECK_TABLE, MT_CHECK_SUMMARY

if RUN_MT_SAVED_REVIEW:
    MT_PAIR_TABLE, MT_PAIR_SUMMARY = mt_compare_saved_beams(MT_BASELINE_DIR)
    MT_CHECK_TABLE, MT_CHECK_SUMMARY = mt_review_saved_flags(
        MT_PAIR_TABLE, MT_BASELINE_DIR
    )


In [40]:
# Cell 41 · Optional NLLB benchmark

def mt_run_nllb_benchmark():
    import sacrebleu
    from sacrebleu.metrics import BLEU, CHRF

    if not MT_BASELINE_DIR:
        raise ValueError('Select a completed Marian baseline folder first.')
    if MT_BENCH_REPEATS < 1 or MT_BENCH_WARMUPS < 1:
        raise ValueError('Repeats and warmups must be positive.')
    if not MT_DATA_READY or MT_DATA_RECORD['settings']['split'] != 'dev':
        raise ValueError('Load FLORES+ development data first.')
    if mt_data_hash(MT_DATA_RECORD['pairs']) != MT_DATA_RECORD['sha256']:
        raise ValueError('Development data changed; reload the saved references.')

    baselines = {}
    for language, target in (('en', 'zh'), ('zh', 'en')):
        expected = [
            dict(id=r['id'], source=r[language], reference=r[target])
            for r in MT_DATA_RECORD['pairs']
        ]
        if MT_DEV[language] != expected:
            raise ValueError('Benchmark rows differ from the saved subset.')

        for beams in (1, 4):
            path = Path(MT_BASELINE_DIR) / f'{language}_beam{beams}.json'
            old = json.loads(path.read_text(encoding='utf-8'))
            old_rows = [
                {k: r[k] for k in ('id', 'source', 'reference')}
                for r in old['predictions']
            ]
            if (
                old['dataset'] != MT_DATA_RECORD['settings']
                or old['reference_sha256'] != MT_DATA_RECORD['sha256']
                or old_rows != expected
            ):
                raise ValueError('Marian used different data; rerun the Marian benchmark with matching settings.')
            baselines[language, beams] = old

    loaded = load_nllb()
    summaries = []

    try:
        model = loaded['model']
        device = next(model.parameters()).device
        environment = dict(
            precision=str(next(model.parameters()).dtype),
            hardware=(
                torch.cuda.get_device_name(device)
                if device.type == 'cuda' else 'CPU'
            ),
            cpu_threads=torch.get_num_threads(),
            versions=dict(
                python=sys.version.split()[0],
                torch=str(torch.__version__),
                transformers=transformers.__version__,
                sacrebleu=sacrebleu.__version__,
            ),
        )

        # Require matching conditions before comparing saved timings.
        for old in baselines.values():
            if any(old[k] != value for k, value in environment.items()):
                raise ValueError('Runtime differs from Marian; rerun the Marian benchmark with matching settings.')
            if (
                old['timing']['repeats'] != MT_BENCH_REPEATS
                or old['timing']['warmups'] != MT_BENCH_WARMUPS
                or old['effective_input_limit'] != MT_BENCH_INPUT_LIMIT
                or old['decoding']['max_new_tokens'] != MT_BENCH_OUTPUT_LIMIT
            ):
                raise ValueError('Benchmark settings changed; rerun the Marian benchmark with matching settings.')

        folder = mt_prepare_run_folder(globals().get('MT_NLLB_RESUME_DIR'), Path(RESULTS_DIR) / 'mt_nllb' / str(time.time_ns()))
        folder.mkdir(parents=True, exist_ok=True)
        print('Results folder:', folder)

        for language in ('en', 'zh'):
            rows = MT_DEV[language]

            for beams in (1, 4):
                for i in range(MT_BENCH_WARMUPS):
                    nllb_decode_loaded(
                        rows[i % len(rows)]['source'],
                        language, loaded, beams,
                    )

                checkpoint = folder / f'{language}_beam{beams}.partial.json'
                signature = mt_checkpoint_signature(loaded, language, f'beam{beams}')
                predictions = mt_checkpoint_load(checkpoint, signature, rows)
                resumed_count = len(predictions)

                for i, row in enumerate(rows, 1):
                    if i <= resumed_count:
                        continue
                    trials = [
                        nllb_decode_loaded(
                            row['source'], language, loaded, beams
                        )
                        for _ in range(MT_BENCH_REPEATS)
                    ]
                    first = trials[0]

                    predictions.append(dict(
                        **row,
                        prediction=first['prediction'],
                        latency_seconds=[t['seconds'] for t in trials],
                        input_tokens=first['input_tokens'],
                        output_tokens=first['output_tokens'],
                        hit_output_limit=any(
                            t['hit_output_limit'] for t in trials
                        ),
                        repeat_output_changed=any(
                            t['prediction'] != first['prediction']
                            for t in trials[1:]
                        ),
                    ))

                    mt_checkpoint_save(checkpoint, signature, predictions)

                    if i % 25 == 0 or i == len(rows):
                        print(
                            f'{language} | beam {beams} | '
                            f'{i}/{len(rows)} sentences'
                        )

                refs = [r['reference'] for r in predictions]
                hyps = [r['prediction'] for r in predictions]
                times = [
                    t for r in predictions for t in r['latency_seconds']
                ]

                bleu = BLEU(tokenize='zh' if language == 'en' else '13a')
                chrf = CHRF(char_order=6, word_order=0, beta=2)

                summary = dict(
                    model='NLLB-600M',
                    direction='en→zh' if language == 'en' else 'zh→en',
                    beams=beams,
                    samples=len(rows),
                    resumed_samples=resumed_count,
                    latency_scope=('multiple sessions; descriptive only' if resumed_count else 'single session'),
                    BLEU=bleu.corpus_score(hyps, [refs]).score,
                    chrF=chrf.corpus_score(hyps, [refs]).score,
                    median_seconds=float(np.median(times)),
                    p95_seconds=float(np.percentile(times, 95)),
                    timing_calls=len(times),
                    empty_outputs=sum(not h.strip() for h in hyps),
                    output_limit_hits=sum(
                        r['hit_output_limit'] for r in predictions
                    ),
                    changed_on_repeat=sum(
                        r['repeat_output_changed'] for r in predictions
                    ),
                )
                signatures = dict(
                    BLEU=str(bleu.get_signature()),
                    chrF=str(chrf.get_signature()),
                )

                old = baselines[language, beams]
                if signatures != old['metric_signatures']:
                    raise ValueError(
                        'Metric settings differ from the Marian baseline.'
                    )

                record = dict(
                    schema_version=1,
                    dataset=MT_DATA_RECORD['settings'],
                    reference_sha256=MT_DATA_RECORD['sha256'],
                    model_id=NLLB_MODEL_ID,
                    revision=NLLB_REVISION,
                    **environment,
                    device=str(device),
                    generation_defaults=model.generation_config.to_dict(),
                    decoding={
                        'num_beams': beams,
                        'num_return_sequences': 1,
                        'max_new_tokens': MT_BENCH_OUTPUT_LIMIT,
                        'max_length': None,
                        'do_sample': False,
                        'early_stopping': beams > 1,
                        'length_penalty': 1.0,
                        'use_cache': True,
                        'forced_bos_token_id':
                            loaded['tokenizer'].convert_tokens_to_ids(
                                NLLB_LANGUAGES[
                                    'zh' if language == 'en' else 'en'
                                ]
                            ),
                    },
                    effective_input_limit=min(
                        MT_BENCH_INPUT_LIMIT,
                        model.config.max_position_embeddings,
                    ),
                    language_codes=NLLB_LANGUAGES,
                    timing=dict(
                        repeats=MT_BENCH_REPEATS,
                        warmups=MT_BENCH_WARMUPS,
                        batch_size=1,
                        includes=(
                            'Language setup, tokenization, transfer, '
                            'generation, decoding'
                        ),
                        excludes=(
                            'Model loading, warmup, metrics and file saving'
                        ),
                    ),
                    metric_signatures=signatures,
                    COMET=None,
                    COMET_status='Pending separate scoring step',
                    summary=summary,
                    predictions=predictions,
                )

                destination = folder / f'{language}_beam{beams}.json'
                destination.write_text(
                    json.dumps(record, ensure_ascii=False, indent=2),
                    encoding='utf-8',
                )
                summaries.append(summary)
                print('Saved private details:', destination.name)

    finally:
        loaded['model'].cpu()
        del loaded
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    table = pd.DataFrame(summaries)
    table.to_csv(folder / 'summary.csv', index=False)

    comparison = pd.DataFrame([
        {'model': 'Marian', **old['summary']}
        for old in baselines.values()
    ] + summaries).sort_values(['direction', 'beams', 'model'])

    comparison.to_csv(folder / 'model_comparison.csv', index=False)
    display(comparison.round(3))

    print(
        'Complete: four new NLLB rows and four saved Marian rows; '
        'dev data only.'
    )
    return table, comparison, folder

if RUN_NLLB_BENCHMARK:
    MT_NLLB_TABLE, MT_MODEL_COMPARISON, MT_NLLB_DIR = mt_run_nllb_benchmark()


In [41]:
# Cell 42 · Optional isolated COMET setup

import os
import subprocess

COMET_ROOT = Path('/content/comet_eval_v1')
COMET_UV = COMET_ROOT / 'tools' / 'bin' / 'uv'
COMET_PY = COMET_ROOT / 'venv' / 'bin' / 'python'
COMET_ENV = dict(os.environ)
COMET_ENV.pop('PYTHONPATH', None)
COMET_ENV['PYTHONNOUSERSITE'] = '1'


def comet_command(*args):
    subprocess.run(
        [str(x) for x in args], check=True, env=COMET_ENV
    )


def mt_setup_comet():
    COMET_ROOT.mkdir(parents=True, exist_ok=True)
    if not COMET_UV.is_file():
        comet_command(
            sys.executable, '-m', 'pip', 'install', '--quiet',
            '--target', COMET_ROOT / 'tools', 'uv==0.13.4',
        )

    if not COMET_PY.is_file():
        comet_command(
            COMET_UV, 'venv', '--python', '3.11.17',
            COMET_ROOT / 'venv',
        )

    comet_command(
        COMET_UV, 'pip', 'install', '--python', COMET_PY,
        'torch==2.6.0', '--index-url',
        'https://download.pytorch.org/whl/cu124',
    )
    comet_command(
        COMET_UV, 'pip', 'install', '--python', COMET_PY,
        'torch==2.6.0', 'unbabel-comet==2.2.7',
        'transformers==4.44.2', 'numpy==1.26.4',
        'pytorch-lightning==2.5.5', 'torchmetrics==0.10.3',
        'huggingface-hub==0.36.0', 'sentencepiece==0.2.1',
        'sacrebleu==2.6.0', 'setuptools==80.9.0',
    )
    comet_command(COMET_UV, 'pip', 'check', '--python', COMET_PY)
    comet_command(COMET_PY, '-c', "\nimport sys, torch, comet, transformers\nfrom importlib.metadata import version\n\nprint('COMET Python:', sys.version.split()[0])\nprint('COMET:', version('unbabel-comet'))\nprint('Transformers:', transformers.__version__)\nprint('PyTorch:', torch.__version__)\nassert torch.cuda.is_available(), 'Select a GPU runtime before scoring.'\nprint('GPU:', torch.cuda.get_device_name(0))\n")

    print('Isolated COMET environment ready.')
if RUN_COMET_SETUP:
    mt_setup_comet()


In [42]:
# Cell 43 · COMET scoring worker

COMET_WORKER_SOURCE = r'''
import csv
import hashlib
import json
import math
import platform
import sys
import time
from importlib.metadata import version
from pathlib import Path

import torch
from comet import load_from_checkpoint
from huggingface_hub import snapshot_download
from pytorch_lightning import seed_everything

MODEL_ID = 'Unbabel/wmt22-comet-da'
REVISION = '2760a223ac957f30acfb18c8aa649b01cf1d75f2'
BATCH_SIZE = 8
SEED = 42


def read_runs(marian_dir, nllb_dir):
    runs = []
    canonical_pairs = None
    dataset = None

    for label, folder in [
        ('Marian', marian_dir), ('NLLB-600M', nllb_dir)
    ]:
        for lang in ('en', 'zh'):
            for beams in (1, 4):
                path = folder / f'{lang}_beam{beams}.json'
                raw = path.read_bytes()
                data = json.loads(raw)
                rows = data['predictions']
                direction = 'en→zh' if lang == 'en' else 'zh→en'

                assert data['dataset']['split'] == 'dev', path
                assert len(rows) == data['summary']['samples'] == data['dataset']['subset_size'], path
                assert len({r['id'] for r in rows}) == len(rows), path
                assert data['summary']['direction'] == direction, path
                assert data['summary']['beams'] == beams, path
                assert data['decoding']['num_beams'] == beams, path
                assert all(
                    isinstance(r[k], str)
                    for r in rows
                    for k in ('source', 'reference', 'prediction')
                ), path

                pairs = [
                    {
                        'id': r['id'],
                        'en': r['source'] if lang == 'en' else r['reference'],
                        'zh': r['reference'] if lang == 'en' else r['source'],
                    }
                    for r in rows
                ]
                fingerprint = hashlib.sha256(
                    json.dumps(
                        pairs, ensure_ascii=False, sort_keys=True
                    ).encode('utf-8')
                ).hexdigest()

                assert fingerprint == data['reference_sha256'], path
                if canonical_pairs is None:
                    canonical_pairs = pairs
                    dataset = data['dataset']

                assert pairs == canonical_pairs, (
                    f'Unmatched sentence order: {path}'
                )
                assert data['dataset'] == dataset, (
                    f'Dataset mismatch: {path}'
                )
                runs.append((
                    label, path, hashlib.sha256(raw).hexdigest(), data
                ))

    return runs


def score_runs(runs, model, output_dir):
    summaries = []
    input_files = []
    token_limit = model.encoder.max_positions - 2

    for label, path, file_hash, data in runs:
        rows = data['predictions']
        samples = [
            {'src': r['source'], 'mt': r['prediction'], 'ref': r['reference']}
            for r in rows
        ]

        # Record any truncation performed by COMET's own tokenizer.
        truncated = [[] for _ in rows]
        for field in ('src', 'mt', 'ref'):
            token_ids = model.encoder.tokenizer(
                [s[field] for s in samples], truncation=False
            )['input_ids']
            for i, ids in enumerate(token_ids):
                if len(ids) > token_limit:
                    truncated[i].append(field)

        print(
            f'Scoring {label} {path.name}: {len(rows)} translations',
            flush=True,
        )
        result = model.predict(
            samples,
            batch_size=BATCH_SIZE,
            gpus=1,
            accelerator='gpu',
            num_workers=0,
            mc_dropout=0,
            length_batching=False,
            progress_bar=True,
        )

        scores = [float(s) for s in result.scores]
        assert len(scores) == len(rows), 'Unexpected COMET score count.'
        assert all(math.isfinite(s) for s in scores), 'Non-finite COMET score.'

        mean_score = math.fsum(scores) / len(scores)
        for row, score, fields in zip(rows, scores, truncated):
            row['COMET'] = score
            row['COMET_truncated_fields'] = fields

        summary = {
            'model': label,
            **data['summary'],
            'COMET': mean_score,
            'COMET_truncated_samples': sum(bool(x) for x in truncated),
        }
        data.update({
            'COMET': mean_score,
            'COMET_status': 'Complete',
            'summary': summary,
            'COMET_evaluator': {
                'model_id': MODEL_ID,
                'revision': REVISION,
                'source_file': str(path),
                'source_file_sha256': file_hash,
                'metadata_file': 'evaluation_metadata.json',
            },
        })

        destination = output_dir / f'{label.lower()}_{path.name}'
        destination.write_text(
            json.dumps(
                data, ensure_ascii=False, indent=2, allow_nan=False
            ),
            encoding='utf-8',
        )
        summaries.append(summary)
        input_files.append({
            'model': label, 'path': str(path), 'sha256': file_hash
        })
        print(
            f'COMET = {mean_score:.4f}; '
            f'truncated samples = {summary["COMET_truncated_samples"]}',
            flush=True,
        )

    with (output_dir / 'model_comparison_with_comet.csv').open(
        'w', encoding='utf-8', newline=''
    ) as f:
        writer = csv.DictWriter(f, fieldnames=list(summaries[0]))
        writer.writeheader()
        writer.writerows(summaries)

    return input_files, token_limit


def main():
    marian_dir, nllb_dir, output_dir = map(Path, sys.argv[1:])
    runs = read_runs(marian_dir, nllb_dir)
    assert torch.cuda.is_available(), 'A GPU runtime is required.'

    seed_everything(SEED, workers=True)
    torch.set_float32_matmul_precision('highest')

    snapshot = Path(snapshot_download(
        MODEL_ID,
        revision=REVISION,
        allow_patterns=['checkpoints/model.ckpt', 'hparams.yaml'],
    ))
    model = load_from_checkpoint(
        str(snapshot / 'checkpoints' / 'model.ckpt')
    )
    model.float().eval()

    output_dir.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    input_files, token_limit = score_runs(runs, model, output_dir)

    metadata = {
        'status': 'Complete',
        'model_id': MODEL_ID,
        'revision': REVISION,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'mc_dropout': 0,
        'precision': 'float32',
        'hardware': torch.cuda.get_device_name(0),
        'versions': {
            'python': platform.python_version(),
            **{p: version(p) for p in (
                'torch', 'unbabel-comet', 'transformers', 'numpy',
                'pytorch-lightning', 'torchmetrics', 'huggingface-hub',
                'sentencepiece', 'sacrebleu', 'setuptools',
            )},
        },
        'encoder_model': model.hparams.pretrained_model,
        'encoder_config_revision': getattr(
            model.encoder.model.config, '_commit_hash', None
        ),
        'token_limit_per_text': token_limit,
        'dataset': runs[0][3]['dataset'],
        'reference_sha256': runs[0][3]['reference_sha256'],
        'input_files': input_files,
        'configurations_scored': len(runs),
        'translations_scored': sum(
            len(r[3]['predictions']) for r in runs
        ),
        'evaluation_seconds': time.perf_counter() - started,
        'timing_note': (
            'COMET evaluation is excluded from translation latency.'
        ),
    }
    (output_dir / 'evaluation_metadata.json').write_text(
        json.dumps(
            metadata, ensure_ascii=False, indent=2, allow_nan=False
        ),
        encoding='utf-8',
    )
    print(
        f'Complete: {len(runs)} configurations, {sum(len(r[3]["predictions"]) for r in runs)} translations.\n'
        f'Saved: {output_dir}'
    )


if __name__ == '__main__':
    main()
'''


In [43]:
# Cell 44 · Optional COMET scoring

def mt_score_saved_comet():
    if not globals().get('MT_BASELINE_DIR') or not globals().get('MT_NLLB_DIR'):
        raise RuntimeError('Select completed MT_BASELINE_DIR and MT_NLLB_DIR first.')
    MARIAN_COMET_INPUT = Path(MT_BASELINE_DIR)
    NLLB_COMET_INPUT = Path(MT_NLLB_DIR)
    if not globals().get('MT_DATA_READY', False):
        raise RuntimeError('Load the saved development references before COMET.')
    for _folder in (MARIAN_COMET_INPUT, NLLB_COMET_INPUT):
        for _lang, _target in (('en', 'zh'), ('zh', 'en')):
            _expected = [dict(id=r['id'], source=r[_lang], reference=r[_target])
                         for r in MT_DATA_RECORD['pairs']]
            for _beam in (1, 4):
                _path = _folder / f'{_lang}_beam{_beam}.json'
                _record = json.loads(_path.read_text(encoding='utf-8'))
                _actual = [{k:r[k] for k in ('id','source','reference')}
                           for r in _record['predictions']]
                if (_record['reference_sha256'] != MT_DATA_RECORD['sha256'] or
                    _record['dataset'] != MT_DATA_RECORD['settings'] or
                    _actual != _expected or
                    _record['summary']['samples'] != len(_expected)):
                    raise ValueError(f'Incomplete or mismatched COMET input: {_path}')
    output_dir = Path(RESULTS_DIR) / 'mt_comet' / str(time.time_ns())
    COMET_WORKER = COMET_ROOT / 'score_saved_mt.py'

    if not COMET_PY.is_file():
        raise RuntimeError('Run the isolated COMET setup before scoring.')
    COMET_ROOT.mkdir(parents=True, exist_ok=True)
    COMET_WORKER.write_text(COMET_WORKER_SOURCE, encoding='utf-8')
    for name in ('release_routed_mt', 'release_ocr'):
        if callable(globals().get(name)):
            globals()[name]()

    comet_command(
        COMET_PY, '-u', COMET_WORKER,
        MARIAN_COMET_INPUT, NLLB_COMET_INPUT, output_dir,
    )

    comet_comparison = pd.read_csv(
        output_dir / 'model_comparison_with_comet.csv'
    )
    display(comet_comparison[[
        'model', 'direction', 'beams', 'BLEU', 'chrF', 'COMET',
        'median_seconds', 'p95_seconds', 'output_limit_hits',
        'COMET_truncated_samples',
    ]].round(4))

    print('Saved:', output_dir)
    return comet_comparison, output_dir

if RUN_COMET_SCORING:
    COMET_COMPARISON, COMET_RESULTS_DIR = mt_score_saved_comet()


In [44]:
# Cell 45 · Paired saved-result statistics

def mt_paired_mean_ci(a, b, *, repeats=2000, seed=42):
    """B-minus-A paired mean difference; percentile 95% interval."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.ndim != 1 or b.shape != a.shape or len(a)<2:
        raise ValueError('Need >=2 aligned one-dimensional pairs.')
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError('Scores must be finite.')
    if not isinstance(repeats, int) or repeats<100:
        raise ValueError('Use at least 100 bootstrap repeats.')
    delta = b-a
    rng = np.random.default_rng(seed)
    # Bounded batches keep memory small, even for much larger datasets.
    means=[]
    for start in range(0,repeats,100):
        idx=rng.integers(0,len(delta),size=(min(100,repeats-start),len(delta)))
        means.extend(delta[idx].mean(axis=1).tolist())
    lo, hi=np.quantile(means,[.025,.975])
    return dict(n=len(delta),mean_A=float(a.mean()),mean_B=float(b.mean()),
                difference_B_minus_A=float(delta.mean()),ci95_low=float(lo),
                ci95_high=float(hi),repeats=repeats,seed=seed,
                interval_excludes_zero=bool(lo>0 or hi<0))


def mt_load_aligned_runs(path_a,path_b):
    runs=[json.loads(Path(p).read_text(encoding='utf-8')) for p in (path_a,path_b)]
    a,b=runs
    if a['dataset']!=b['dataset'] or a['dataset'].get('split')!='dev':
        raise ValueError('Compare the same DEVELOPMENT dataset only.')
    if a['reference_sha256']!=b['reference_sha256']:
        raise ValueError('Reference fingerprints differ.')
    pairs=[]
    for run in runs:
        rows=run['predictions']
        if len(rows)!=run['summary']['samples'] or not rows:
            raise ValueError('Incomplete result file.')
        ids=[r['id'] for r in rows]
        if len(set(ids))!=len(ids): raise ValueError('Duplicate sentence IDs.')
        pairs.append([{k:r[k] for k in ('id','source','reference')} for r in rows])
    if pairs[0]!=pairs[1]:
        raise ValueError('IDs, order, source or references differ; do not zip unmatched runs.')
    return runs


def mt_punctuation_normalize(text):
    """A symmetric scoring variant only; preserve ordinary decimals and URLs.

    Map punctuation code points, without adding/removing sentence boundaries,
    deleting punctuation or applying broad NFKC/number conversion.
    """
    if not isinstance(text, str):
        raise TypeError('Metric inputs must be strings.')
    return text.translate(str.maketrans({
        '，':',', '。':'.', '！':'!', '？':'?', '；':';', '：':':',
        '（':'(', '）':')', '“':'"', '”':'"', '‘':"'", '’':"'",
    }))


def mt_paired_corpus_ci(pred_a, pred_b, references, metric, *, repeats=200, seed=42):
    """Recompute corpus metric B-minus-A on the SAME sentence draws."""
    if len(pred_a)!=len(pred_b) or len(pred_a)!=len(references) or len(pred_a)<2:
        raise ValueError('Need >=2 aligned prediction/reference triples.')
    if not isinstance(repeats,int) or repeats<100:
        raise ValueError('Use at least 100 bootstrap repetitions.')
    if not all(isinstance(x,str) for seq in (pred_a,pred_b,references) for x in seq):
        raise TypeError('All predictions and references must be strings.')
    def score(pred,ref):
        value=float(metric.corpus_score(pred,[ref]).score)
        if not np.isfinite(value): raise ValueError('Non-finite corpus score.')
        return value
    full_a,full_b=score(pred_a,references),score(pred_b,references)
    rng=np.random.default_rng(seed)
    delta=[]
    for _ in range(repeats):
        draw=rng.integers(0,len(references),size=len(references))
        ref=[references[i] for i in draw]
        pa=[pred_a[i] for i in draw]
        pb=[pred_b[i] for i in draw]
        delta.append(score(pb,ref)-score(pa,ref))
    lo,hi=np.quantile(delta,[.025,.975])
    return dict(n=len(references),score_A=full_a,score_B=full_b,
                difference_B_minus_A=full_b-full_a,ci95_low=float(lo),
                ci95_high=float(hi),repeats=repeats,seed=seed,
                interval_excludes_zero=bool(lo>0 or hi<0))


def mt_compare_saved_pair(path_a,path_b, *, repeats=200,seed=42):
    from sacrebleu.metrics import BLEU,CHRF
    a,b=mt_load_aligned_runs(path_a,path_b)
    rows_a,rows_b=a['predictions'],b['predictions']
    direction=a['summary']['direction']
    if direction!=b['summary']['direction'] or direction not in ('en→zh','zh→en'):
        raise ValueError('Direction differs or is unsupported.')
    pred_a=[r['prediction'] for r in rows_a]
    pred_b=[r['prediction'] for r in rows_b]
    refs=[r['reference'] for r in rows_a]
    results={}
    for variant,normalizer in [('raw',lambda x:x),
                               ('punctuation_normalized',mt_punctuation_normalize)]:
        # Apply EXACTLY the same transformation to both systems AND references.
        pa,pb,rr=([normalizer(x) for x in seq] for seq in (pred_a,pred_b,refs))
        results[variant]={}
        metrics={'corpus_BLEU':BLEU(tokenize='zh' if direction=='en→zh' else '13a'),
                 'corpus_chrF':CHRF(char_order=6,word_order=0,beta=2)}
        for name,metric in metrics.items():
            result=mt_paired_corpus_ci(pa,pb,rr,metric,repeats=repeats,seed=seed)
            result['metric_signature']=str(metric.get_signature())
            results[variant][name]=result
    # COMET is a separate paired mean of the already-scored RAW examples.
    # Never relabel raw COMET scores as punctuation-normalized COMET.
    if all('COMET' in r for r in rows_a+rows_b):
        ev_a,ev_b=a.get('COMET_evaluator',{}),b.get('COMET_evaluator',{})
        if (any(not ev.get(k) for ev in (ev_a,ev_b) for k in ('model_id','revision'))
            or any(ev_a[k]!=ev_b[k] for k in ('model_id','revision'))):
            raise ValueError('COMET evaluator metadata differs or is missing.')
        results['raw_mean_COMET']=mt_paired_mean_ci(
            [r['COMET'] for r in rows_a],[r['COMET'] for r in rows_b],
            repeats=repeats,seed=seed)
    return dict(A=str(path_a),B=str(path_b),direction=direction,
                reference_sha256=a['reference_sha256'],results=results,
                caution='Development-set, sentence-level uncertainty; not a final-test claim.')


In [45]:
# Cell 46 · Optional paired-bootstrap analysis

# Sentence-level dev uncertainty only. Resample document groups when available.
# Corpus BLEU/chrF are recomputed on each paired draw; this runs on CPU.
def mt_run_saved_bootstrap():

    if not globals().get('COMET_RESULTS_DIR'):
        raise RuntimeError('Set COMET_RESULTS_DIR to the completed saved COMET folder; do not rerun models.')
    root=Path(COMET_RESULTS_DIR)
    comparisons=[
        ('en_marian_beam1_vs_beam4','marian_en_beam1.json','marian_en_beam4.json'),
        ('zh_marian_beam1_vs_beam4','marian_zh_beam1.json','marian_zh_beam4.json'),
        ('en_marian4_vs_nllb4','marian_en_beam4.json','nllb-600m_en_beam4.json'),
        ('zh_marian4_vs_nllb4','marian_zh_beam4.json','nllb-600m_zh_beam4.json'),
    ]
    MT_PAIRED_BOOTSTRAP={name:mt_compare_saved_pair(root/a,root/b,
                              repeats=BOOTSTRAP_REPEATS,seed=BOOTSTRAP_SEED)
                         for name,a,b in comparisons}
    print(json.dumps(MT_PAIRED_BOOTSTRAP,ensure_ascii=False,indent=2))

    return MT_PAIRED_BOOTSTRAP

if RUN_PAIRED_BOOTSTRAP:
    MT_PAIRED_BOOTSTRAP = mt_run_saved_bootstrap()


In [46]:
# Cell 47 · Optional flag-quality analysis

def mt_flag_quality_association(path):
    from sacrebleu.metrics import CHRF
    from collections import Counter
    if not callable(globals().get('mt_check_translation')):
        raise RuntimeError('Run the reference-free review-check definitions first.')
    run=json.loads(Path(path).read_text(encoding='utf-8'))
    rows=run['predictions']
    if run['dataset'].get('split')!='dev' or len(rows)!=run['summary']['samples']:
        raise ValueError('Use a complete development result file.')
    direction=run['summary']['direction']
    if direction not in ('en→zh','zh→en'): raise ValueError('Unknown direction.')
    lang='en' if direction=='en→zh' else 'zh'
    chrf=CHRF(char_order=6,word_order=0,beta=2)
    groups={True:[],False:[]}; counts=Counter()
    for row in rows:
        check=mt_check_translation(row['source'],row['prediction'],lang)
        flags=set(check.get('flags',[]))
        if row.get('hit_output_limit'): flags.add('output_limit')
        counts.update(flags)
        value={'sentence_chrF':chrf.sentence_score(row['prediction'],[row['reference']]).score}
        if 'COMET' in row: value['COMET']=float(row['COMET'])
        groups[bool(flags)].append(value)
    result={}
    for flagged,values in groups.items():
        item={'n':len(values)}
        for metric in ('sentence_chrF','COMET'):
            available=[v[metric] for v in values if metric in v]
            item['mean_'+metric]=(float(np.mean(available)) if available else None)
            item['n_'+metric]=len(available)
        result['flagged' if flagged else 'unflagged']=item
    return dict(path=str(path),direction=direction,groups=result,flag_counts=dict(counts),
                check_version=globals().get('MT_CHECK_VERSION','unknown'),
                note='Current heuristic flags recomputed on raw saved predictions; descriptive association only.')


# This is association, not detector accuracy: human error labels are absent.
def mt_run_saved_flag_association():

    if not globals().get('COMET_RESULTS_DIR'):
        raise RuntimeError('Select the completed COMET result folder first.')
    root=Path(COMET_RESULTS_DIR)
    MT_FLAG_ASSOCIATION={name:mt_flag_quality_association(root/name) for name in (
        'marian_en_beam1.json','marian_en_beam4.json',
        'marian_zh_beam1.json','marian_zh_beam4.json','nllb-600m_zh_beam4.json')}
    print(json.dumps(MT_FLAG_ASSOCIATION,ensure_ascii=False,indent=2))
    return MT_FLAG_ASSOCIATION

if RUN_FLAG_ASSOCIATION:
    MT_FLAG_ASSOCIATION = mt_run_saved_flag_association()


In [47]:
# Cell 48 · Paired-statistics smoke tests

# Synthetic data only; no model, dataset or metric-package downloads.
assert mt_paired_mean_ci([1, 2, 3], [1, 2, 3], repeats=100)['ci95_low'] == 0
assert mt_paired_mean_ci([1, 2, 3], [2, 3, 4], repeats=100)['ci95_low'] == 1
assert mt_paired_mean_ci([1, 2, 3], [0, 1, 2], repeats=100)['ci95_high'] == -1
assert mt_punctuation_normalize('中文，3.14。') == '中文,3.14.'
print('Synthetic paired-statistics checks passed.')


Synthetic paired-statistics checks passed.


In [48]:
# Cell 49 · Page detection and reading-order OCR

"""Page-to-line OCR without changing the notebook's line-level training API.

The default detector uses only Pillow and NumPy. It is a heuristic for dark ink
on light, horizontal, single-column pages, not a document-layout model. Boxes
are [left, top, right, bottom], right/bottom exclusive, in the returned working
RGB image. This image is EXIF-oriented, size-limited and optionally deskewed;
the caller's original image is never modified. Multi-column/vertical writing,
touching lines, heavy perspective and decorative backgrounds need manual boxes.
"""
import math
import time
from numbers import Real

import numpy as np
from PIL import Image, ImageFilter, ImageOps


_PAGE_DETECTOR_CACHE = None
PAGE_DETECTOR_ID = 'PaddlePaddle/PP-OCRv5_mobile_det_safetensors'
PAGE_DETECTOR_REVISION = 'c5041d225cf951ff06900ab81a3c7d543c45e2ad'


def _page_rgb(image, max_image_pixels):
    if not isinstance(image, Image.Image):
        raise TypeError('image must be a PIL Image; decode uploads before calling.')
    if isinstance(max_image_pixels, bool) or not isinstance(max_image_pixels, int) or max_image_pixels < 10_000:
        raise ValueError('max_image_pixels must be an integer of at least 10,000.')
    if image.width < 2 or image.height < 2:
        raise ValueError('The image must have at least two pixels on each side.')
    if max(image.size) / min(image.size) > 1000:
        raise ValueError('The image is too narrow for bounded page processing; use a normal page crop.')
    original_size = tuple(image.size)
    orientation = image.getexif().get(274, 1)
    oriented_size = original_size[::-1] if orientation in (5, 6, 7, 8) else original_size
    scale = min(1.0, math.sqrt(max_image_pixels / (image.width * image.height)))
    # Reduce before color/alpha/EXIF copies so processing buffers remain bounded.
    reduced = image.resize((max(2, int(image.width * scale)), max(2, int(image.height * scale))), Image.Resampling.LANCZOS) if scale < 1 else image
    oriented = ImageOps.exif_transpose(reduced)
    # Flatten transparent images on paper-white instead of interpreting alpha as ink.
    if oriented.mode in ('RGBA', 'LA') or (oriented.mode == 'P' and 'transparency' in oriented.info):
        rgba = oriented.convert('RGBA')
        white = Image.new('RGBA', rgba.size, 'white')
        white.alpha_composite(rgba)
        rgb = white.convert('RGB')
    else:
        rgb = oriented.convert('RGB').copy()
    return rgb, {'original_size': original_size, 'oriented_size': oriented_size,
                 'resize_scale': scale, 'post_deskew_resize_scale': 1.0,
                 'exif_transposed': orientation in (2, 3, 4, 5, 6, 7, 8)}



def _page_otsu(gray):
    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    weight = np.cumsum(histogram)
    moment = np.cumsum(histogram * np.arange(256))
    denom = weight * (weight[-1] - weight)
    scores = np.zeros(256)
    valid = denom > 0
    scores[valid] = (moment[-1] * weight[valid] - moment[valid] * weight[-1]) ** 2 / denom[valid]
    return int(np.argmax(scores))


def _page_ink_mask(rgb):
    gray_image = rgb.convert('L')
    gray = np.asarray(gray_image)
    # Low-contrast paper, including near-uniform sensor noise, must not hallucinate lines.
    contrast = float(np.percentile(gray, 99.5) - np.percentile(gray, 0.5))
    if int(gray.max()) - int(gray.min()) < 12:
        return np.zeros(gray.shape, dtype=bool), {'threshold': None, 'contrast': contrast, 'ink_fraction': 0.0}
    threshold = min(230, _page_otsu(gray) + 12)
    radius = max(8, min(rgb.size) // 35)
    local_paper = np.asarray(gray_image.filter(ImageFilter.BoxBlur(radius))).astype(np.int16)
    local_difference = local_paper - gray.astype(np.int16)
    # Global dark ink plus local contrast accommodates moderate paper shading.
    mask = ((gray <= threshold) & (local_difference >= 7)) | (gray < min(110, threshold))
    # Remove isolated specks, but keep thin strokes and 2x2 punctuation.
    padded = np.pad(mask.astype(np.uint8), 1)
    support = sum(padded[dy:dy + gray.shape[0], dx:dx + gray.shape[1]] for dy in range(3) for dx in range(3))
    mask &= support >= 3
    return mask, {'threshold': threshold, 'contrast': contrast, 'ink_fraction': float(mask.mean())}


def _page_runs(values):
    edges = np.flatnonzero(np.diff(np.r_[False, np.asarray(values, dtype=bool), False].astype(np.int8)))
    return [(int(a), int(b)) for a, b in zip(edges[::2], edges[1::2])]


def _page_skew(mask, max_angle=5.0):
    """Return a conservative correction, using a small downsampled binary mask."""
    if mask.sum() < 30 or min(mask.shape) < 30 or max_angle <= 0:
        return 0.0, 'insufficient_ink', 0.0
    probe = Image.fromarray((mask * 255).astype(np.uint8))
    probe.thumbnail((720, 720), Image.Resampling.NEAREST)

    def score(angle):
        rotated = np.asarray(probe.rotate(float(angle), resample=Image.Resampling.NEAREST, expand=True, fillcolor=0)) > 0
        counts = rotated.sum(axis=1).astype(np.float64)
        return float((counts * counts).sum() / max(1.0, counts.sum()))

    baseline = score(0)
    candidates = np.arange(-max_angle, max_angle + 0.001, 0.5)
    best_angle, best_score = max(((float(a), score(a)) for a in candidates), key=lambda pair: pair[1])
    gain = best_score / max(baseline, 1e-9) - 1.0
    if abs(best_angle) >= max_angle - 0.01 and gain >= 0.12:
        return 0.0, 'outside_safe_angle', gain
    if gain < 0.12 or abs(best_angle) < 0.4:
        return 0.0, 'no_clear_improvement', gain
    refined = np.arange(max(-max_angle, best_angle - 0.4), min(max_angle, best_angle + 0.4) + 0.001, 0.1)
    best_angle, best_score = max(((float(a), score(a)) for a in refined), key=lambda pair: pair[1])
    return round(best_angle, 2), 'applied', best_score / max(baseline, 1e-9) - 1.0


def _page_projection_boxes(mask):
    height, width = mask.shape
    row_counts = mask.sum(axis=1)
    bands = _page_runs(row_counts >= 2)
    if not bands:
        return [], []
    heights = [b - a for a, b in bands if b - a >= 4]
    typical = float(np.median(heights)) if heights else 4.0
    join_gap = max(2, min(12, int(round(typical * 0.30))))
    merged = []
    for top, bottom in bands:
        if merged and top - merged[-1][1] <= join_gap:
            merged[-1][1] = bottom
        else:
            merged.append([top, bottom])
    # Attach detached ascenders/dots to a nearby overlapping body, not another row.
    typical = float(np.median([b - a for a, b in merged if b - a >= 4])) if any(b - a >= 4 for a, b in merged) else 4.0
    i = 0
    while i < len(merged):
        top, bottom = merged[i]
        if bottom - top <= max(3, typical * 0.28) and len(merged) > 1:
            xs = np.flatnonzero(mask[top:bottom].any(axis=0))
            options = []
            for j in (i - 1, i + 1):
                if 0 <= j < len(merged):
                    a, b = merged[j]
                    neighbor_x = np.flatnonzero(mask[a:b].any(axis=0))
                    gap = max(top, a) - min(bottom, b)
                    overlap = bool(len(xs) and len(neighbor_x) and xs[-1] >= neighbor_x[0] and neighbor_x[-1] >= xs[0])
                    if overlap and gap <= max(4, typical * 0.55) and b - a > bottom - top:
                        options.append((gap, j))
            if options:
                _, j = min(options)
                merged[j] = [min(top, merged[j][0]), max(bottom, merged[j][1])]
                del merged[i]
                i = max(0, i - 1)
                continue
        i += 1
    boxes = []
    warnings = []
    for top, bottom in merged:
        crop_mask = mask[top:bottom]
        xs = np.flatnonzero(crop_mask.any(axis=0))
        if not len(xs) or crop_mask.sum() < 4:
            continue
        left, right = int(xs[0]), int(xs[-1]) + 1
        boxes.append([left, top, right, bottom])
        if bottom - top > max(80, height * 0.30) and (right - left) / max(1, bottom - top) < 5:
            warnings.append('A tall text region may contain touching/merged lines. Review the boxes or provide manual boxes.')
        if bottom - top < 5:
            warnings.append('A very short region may be punctuation or noise. Review it before using the transcription.')
    return boxes, list(dict.fromkeys(warnings))


def _page_pad_boxes(boxes, size):
    width, height = size
    result = []
    for i, (left, top, right, bottom) in enumerate(boxes):
        line_height = max(1, bottom - top)
        py = max(2, min(14, int(round(line_height * 0.18))))
        px = max(4, min(24, int(round(line_height * 0.30))))
        # Don't pad into the adjacent line when projection bands are close.
        before = boxes[i - 1][3] if i else 0
        after = boxes[i + 1][1] if i + 1 < len(boxes) else height
        y0 = max(0, top - py, (before + top) // 2 if i else 0)
        y1 = min(height, bottom + py, (bottom + after + 1) // 2 if i + 1 < len(boxes) else height)
        result.append([max(0, left - px), y0, min(width, right + px), y1])
    return result


def _page_validate_boxes(boxes, size):
    if not isinstance(boxes, (list, tuple)):
        raise TypeError('boxes_override must be a list of [left, top, right, bottom] boxes.')
    width, height = size
    result = []
    for index, box in enumerate(boxes):
        if not isinstance(box, (list, tuple, np.ndarray)) or len(box) != 4:
            raise ValueError(f'Box {index + 1} must contain exactly four coordinates.')
        if any(isinstance(v, (bool, np.bool_)) or not isinstance(v, Real) or not math.isfinite(float(v)) or float(v) != int(v) for v in box):
            raise ValueError(f'Box {index + 1} coordinates must be finite integers.')
        left, top, right, bottom = map(int, box)
        if not (0 <= left < right <= width and 0 <= top < bottom <= height):
            raise ValueError(f'Box {index + 1} is outside the {width}x{height} working image or has no area.')
        result.append([left, top, right, bottom])
    return result


def release_page_detector():
    """Drop only the optional detection model; this never evicts the shared OCR/MT cache."""
    global _PAGE_DETECTOR_CACHE
    _PAGE_DETECTOR_CACHE = None


def _page_ppocr_boxes(rgb, detector_fn=None, score_threshold=0.35):
    """Optional detector; CPU resident so it does not compete with OCR/MT GPU caches.

    Transformers 5.17 documents AutoModelForObjectDetection and target_sizes.
    Its mobile detector shares PPOCRV5ServerDetImageProcessor, returning quads.
    Requires torch/torchvision/OpenCV, installed by the full notebook environment.
    Contract source: https://huggingface.co/docs/transformers/v5.17.0/en/model_doc/pp_ocrv5_mobile_det
    Real weights/inference must be smoke-tested in the notebook runtime.
    """
    global _PAGE_DETECTOR_CACHE
    if detector_fn is not None:
        raw = detector_fn(rgb)
        # Injection contract: dict with boxes (N,4,2) quads or (N,4) XYXY, optional scores.
    else:
        try:
            import torch
            from transformers import AutoImageProcessor, AutoModelForObjectDetection
        except ImportError as exc:
            raise RuntimeError('Optional PP-OCR detection requires the notebook Transformers/PyTorch dependencies.') from exc
        if _PAGE_DETECTOR_CACHE is None:
            processor = AutoImageProcessor.from_pretrained(PAGE_DETECTOR_ID, revision=PAGE_DETECTOR_REVISION)
            model = AutoModelForObjectDetection.from_pretrained(PAGE_DETECTOR_ID, revision=PAGE_DETECTOR_REVISION).to('cpu').eval()
            _PAGE_DETECTOR_CACHE = (model, processor)
        model, processor = _PAGE_DETECTOR_CACHE
        inputs = processor(images=rgb, return_tensors='pt')
        with torch.inference_mode():
            outputs = model(**inputs)
        raw = processor.post_process_object_detection(outputs, target_sizes=inputs['target_sizes'])[0]
    if not isinstance(raw, dict) or 'boxes' not in raw:
        raise ValueError('PP-OCR detector must return a dictionary containing boxes.')
    value = raw['boxes']
    array = value.detach().cpu().numpy() if hasattr(value, 'detach') else np.asarray(value)
    if array.size == 0:
        return []
    if array.ndim == 3 and array.shape[1:] == (4, 2):
        xyxy = np.column_stack((array[:, :, 0].min(axis=1), array[:, :, 1].min(axis=1),
                                array[:, :, 0].max(axis=1), array[:, :, 1].max(axis=1)))
    elif array.ndim == 2 and array.shape[1] == 4:
        xyxy = array
    else:
        raise ValueError(f'Unsupported detector box shape {array.shape}; expected (N,4,2) or (N,4).')
    if not np.isfinite(xyxy).all():
        raise ValueError('Detector returned non-finite coordinates.')
    scores = raw.get('scores')
    if scores is not None:
        scores = scores.detach().cpu().numpy() if hasattr(scores, 'detach') else np.asarray(scores)
        if scores.shape != (len(xyxy),) or not np.isfinite(scores).all():
            raise ValueError('Detector scores must be finite and aligned with boxes.')
        xyxy = xyxy[scores >= score_threshold]
    boxes = []
    for left, top, right, bottom in xyxy:
        box = [max(0, int(math.floor(left))), max(0, int(math.floor(top))),
               min(rgb.width, int(math.ceil(right))), min(rgb.height, int(math.ceil(bottom)))]
        if box[0] < box[2] and box[1] < box[3]:
            boxes.append(box)
    # Explicit single-column reading order. No inference of a page's column hierarchy.
    return sorted(boxes, key=lambda box: (box[1], box[0]))


def detect_page_lines(image, *, max_lines=40, max_image_pixels=2_000_000,
                      deskew=True, max_skew_angle=5.0, detection_mode='projection',
                      boxes_override=None, detector_fn=None, detector_score_threshold=0.35):
    """Detect padded ordered lines and return their RGB coordinate image + metadata.

    Defaults cap inference at 24 lines and segmentation at 2 MP. Manual boxes are
    exact (not padded/reordered), in the returned working image's coordinates.
    For editing previously detected boxes, pass that returned image and use
    deskew=False. Automatic results assume one horizontal column.
    Optional PP-OCR detection is opt-in and may download a model on first use.
    """
    started = time.perf_counter()
    if isinstance(max_lines, bool) or not isinstance(max_lines, int) or not 1 <= max_lines <= 100:
        raise ValueError('max_lines must be an integer from 1 to 100.')
    if detection_mode not in ('projection', 'ppocr'):
        raise ValueError("detection_mode must be 'projection' or 'ppocr'.")
    if not isinstance(deskew, (bool, np.bool_)):
        raise ValueError('deskew must be a boolean.')
    if not isinstance(max_skew_angle, Real) or not math.isfinite(float(max_skew_angle)) or not 0 <= max_skew_angle <= 7:
        raise ValueError('max_skew_angle must be between 0 and 7 degrees.')
    if not isinstance(detector_score_threshold, Real) or not 0 <= detector_score_threshold <= 1:
        raise ValueError('detector_score_threshold must be from 0 to 1.')
    rgb, metadata = _page_rgb(image, max_image_pixels)
    warnings = []
    if metadata['resize_scale'] < 1:
        warnings.append('Large image resized for bounded processing; upload a tighter crop if small handwriting becomes unreadable.')
    mask, ink_metadata = _page_ink_mask(rgb)
    angle, skew_reason, gain = _page_skew(mask, float(max_skew_angle)) if deskew else (0.0, 'disabled', 0.0)
    if angle:
        rgb = rgb.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor='white')
        if rgb.width * rgb.height > max_image_pixels:
            scale = math.sqrt(max_image_pixels / (rgb.width * rgb.height))
            rgb = rgb.resize((max(2, int(rgb.width * scale)), max(2, int(rgb.height * scale))), Image.Resampling.LANCZOS)
            metadata['post_deskew_resize_scale'] = scale
        mask, ink_metadata = _page_ink_mask(rgb)
    elif skew_reason == 'outside_safe_angle':
        warnings.append('Possible tilt exceeds the safe deskew range. Retake the page straighter or review manual boxes.')
    metadata.update(ink_metadata)
    metadata.update(working_size=tuple(rgb.size), coordinate_system='working_image_xyxy_exclusive',
                    reading_order='manual' if boxes_override is not None else 'single_column_top_to_bottom',
                    detector='manual' if boxes_override is not None else detection_mode,
                    deskew_angle_degrees=angle, deskew_reason=skew_reason,
                    deskew_projection_gain=round(float(gain), 4), max_lines=max_lines,
                    max_image_pixels=max_image_pixels, calibrated_confidence=None)
    if boxes_override is not None:
        boxes = _page_validate_boxes(boxes_override, rgb.size)
    elif mask.sum() < 4:
        boxes = []
    elif detection_mode == 'ppocr':
        boxes = _page_ppocr_boxes(rgb, detector_fn, detector_score_threshold)
        # Detector boxes may share a row; use symmetric safe padding rather than
        # pretending sorted top edges imply non-overlapping vertical intervals.
        boxes = [[max(0, a - 4), max(0, b - 3), min(rgb.width, c + 4), min(rgb.height, d + 3)] for a, b, c, d in boxes]
        metadata['detector_model_id'] = PAGE_DETECTOR_ID
        metadata['detector_revision'] = PAGE_DETECTOR_REVISION
        warnings.append('PP-OCR boxes are sorted top-to-bottom for one column; review fragments, columns and handwriting boundaries. Slanted quads use enclosing rectangular crops, not perspective rectification.')
    else:
        boxes, projection_warnings = _page_projection_boxes(mask)
        warnings.extend(projection_warnings)
        boxes = _page_pad_boxes(boxes, rgb.size)
        pixels = np.asarray(rgb)
        width, height = rgb.size
        center = pixels[height // 4:3 * height // 4:4, width // 4:3 * width // 4:4]
        paper_light = np.median(center.mean(axis=2))
        paper_chroma = np.median(np.ptp(center, axis=2))

        kept = []
        for box in boxes:
            left, top, right, bottom = box
            if left == 0 or top == 0 or right == width or bottom == height:
                crop = pixels[top:bottom:2, left:right:2]
                off_paper = (
                    (crop.mean(axis=2) < paper_light - 20)
                    & (np.ptp(crop, axis=2) > paper_chroma + 16)
                )
                if off_paper.mean() >= 0.70:
                    continue
            kept.append(box)
        boxes = kept
    detected_count = len(boxes)
    if detected_count > max_lines:
        warnings.append(f'Detected {detected_count} regions; only the first {max_lines} will be read. Split the page to read the rest.')
        boxes = boxes[:max_lines]
    if not boxes:
        warnings.append('No readable text lines detected. The page may be blank, faint or poorly lit; no OCR was run.')
    if ink_metadata['ink_fraction'] > 0.35:
        warnings.append('The image contains a large dark region. Detection is intended for dark writing on light paper; review all boxes.')
    metadata.update(detected_line_count=detected_count, returned_line_count=len(boxes),
                    truncated=detected_count > max_lines,
                    segmentation_seconds=round(time.perf_counter() - started, 4))
    return {'image': rgb, 'boxes': boxes, 'warnings': list(dict.fromkeys(warnings)), 'metadata': metadata}


def recognize_page(image, language, version, recognition_mode='selected', *,
                   load_ocr_fn=None, transcribe_fn=None, mixed_version=None, **detection_kwargs):
    """Read every detected line once, preserving raw model text and line order.

    selected: use exactly language/version. mixed: explicitly use zh PP-OCR for
    every complete line, baseline by default. Fine-tuned Chinese-only checkpoints
    can forget Latin characters; opting into one requires mixed_version. The
    caller's selected language stays unchanged and controls translation direction.
    No script-detected rerouting, punctuation replacement or autocorrection occurs.
    """
    if language not in ('en', 'zh'):
        raise ValueError("language must be 'en' or 'zh'.")
    if recognition_mode not in ('selected', 'mixed'):
        raise ValueError("recognition_mode must be 'selected' or 'mixed'.")
    if not isinstance(version, str) or not version:
        raise ValueError('version must name an available OCR checkpoint.')
    if mixed_version is not None and (not isinstance(mixed_version, str) or not mixed_version):
        raise ValueError('mixed_version must be None or an explicit checkpoint version.')
    result = detect_page_lines(image, **detection_kwargs)
    ocr_language = 'zh' if recognition_mode == 'mixed' else language
    actual_version = (mixed_version or 'baseline') if recognition_mode == 'mixed' else version
    result.update(lines=[], text='', language=language, source_language=language,
                  recognition_mode=recognition_mode, ocr_language=ocr_language,
                  requested_version=version, actual_version=actual_version)
    if recognition_mode == 'mixed':
        result['warnings'].append('Mixed mode uses Chinese PP-OCR for complete English/Chinese lines. Mixed handwriting accuracy is not guaranteed; review the raw text. Translation direction still follows the selected input language.')
        if version != actual_version:
            result['warnings'].append(f'Mixed mode uses {actual_version!r}, not the selected {version!r} checkpoint.')
        if actual_version != 'baseline':
            result['warnings'].append('The explicitly selected Chinese fine-tuned checkpoint may have reduced English coverage; validate mixed-language examples.')
    if not result['boxes']:
        result['metadata'].update(ocr_seconds=0.0, model_load_seconds=0.0)
        return result
    loader = load_ocr_fn or globals().get('load_ocr')
    recognizer = transcribe_fn or globals().get('transcribe')
    if not callable(loader) or not callable(recognizer):
        raise RuntimeError('Run OCR loading helpers first, or pass load_ocr_fn and transcribe_fn explicitly.')
    started = time.perf_counter()
    model, processor = loader(ocr_language, actual_version)
    result['metadata']['model_load_seconds'] = round(time.perf_counter() - started, 4)
    started = time.perf_counter()
    for index, box in enumerate(result['boxes']):
        crop = result['image'].crop(tuple(box)).convert('RGB')
        line_started = time.perf_counter()
        text = recognizer(crop, ocr_language, model, processor)
        if not isinstance(text, str):
            raise TypeError(f'OCR line {index + 1} returned {type(text).__name__}; expected raw text.')
        result['lines'].append({'index': index + 1, 'text': text, 'box': list(box),
                                'ocr_seconds': round(time.perf_counter() - line_started, 4),
                                'calibrated_confidence': None})
        if not text.strip():
            result['warnings'].append(f'Line {index + 1} returned no text; check its crop.')
    result['text'] = '\n'.join(line['text'] for line in result['lines'])
    result['metadata']['ocr_seconds'] = round(time.perf_counter() - started, 4)
    result['warnings'] = list(dict.fromkeys(result['warnings']))
    return result


In [49]:
# Cell 50 · Rule-aware line segmentation
# No training/downloads. Uses OpenCV already installed by Cell 2.
# Cleanup affects detection masks ONLY: OCR still receives original RGB crops.

import cv2


def _page_mask_components(mask):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype('uint8'), connectivity=8)
    return labels, [(i, *map(int, stats[i])) for i in range(1, count)]


def _page_ink_mask(rgb):
    """Find local ink; suppress long straight rules and page-edge clutter."""
    gray = np.asarray(rgb.convert('L'))
    height, width = gray.shape
    contrast = float(np.percentile(gray, 99.5) - np.percentile(gray, 0.5))
    if int(gray.max()) - int(gray.min()) < 12:
        return np.zeros_like(gray, dtype=bool), dict(threshold=None, contrast=contrast, ink_fraction=0.0)
    window = max(15, (min(height, width) // 20) | 1)
    mask = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV, window, 8)
    # A long straight printed rule is not a text line, regardless of ink colour.
    lines = cv2.HoughLinesP(mask, 1, np.pi / 1800,
                           threshold=max(30, width // 7),
                           minLineLength=max(40, width // 3), maxLineGap=max(5, width // 50))
    rules = np.zeros_like(mask)
    if lines is not None:
        for (x1, y1, x2, y2), in lines:
            dx, dy = int(x2 - x1), int(y2 - y1)
            if abs(dy) < abs(dx) * .15 or abs(dx) < abs(dy) * .12:
                cv2.line(rules, (x1, y1), (x2, y2), 255, max(3, round(min(height, width) * .009)))
    mask[rules != 0] = 0
    labels, components = _page_mask_components(mask)
    clean = np.zeros_like(mask, dtype=bool)
    minimum_area = max(3, round(height * width / 250_000))
    for ident, x, y, w, h, area in components:
        edge = x == 0 or y == 0 or x + w == width or y + h == height
        slender_rule = w > max(30, h * 12) or h > max(40, w * 15)
        if max(w, h) > max(40, min(height, width) * .05):
            yy, xx = np.nonzero(labels[y:y+h, x:x+w] == ident)
            eigen = np.linalg.eigvalsh(np.cov(np.stack((xx, yy)))) if len(xx) > 2 else [0, 0]
            slender_rule |= eigen[0] < max(1.5, min(height, width) * .002) ** 2 and eigen[1] > 200
        large_edge = edge and (w > width * .10 or h > height * .10)
        if area >= minimum_area and w < width * .8 and h < height * .25 and not slender_rule and not large_edge:
            clean[labels == ident] = True
    return clean, dict(threshold='adaptive_rule_suppressed', contrast=contrast,
                       ink_fraction=float(clean.mean()), rules_suppressed=lines is not None)


def _page_projection_boxes(mask):
    """Group component bodies around row-density peaks, retaining dots/descenders."""
    labels, components = _page_mask_components(mask)
    if not components:
        return [], []
    heights = [h for _, x, y, w, h, area in components if h >= 5 and area >= 10]
    if not heights:
        return [], ['Only tiny marks were detected; no reliable text line was found.']
    typical = max(6.0, float(np.percentile(heights, 75)))
    counts = mask.sum(axis=1).astype(np.float32)
    smooth = cv2.GaussianBlur(counts.reshape(-1, 1), (1, 0), typical * .25).ravel()
    candidates = np.flatnonzero((smooth >= np.r_[0, smooth[:-1]]) &
                               (smooth > np.r_[smooth[1:], 0]) &
                               (smooth >= max(2, float(smooth.max()) * .12)))
    peaks = []
    for peak in sorted(candidates, key=lambda i: -smooth[i]):
        if all(abs(int(peak) - p) >= typical * 1.1 for p in peaks):
            peaks.append(int(peak))
    peaks.sort()
    groups = [[] for _ in peaks]
    for component in components:
        ident, x, y, w, h, area = component
        if not peaks:
            break
        center = y + h / 2
        index = min(range(len(peaks)), key=lambda i: abs(center - peaks[i]))
        if abs(center - peaks[index]) <= typical * 2.4:
            groups[index].append(component)
    boxes = []
    for group in groups:
        if not group:
            continue
        bodies = [c for c in group if c[4] >= typical * .4]
        if not bodies:
            continue
        def near_body(mark):
            _, x, y, w, h, _ = mark
            return any(max(0, x - (b[1] + b[3]), b[1] - (x + w)) <= typical * .9
                       for b in bodies)
        group = [c for c in group if c in bodies or near_body(c)]
        # Reject sparse printer dots/noise, but retain an isolated handwritten character.
        if sum(c[-1] for c in group) < max(10, typical * 1.5):
            continue
        left = min(c[1] for c in group); top = min(c[2] for c in group)
        right = max(c[1] + c[3] for c in group); bottom = max(c[2] + c[4] for c in group)
        if bottom - top < typical * .35:
            continue
        boxes.append([left, top, right, bottom])
    return boxes, ['Rule-aware segmentation is heuristic. Check the numbered boxes, especially touching, slanted or very faint writing.']


def _page_pad_boxes(boxes, size):
    # Vertical boxes may overlap for descenders/slanted baselines; don't truncate
    # their ink at a neighbour's bounding edge. We do not alter crop pixels.
    width, height = size
    result = []
    for index, (left, top, right, bottom) in enumerate(boxes):
        py = max(3, min(14, round((bottom - top) * .15)))
        px = max(4, min(24, round((bottom - top) * .25)))
        y0, y1 = max(0, top - py), min(height, bottom + py)
        if index and boxes[index - 1][3] <= top:
            y0 = max(y0, (boxes[index - 1][3] + top) // 2)
        if index + 1 < len(boxes) and bottom <= boxes[index + 1][1]:
            y1 = min(y1, (bottom + boxes[index + 1][1]) // 2)
        result.append([max(0, left - px), y0, min(width, right + px), y1])
    return result


# Segmentation changed: don't reuse a baseline/trained probe from before this cell.
_SIMPLE_PROBE_CACHE = None
print('Rule-aware segmentation enabled. Rerun the demo UI cell; compare one box per handwritten line.')


Rule-aware segmentation enabled. Rerun the demo UI cell; compare one box per handwritten line.


In [50]:
# Cell 51 · Paragraph translation with an explicit target

"""Paragraph translation with a fixed target and review-only safeguards.

Execute after the notebook's existing translation definition cells, or import
and call configure_document_translation(globals()). No model loads occur here.
Mixed input uses cached NLLB with independent source hints and target tokens.
Code-switch quality is unvalidated: warnings are review requests, not repairs.
"""
import re
import time
import unicodedata
from collections import Counter

DOCUMENT_TRANSLATION_POLICY = 'fixed_target_paragraph_review_v1'
_DOCUMENT_TRANSLATION_BACKEND = None
_DOCUMENT_HAN = re.compile(r'[\u3400-\u4dbf\u4e00-\u9fff\U00020000-\U0002FA1F]')
_DOCUMENT_LATIN = re.compile(r'[A-Za-z]')
_DOCUMENT_PARAGRAPH_BREAK = re.compile(r'\r?\n[ \t]*(?:\r?\n[ \t]*)+')
_DOCUMENT_URL = re.compile(r'(?:https?://|www\.)\S+|[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}', re.I)


def configure_document_translation(namespace):
    """Bind existing notebook loaders/decoders without copying their caches."""
    global _DOCUMENT_TRANSLATION_BACKEND
    if not hasattr(namespace, 'get'):
        raise TypeError('Pass the notebook globals() mapping.')
    _DOCUMENT_TRANSLATION_BACKEND = namespace


def _document_backend():
    return globals() if _DOCUMENT_TRANSLATION_BACKEND is None else _DOCUMENT_TRANSLATION_BACKEND


def _document_script_counts(text):
    return {'han': len(_DOCUMENT_HAN.findall(text)), 'latin': len(_DOCUMENT_LATIN.findall(text))}


def _document_source_hint(text, selected_language):
    """A transparent script-count hint, not language or sentence detection."""
    counts = _document_script_counts(text)
    if not counts['han'] and not counts['latin']:
        return selected_language
    return 'zh' if counts['han'] > counts['latin'] else 'en'


def _document_token_count(text, loaded, family, source_hint, namespace):
    tokenizer = loaded['tokenizer']
    if family == 'nllb':
        tokenizer.src_lang = namespace.get('NLLB_LANGUAGES', {'en': 'eng_Latn', 'zh': 'zho_Hans'})[source_hint]
    prepared = text.strip()
    if family == 'marian':
        prepared = namespace.get('MT_ZH_TAG', '>>cmn_Hans<<') + ' ' + prepared
    ids = tokenizer(prepared, truncation=False, add_special_tokens=True)['input_ids']
    if hasattr(ids, 'shape'):
        return int(ids.shape[-1])
    if ids and isinstance(ids[0], (list, tuple)):
        ids = ids[0]
    return len(ids)


def _document_input_limit(loaded, namespace):
    configured = int(namespace.get('MT_BENCH_INPUT_LIMIT', 512))
    model_limit = getattr(loaded['model'].config, 'max_position_embeddings', configured)
    tokenizer_limit = getattr(loaded['tokenizer'], 'model_max_length', configured)
    # HF uses huge sentinel values when tokenizer metadata has no real limit.
    limits = [configured] + [int(n) for n in (model_limit, tokenizer_limit)
                             if isinstance(n, (int, float)) and 0 < n < 1_000_000]
    limit = min(limits)
    if limit < 8:
        raise ValueError('Translation input token limit is too small.')
    return limit


def _document_chunks(text, start, end, count_tokens, token_limit, char_limit=12000):
    """Exact source offsets; prefer sentence/word boundaries and never truncate.

    A character ceiling bounds tokenizer work even on a paragraph with millions
    of characters. Every returned chunk is checked with the actual tokenizer.
    """
    pieces = []
    cursor = start
    while cursor < end:
        upper = min(end, cursor + char_limit)
        candidate = text[cursor:upper]
        if count_tokens(candidate) <= token_limit:
            cut = upper
        else:
            low, high = 1, upper - cursor
            while low < high:
                middle = (low + high + 1) // 2
                if count_tokens(text[cursor:cursor + middle]) <= token_limit:
                    low = middle
                else:
                    high = middle - 1
            cut = cursor + low
            if count_tokens(text[cursor:cut]) > token_limit:
                raise ValueError('A source character exceeds the model input token limit.')
        if cut < end:
            # Prefer a nearby sentence boundary, then whitespace. Do not throw
            # away the rest of the paragraph or assume character/token parity.
            window = text[cursor:cut]
            floor = max(1, len(window) // 2)
            sentence_cuts = [m.end() for m in re.finditer(r'[.!?。！？;；](?:\s+|(?=[\u3400-\u9fff])|$)', window)
                             if m.end() >= floor]
            word_cuts = [m.end() for m in re.finditer(r'\s+', window) if m.end() >= floor]
            preferred = sentence_cuts or word_cuts
            if preferred:
                cut = cursor + preferred[-1]
        # Re-tokenize after boundary selection; token counts need not be
        # perfectly monotonic for all tokenizers.
        while cut > cursor and count_tokens(text[cursor:cut]) > token_limit:
            cut -= 1
        if cut <= cursor:
            raise ValueError('Unable to split input safely within the token limit.')
        pieces.append((cursor, cut))
        cursor = cut
    return pieces


def _document_decode_nllb(text, source_hint, target_language, loaded, beams, namespace):
    """Reuse cached NLLB, independently forcing the user's target language."""
    custom = namespace.get('document_nllb_decoder')
    if callable(custom):
        return custom(text, source_hint, target_language, loaded, beams)
    torch_module = namespace.get('torch')
    if torch_module is None:
        raise RuntimeError('Run the notebook translation definitions first (torch is missing).')
    model, tokenizer = loaded['model'], loaded['tokenizer']
    device = next(model.parameters()).device
    language_ids = namespace.get('NLLB_LANGUAGES', {'en': 'eng_Latn', 'zh': 'zho_Hans'})
    tokenizer.src_lang = language_ids[source_hint]
    target_key = 'zh' if target_language == 'zh-Hans' else 'en'
    target_id = tokenizer.convert_tokens_to_ids(language_ids[target_key])
    if target_id is None or target_id == getattr(tokenizer, 'unk_token_id', None):
        raise RuntimeError('The cached NLLB tokenizer does not contain the required target token.')
    if device.type == 'cuda':
        torch_module.cuda.synchronize(device)
    started = time.perf_counter()
    inputs = tokenizer(text.strip(), return_tensors='pt', truncation=False)
    input_tokens = int(inputs['input_ids'].shape[1])
    if input_tokens > _document_input_limit(loaded, namespace):
        raise ValueError('Input exceeds the token limit after chunking; nothing was truncated.')
    output_limit = int(namespace.get('MT_BENCH_OUTPUT_LIMIT', 256))
    with torch_module.inference_mode():
        generated = model.generate(
            **inputs.to(device), forced_bos_token_id=target_id,
            max_length=None, max_new_tokens=output_limit,
            num_beams=beams, num_return_sequences=1, do_sample=False,
            early_stopping=(beams > 1), length_penalty=1.0, use_cache=True)
    token_ids = generated[0].tolist()
    prediction = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
    if device.type == 'cuda':
        torch_module.cuda.synchronize(device)
    return {'prediction': prediction, 'input_tokens': input_tokens,
            'output_tokens': max(0, len(token_ids) - 1),
            'seconds': time.perf_counter() - started,
            'hit_output_limit': len(token_ids) - 1 >= output_limit}


def _document_review(source, prediction, selected_language, decoded, protected_terms, namespace):
    flags, details = [], {}
    check = namespace.get('mt_check_decoded')
    if callable(check):
        try:
            checked = check(source, decoded, selected_language)
            flags.extend(checked.get('flags', []))
            details.update(checked)
        except Exception as exc:
            # A failed optional review must never turn into a clean result.
            flags.append('review_check_unavailable')
            details['review_check_error'] = f'{type(exc).__name__}: {exc}'
    if not prediction.strip():
        flags.append('empty_output')
    if decoded.get('hit_output_limit'):
        flags.append('output_limit')
    remaining = prediction
    for term in protected_terms:
        remaining = remaining.replace(term, '')
    remaining = _DOCUMENT_URL.sub('', remaining)
    residual = (_DOCUMENT_LATIN.search(remaining) if selected_language == 'en'
                else _DOCUMENT_HAN.search(remaining))
    if residual:
        flags.append('residual_source_script')
    # Exact digit changes can be legitimate (e.g. 12 -> twelve); review, not a
    # semantic assertion. Names cannot reliably be detected by these models.
    numbers = lambda value: Counter(re.findall(r'\d+(?:[.,:/-]\d+)*', unicodedata.normalize('NFKC', value)))
    if numbers(source) != numbers(prediction):
        flags.append('numeric_tokens_changed')
    missing = [term for term in protected_terms if term in source and term not in prediction]
    if missing:
        flags.append('protected_term_changed')
        details['missing_protected_terms'] = missing
    flags = list(dict.fromkeys(flags))
    details.update(flags=flags, flagged=bool(flags))
    return details


def translate_document(text, selected_language, mixed_mode=False, display_punctuation=False,
                       *, keep_ocr=True, protected_terms=()):
    """Translate editable text, retaining raw source and one selected target.

    selected_language='en' always targets zh-Hans; 'zh' always targets en.
    Mixed mode gives NLLB whole paragraphs (bounded by its input budget), with
    a dominant-script source hint independent of the target. Already-target
    paragraphs are preserved. No dictionary edits, model retries, backtranslation
    or silent truncation occur. Names/numbers need human review; protected_terms
    requests an exact-preservation check, not forced substitution.
    """
    if selected_language not in ('en', 'zh'):
        raise ValueError("selected_language must be 'en' or 'zh'.")
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Enter non-empty source text.')
    if isinstance(protected_terms, str):
        raise TypeError('protected_terms must be a collection of complete terms, not one string.')
    terms = tuple(dict.fromkeys(term for term in protected_terms if isinstance(term, str) and term))
    namespace = _document_backend()
    target = 'zh-Hans' if selected_language == 'en' else 'en'
    family = 'nllb' if mixed_mode or selected_language == 'zh' else 'marian'
    loader = namespace.get('_load_route')
    started = time.perf_counter()
    loaded, limit, load_seconds = None, None, 0.0
    beams = int(namespace.get('MT_ROUTE_BEAMS', 4))
    chunks, assembled, warnings, flags = [], [], [], []
    paragraph_start = 0
    spans = []
    for match in _DOCUMENT_PARAGRAPH_BREAK.finditer(text):
        spans.append((paragraph_start, match.start(), match.group()))
        paragraph_start = match.end()
    spans.append((paragraph_start, len(text), ''))
    has_mixed = bool(_DOCUMENT_HAN.search(text) and _DOCUMENT_LATIN.search(text))
    if mixed_mode and has_mixed:
        warnings.append('Mixed English/Chinese translation is experimental. Review meaning, names and numbers against the unchanged source; NLLB is not validated for code-switching.')
        flags.append('mixed_translation_requires_review')
    elif has_mixed:
        warnings.append('Both English and Chinese scripts were found with mixed mode off. Enable mixed mode or review for omitted or untranslated text.')
        flags.append('mixed_input_without_mode')
    # Other alphabetic scripts are outside this two-language OCR/MT workflow.
    if any(c.isalpha() and not _DOCUMENT_HAN.fullmatch(c) and not _DOCUMENT_LATIN.fullmatch(c)
           for c in unicodedata.normalize('NFKC', text)):
        flags.append('unsupported_script')
        warnings.append('Other alphabetic scripts are present; this English/Chinese workflow has not validated them.')
    for paragraph_index, (start, end, separator) in enumerate(spans):
        paragraph = text[start:end]
        if not paragraph.strip():
            assembled.append(paragraph + separator)
            continue
        source_hint = _document_source_hint(paragraph, selected_language) if mixed_mode else selected_language
        counts = _document_script_counts(paragraph)
        already_target = mixed_mode and ((selected_language == 'en' and counts['han'] and not counts['latin'])
                                        or (selected_language == 'zh' and counts['latin'] and not counts['han']))
        has_no_letters = not any(character.isalpha() for character in paragraph)
        if already_target or has_no_letters:
            chunks.append({'index': len(chunks), 'paragraph_index': paragraph_index,
                           'source_start': start, 'source_end': end, 'source_text': paragraph,
                           'source_hint': source_hint, 'target_language': target,
                           'translation': paragraph, 'prediction': paragraph,
                           'input_tokens': 0, 'output_tokens': 0, 'seconds': 0.0,
                           'hit_output_limit': False, 'passthrough': True,
                           'checks': {'flags': [], 'flagged': False}})
            assembled.append(paragraph + separator)
            continue
        if loaded is None:
            if not callable(loader):
                raise RuntimeError('Run the existing translation definitions before translate_document.')
            load_started = time.perf_counter()
            loaded = loader('zh' if family == 'nllb' else 'en', keep_ocr=keep_ocr)
            load_seconds += time.perf_counter() - load_started
            limit = _document_input_limit(loaded, namespace)
        count_tokens = lambda value: _document_token_count(value, loaded, family, source_hint, namespace)
        ranges = _document_chunks(text, start, end, count_tokens, limit)
        if len(ranges) > 1:
            flags.append('paragraph_chunked')
            warnings.append(f'Paragraph {paragraph_index + 1} was split into {len(ranges)} token-safe chunks; cross-chunk context may be lost. All source characters are retained.')
        translated_pieces = []
        for chunk_start, chunk_end in ranges:
            source = text[chunk_start:chunk_end]
            if family == 'nllb':
                decoded = _document_decode_nllb(source, source_hint, target, loaded, beams, namespace)
            else:
                decode = namespace.get('mt_decode_loaded')
                if not callable(decode):
                    raise RuntimeError('Run the existing Marian translation definitions first.')
                decoded = decode(source, 'en', loaded, beams)
            prediction = decoded['prediction']
            if not isinstance(prediction, str):
                raise TypeError('Translation backend returned non-text output.')
            checked = _document_review(source, prediction, selected_language, decoded, terms, namespace)
            flags.extend(checked['flags'])
            chunks.append(dict(decoded, index=len(chunks), paragraph_index=paragraph_index,
                               source_start=chunk_start, source_end=chunk_end,
                               source_text=source, source_hint=source_hint,
                               target_language=target, translation=prediction,
                               passthrough=False, checks=checked))
            translated_pieces.append(prediction)
        # A space separates independently decoded chunks; blank-line paragraph
        # separators survive exactly, while model-generated line breaks are not
        # represented as original OCR layout.
        assembled.append(' '.join(translated_pieces) + separator)
    raw_translation = ''.join(assembled)
    flags = list(dict.fromkeys(flags))
    warnings = list(dict.fromkeys(warnings))
    display_function = namespace.get('mt_display_text')
    display_prediction = (display_function(raw_translation, target, bool(display_punctuation))
                          if callable(display_function) else raw_translation)
    if display_punctuation and not callable(display_function):
        flags.append('display_formatter_unavailable')
        warnings.append('Display punctuation formatter is unavailable; raw translation is shown unchanged.')
    if 'residual_source_script' in flags:
        warnings.append('Source-script text remains in the translation. This may be an untranslated span, name or abbreviation; review it manually.')
    if 'numeric_tokens_changed' in flags:
        warnings.append('Numeric spellings changed. Compare numbers with the source, including legitimate digits-to-words conversions.')
    if 'output_limit' in flags:
        warnings.append('A decoder reached its output cap. Output may be incomplete; shorten that source chunk and translate again.')
    seconds = time.perf_counter() - started - load_seconds
    checks = {'flags': flags, 'flagged': bool(flags),
              'chunks': [chunk['checks'] for chunk in chunks]}
    return {'raw_source': text, 'source_text': text, 'source_language': selected_language,
            'selected_language': selected_language, 'target_language': target,
            'translation': raw_translation, 'prediction': raw_translation,
            'display_prediction': display_prediction, 'chunks': chunks,
            'warnings': warnings, 'checks': checks, 'needs_review': bool(flags),
            'review_status': ('REVIEW NEEDED: ' + ', '.join(flags) if flags else
                              'No heuristic flags; correctness is not guaranteed'),
            'input_tokens': sum(c.get('input_tokens', 0) for c in chunks),
            'output_tokens': sum(c.get('output_tokens', 0) for c in chunks),
            'hit_output_limit': any(c.get('hit_output_limit', False) for c in chunks),
            'output_limit_reached': any(c.get('hit_output_limit', False) for c in chunks),
            'seconds': seconds, 'translation_seconds': seconds, 'load_seconds': load_seconds,
            'model_id': loaded.get('model_id', family) if loaded else 'passthrough',
            'revision': loaded.get('revision') if loaded else None,
            'selected_beams': beams, 'retried': False, 'mixed_mode': bool(mixed_mode),
            'display_punctuation': bool(display_punctuation), 'protected_terms': list(terms),
            'policy': DOCUMENT_TRANSLATION_POLICY}


In [51]:
# Cell 52 · Reviewed vocabulary suggestions

"""Optional, local vocabulary suggestions. Nothing is silently corrected.

Default initialization imports no optional packages and downloads no data.
English support uses pyspellchecker only if explicitly enabled and installed.
Chinese support uses supplied terms; optional jieba supplies token boundaries,
not semantic corrections. Never treat a dictionary miss as proof of an OCR error.
"""
import re

_VOCABULARY_STATE = {'english_checker': None, 'chinese_tokenizer': None,
                     'english_words': set(), 'chinese_terms': set(),
                     'glossary': {}, 'warnings': []}
_VOCABULARY_WORD = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
_VOCABULARY_HAN = re.compile(r'[\u3400-\u4dbf\u4e00-\u9fff]+')
_VOCABULARY_PROTECTED = re.compile(
    r'(?:https?://|www\.)\S+|[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}|'
    r'(?<!\w)(?=[A-Za-z0-9_-]*\d)[A-Za-z0-9]+(?:[_-][A-Za-z0-9]+)*(?!\w)', re.I)


def initialize_vocabulary_helpers(*, enable_english=False, enable_chinese_segmentation=False,
                                  english_words=(), chinese_terms=(), glossary=None):
    """Opt in to installed packages. This function never installs/downloads.

    english_words/chinese_terms are accepted vocabulary, including names.
    glossary maps a source spelling to one or more user-provided candidates.
    Missing optional packages disable that helper and return an explicit warning.
    """
    global _VOCABULARY_STATE
    state = {'english_checker': None, 'chinese_tokenizer': None,
             'english_words': {str(word).casefold() for word in english_words},
             'chinese_terms': {str(term) for term in chinese_terms if term},
             'glossary': dict(glossary or {}), 'warnings': []}
    if enable_english:
        try:
            from spellchecker import SpellChecker
            state['english_checker'] = SpellChecker(language='en', distance=1)
            state['english_checker'].word_frequency.load_words(state['english_words'])
        except ImportError:
            state['warnings'].append('English suggestions are off: optionally install pyspellchecker, then enable them again. No package was installed automatically.')
    if enable_chinese_segmentation:
        try:
            import jieba
            state['chinese_tokenizer'] = jieba.Tokenizer()
            for term in state['chinese_terms']:
                state['chinese_tokenizer'].add_word(term)
        except ImportError:
            state['warnings'].append('Chinese segmentation is off: optionally install jieba, or continue with supplied short Chinese terms only. No package was installed automatically.')
    _VOCABULARY_STATE = state
    return vocabulary_helper_status()


def vocabulary_helper_status():
    state = _VOCABULARY_STATE
    return {'english_enabled': state['english_checker'] is not None,
            'chinese_segmentation_enabled': state['chinese_tokenizer'] is not None,
            'english_words': len(state['english_words']),
            'chinese_terms': len(state['chinese_terms']),
            'glossary_entries': len(state['glossary']),
            'warnings': list(state['warnings']),
            'notice': 'Suggestions need context review. Dictionaries cannot establish OCR correctness; names and technical terms may be valid.'}


def _vocabulary_distance_one(left, right):
    """Bounded edit-distance test; no external lexicon or network involved."""
    if left == right or abs(len(left) - len(right)) > 1:
        return False
    if len(left) == len(right):
        return sum(a != b for a, b in zip(left, right)) == 1
    shorter, longer = sorted((left, right), key=len)
    return any(shorter == longer[:index] + longer[index + 1:] for index in range(len(longer)))


def suggest_transcript_edits(text, selected_language=None, *, user_glossary=None,
                             english_words=(), chinese_terms=(), protected_terms=(),
                             max_suggestions=30, max_candidates=5):
    """Return suggestion records with offsets into the unchanged input.

    Each record has start/end/original/suggestions/language/reason/context and
    requires_review=True. Display records beside editable text; never apply
    automatically. selected_language is a UI hint, not a filter: mixed text can
    receive English and Chinese suggestions together. Default empty dictionaries
    yield an empty list. URL/email/number/identifier spans are protected.
    """
    if not isinstance(text, str):
        raise TypeError('text must be a string.')
    if selected_language not in (None, 'en', 'zh'):
        raise ValueError("selected_language must be None, 'en' or 'zh'.")
    if not isinstance(max_suggestions, int) or not 1 <= max_suggestions <= 200:
        raise ValueError('max_suggestions must be between 1 and 200.')
    if not isinstance(max_candidates, int) or not 1 <= max_candidates <= 20:
        raise ValueError('max_candidates must be between 1 and 20.')
    if any(isinstance(value, str) for value in (english_words, chinese_terms, protected_terms)):
        raise TypeError('Word/term arguments must be collections, not individual strings.')
    state = _VOCABULARY_STATE
    accepted_en = state['english_words'] | {str(word).casefold() for word in english_words}
    accepted_zh = state['chinese_terms'] | {str(term) for term in chinese_terms if term}
    glossary = dict(state['glossary'])
    glossary.update(user_glossary or {})
    blocked = [(match.start(), match.end()) for match in _VOCABULARY_PROTECTED.finditer(text)]
    for term in protected_terms:
        if isinstance(term, str) and term:
            blocked.extend((match.start(), match.end()) for match in re.finditer(re.escape(term), text))
    results, seen = [], set()

    def overlaps(start, end):
        return any(start < blocked_end and end > blocked_start for blocked_start, blocked_end in blocked)

    def add(start, end, candidates, language, reason):
        if len(results) >= max_suggestions or overlaps(start, end) or (start, end) in seen:
            return
        original = text[start:end]
        options = [candidate for candidate in dict.fromkeys(candidates)
                   if isinstance(candidate, str) and candidate and candidate != original][:max_candidates]
        if not options:
            return
        seen.add((start, end))
        results.append({'start': start, 'end': end, 'original': original,
                        'suggestions': options, 'language': language, 'reason': reason,
                        'context': text[max(0, start - 35):min(len(text), end + 35)],
                        'requires_review': True, 'applied': False})

    # Explicit glossary entries can include capitalization or phrases because
    # they came from the user. Still do not touch protected URLs or identifiers.
    for original, candidates in glossary.items():
        if not isinstance(original, str) or not original:
            continue
        if isinstance(candidates, str):
            candidates = [candidates]
        if not isinstance(candidates, (list, tuple, set)):
            continue
        pattern = re.escape(original)
        if original[0].isascii() and original[0].isalnum():
            pattern = r'(?<![A-Za-z0-9_])' + pattern
        if original[-1].isascii() and original[-1].isalnum():
            pattern += r'(?![A-Za-z0-9_])'
        for match in re.finditer(pattern, text):
            add(match.start(), match.end(), candidates,
                'zh' if _VOCABULARY_HAN.search(original) else 'en',
                'User glossary candidate; compare the handwriting and sentence context before editing.')

    checker = state['english_checker']
    for match in _VOCABULARY_WORD.finditer(text):
        word = match.group()
        # Preserve likely names/acronyms and very short fragments. This is
        # deliberately conservative, not a complete named-entity recognizer.
        if (len(word) < 3 or not word.islower() or word.casefold() in accepted_en
                or overlaps(match.start(), match.end())):
            continue
        if checker is not None:
            if not checker.unknown([word]):
                continue
            candidates = checker.candidates(word) or set()
            candidates = sorted(candidates, key=lambda item: (-checker.word_frequency.dictionary.get(item, 0), item))
        else:
            # User vocabulary can help offline even when pyspellchecker is off.
            candidates = sorted(term for term in accepted_en if _vocabulary_distance_one(word, term))
        add(match.start(), match.end(), candidates, 'en',
            'Possible dictionary spelling mismatch, not a confirmed OCR error. Review meaning and names.')

    if accepted_zh:
        tokenizer = state['chinese_tokenizer']
        if tokenizer is None:
            tokens = [(match.group(), match.start(), match.end()) for match in _VOCABULARY_HAN.finditer(text)]
        else:
            tokens = list(tokenizer.tokenize(text))
        for word, start, end in tokens:
            if (not _VOCABULARY_HAN.fullmatch(word) or len(word) < 2 or word in accepted_zh
                    or overlaps(start, end)):
                continue
            # Only compare similar-length complete terms. Without segmentation,
            # long clauses are intentionally skipped instead of guessed apart.
            candidates = sorted(term for term in accepted_zh
                                if len(term) >= 2 and _vocabulary_distance_one(word, term))
            add(start, end, candidates, 'zh',
                'Similar user-supplied Chinese term. Character similarity does not establish the intended word; review in context.')
    return sorted(results, key=lambda item: (item['start'], item['end']))

# Names and domain terms belong in your own glossary; suggestions never auto-apply.
ENABLE_ENGLISH_DICTIONARY = True
ENABLE_CHINESE_SEGMENTATION = True
USER_ENGLISH_TERMS = []
USER_CHINESE_TERMS = []
USER_GLOSSARY = {}  # Example: {'teh': ['the']}; only use corrections you intend.
VOCABULARY_STATUS = initialize_vocabulary_helpers(
    enable_english=ENABLE_ENGLISH_DICTIONARY,
    enable_chinese_segmentation=ENABLE_CHINESE_SEGMENTATION,
    english_words=USER_ENGLISH_TERMS, chinese_terms=USER_CHINESE_TERMS, glossary=USER_GLOSSARY)
print(VOCABULARY_STATUS)
print('English spelling suggestions and Chinese user-term review only; dictionaries cannot guarantee transcription or meaning.')


/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


{'english_enabled': True, 'chinese_segmentation_enabled': True, 'english_words': 0, 'chinese_terms': 0, 'glossary_entries': 0, 'warnings': [], 'notice': 'Suggestions need context review. Dictionaries cannot establish OCR correctness; names and technical terms may be valid.'}
English spelling suggestions and Chinese user-term review only; dictionaries cannot guarantee transcription or meaning.


In [52]:
# Cell 53 · Simple image and live OCR translator
import base64
import hashlib
import io
import threading
import time
import unicodedata
from PIL import Image, ImageDraw, ImageFont

_SIMPLE_LOCK = globals().get('_SIMPLE_LOCK', threading.Lock())
_SIMPLE_PROBE_CACHE = None  # One image only; no captured images are saved to Drive.


def _simple_script(text):
    counts = {'en': 0, 'zh': 0, 'other': 0}
    for c in unicodedata.normalize('NFKC', text):
        cp = ord(c)
        han = any(a <= cp <= b for a, b in (
            (0x3400, 0x4DBF), (0x4E00, 0x9FFF), (0xF900, 0xFAFF),
            (0x20000, 0x2EBEF), (0x30000, 0x323AF)))
        if han:
            counts['zh'] += 1
        elif c.isalpha():
            counts['en' if 'LATIN' in unicodedata.name(c, '') else 'other'] += 1
    kind = ('unsupported' if counts['other'] else
            'mixed' if counts['en'] and counts['zh'] else
            'zh' if counts['zh'] else 'en' if counts['en'] else 'unknown')
    return kind, counts


def _simple_read(crops, language, version):
    # Keep model references inside this function so memory can be released later.
    model, processor = load_ocr(language, version)
    texts = [transcribe(crop, language, model, processor) for crop in crops]
    if any(not isinstance(text, str) for text in texts):
        raise TypeError('OCR returned non-text output.')
    return texts


def _simple_checkpoint(language, choice):
    if choice == 'baseline':
        return 'baseline'
    if language not in TRAINED_READY or 'public' not in model_sources(language):
        raise RuntimeError('Train fresh or restore a completed run first; no silent baseline fallback.')
    return 'public'


def _simple_regions(page):
    """Draw the exact OCR crops on their own resized/deskewed coordinate image."""
    working = page['image'].convert('RGB')
    annotated = working.copy()  # Never draw on the image sent to the recognizer.
    draw = ImageDraw.Draw(annotated)
    width = max(2, round(min(working.size) / 250))
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', max(14, round(working.width / 55)))
    except OSError:
        font = ImageFont.load_default()
    boxes = [list(map(int, box)) for box in page['boxes']]
    for number, (left, top, right, bottom) in enumerate(boxes, 1):
        # PIL rectangles include the last pixel; crop right/bottom are exclusive.
        draw.rectangle((left, top, right - 1, bottom - 1), outline='#e00036', width=width)
        label = str(number)
        bounds = draw.textbbox((0, 0), label, font=font)
        tw, th = bounds[2] - bounds[0], bounds[3] - bounds[1]
        x = min(left, max(0, working.width - tw - 6))
        y = max(0, top - th - 6)
        draw.rectangle((x, y, x + tw + 6, y + th + 6), fill='#e00036')
        draw.text((x + 3 - bounds[0], y + 3 - bounds[1]), label, fill='white', font=font)
    buffer = io.BytesIO()
    annotated.save(buffer, format='PNG')
    return dict(image='data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii'),
                boxes=boxes, width=working.width, height=working.height,
                detector=page.get('metadata', {}).get('detector', 'projection'))


def _simple_probe(image, key, on_page=None):
    global _SIMPLE_PROBE_CACHE
    if _SIMPLE_PROBE_CACHE is not None and _SIMPLE_PROBE_CACHE['key'] == key:
        if on_page is not None:
            on_page(_SIMPLE_PROBE_CACHE['page'])
        return _SIMPLE_PROBE_CACHE
    page = detect_page_lines(image, max_image_pixels=2_000_000,
                             deskew=True, detection_mode='projection')
    if on_page is not None:
        on_page(page)  # Show segmentation even if loading/recognition later fails.
    crops = [page['image'].crop(tuple(box)) for box in page['boxes']]
    probe = _simple_read(crops, 'zh', 'baseline') if crops else []
    kinds = [_simple_script(text)[0] for text in probe]
    # The same baseline probe fixes the route for BOTH model choices.
    # Unknown lines stay with the multilingual recognizer, visibly marked below.
    routes = ['en' if kind == 'en' else 'zh' for kind in kinds]
    result = dict(key=key, page=page, crops=crops, probe=probe,
                  kinds=kinds, routes=routes)
    _SIMPLE_PROBE_CACHE = result
    return result


def simple_ocr_translate(data_url, choice):
    """One callback for uploads and stable live frames. Returns JSON-safe data."""
    if choice not in ('baseline', 'trained'):
        return {'error': 'Choose Baseline or Trained.'}
    # Also guards old callbacks while a rerun or another browser request is active.
    if not _SIMPLE_LOCK.acquire(blocking=False):
        return {'error': 'A request is still finishing. Try again shortly.'}
    transcript = ''
    regions = {}
    def capture_regions(page):
        regions.update(_simple_regions(page))
    try:
        started = time.perf_counter()
        if not isinstance(data_url, str) or not data_url.startswith('data:image/') or ',' not in data_url:
            raise ValueError('Upload an image or start the camera.')
        if len(data_url) > 24_000_000:
            raise ValueError('Use an image smaller than 18 MB.')
        raw = base64.b64decode(data_url.split(',', 1)[1], validate=True)
        with Image.open(io.BytesIO(raw)) as source:
            if source.width * source.height > 30_000_000:
                raise ValueError('Resize the image below 30 megapixels.')
            # EXIF orientation and transparency are handled by detect_page_lines.
            image = source.copy()
        release_routed_mt()  # Do not keep translation + OCR weights resident.
        cached = _simple_probe(image, hashlib.sha256(raw).hexdigest(), capture_regions)
        page, crops = cached['page'], cached['crops']
        warnings = [message.replace('Review the boxes or provide manual boxes.',
                                    'Check the numbered boxes below; retake with separated lines if needed.')
                    .replace('or review manual boxes.', 'with the text upright.')
                    .replace('review all boxes.', 'review the numbered boxes below.')
                    for message in page['warnings']]
        if not crops:
            return dict(text='', translations={}, language='No readable text detected.',
                        model=choice.title(), regions=regions, warnings=warnings, seconds=round(time.perf_counter()-started, 1))
        versions = {lang: _simple_checkpoint(lang, choice) for lang in set(cached['routes'])}
        texts = list(cached['probe'])
        # Identical boxes, preprocessing and language routing for both choices.
        for language in ('en', 'zh'):
            indices = [i for i, route in enumerate(cached['routes']) if route == language]
            if not indices or (language == 'zh' and versions[language] == 'baseline'):
                continue  # Reuse the exact zh/baseline probe, not a second decode.
            decoded = _simple_read([crops[i] for i in indices], language, versions[language])
            for index, text in zip(indices, decoded):
                texts[index] = text
        transcript = '\n'.join(texts)  # Raw OCR line order, without autocorrection.
        kind, counts = _simple_script('\n'.join(cached['probe']))
        labels = {'en': 'English', 'zh': 'Chinese', 'mixed': 'Mixed English/Chinese',
                  'unknown': 'Uncertain / no letters', 'unsupported': 'Unsupported script'}
        language_label = labels[kind] + ' (automatic script estimate)'
        if any(k == 'unknown' for k in cached['kinds']):
            warnings.append('Some lines have no identifiable letters in the baseline probe; they use multilingual Chinese OCR. Their language is uncertain.')
        if counts['en'] + counts['zh'] < 4:
            warnings.append('Very short text: the automatic language estimate is especially uncertain.')
        if choice == 'trained' and 'zh' in versions and 'mixed' in cached['kinds']:
            warnings.append('Chinese-only fine-tuning may reduce English accuracy on mixed lines. Review the raw transcript.')
        if any(not t.strip() for t in texts):
            warnings.append('At least one detected line returned no text; output may be incomplete.')
        if _simple_script(transcript)[0] != kind:
            warnings.append('The selected OCR output differs in script from the shared baseline probe; automatic routing may be wrong or recognition may have lost text.')
        model_label = '; '.join(
            f'{labels[lang]}: {version} ({checkpoint_status(lang, version)})'
            for lang, version in sorted(versions.items()))
        translations = {}
        if not transcript.strip():
            warnings.append('No text was recognized; no translation was run.')
        elif kind in ('unknown', 'unsupported'):
            warnings.append('Translation withheld: the source has no identifiable letters or has unsupported scripts. No target was guessed.')
        elif kind in ('en', 'zh') and _simple_script(transcript)[0] != kind:
            warnings.append('Translation withheld: the OCR script disagrees with the automatic estimate. Raw OCR is retained; no different translation model was silently substituted.')
        elif _simple_script(transcript)[0] == 'unsupported':
            warnings.append('Translation withheld because the selected OCR output contains unsupported scripts.')
        else:
            # Both targets for mixed input. No guessed dominant language or selector.
            directions = ('zh', 'en') if kind == 'mixed' else (kind,)
            mixed = kind == 'mixed'
            release_ocr()
            for direction in directions:
                target = 'en' if direction == 'zh' else 'zh'
                translated = translate_document(transcript, direction,
                                                 mixed_mode=mixed, keep_ocr=False,
                                                 display_punctuation=False)
                translations[target] = translated['translation']
                warnings.extend(translated.get('warnings', []))
                if translated.get('needs_review'):
                    warnings.append(labels[target] + ': ' + translated.get('review_status', 'Translation needs review.'))
            language_label += (' → English + Chinese' if kind == 'mixed' else
                               ' → ' + labels[next(iter(translations))])
        return dict(text=transcript, translations=translations, language=language_label,
                    model=model_label, mixed=(kind == 'mixed'), regions=regions,
                    warnings=list(dict.fromkeys(warnings)),
                    seconds=round(time.perf_counter()-started, 1))
    except Exception as exc:
        # Keep newly recognized text if only translation failed; never reuse an old
        # translation or quietly switch trained output back to the baseline.
        return {'error': f'{type(exc).__name__}: {exc}', 'text': transcript, 'translations': {}, 'regions': regions}
    finally:
        _SIMPLE_LOCK.release()


# Compatibility for the unchanged, mocked page-workflow tests, not UI controls.
def interactive_ocr(data_url, lang, version, compare=False, recognition_mode='selected',
                    detection_mode='projection', boxes_override=None, deskew=True):
    from IPython.display import JSON
    try:
        with Image.open(io.BytesIO(base64.b64decode(data_url.split(',', 1)[1]))) as image:
            page = recognize_page(image.copy(), lang, version, recognition_mode=recognition_mode,
                                  detection_mode=detection_mode, boxes_override=boxes_override,
                                  deskew=deskew, load_ocr_fn=load_ocr, transcribe_fn=transcribe)
        return JSON(dict(lines=page['lines'], actual_version=page['actual_version'],
                         coordinate_frame='working_image_pixels'))
    except Exception as exc:
        return JSON({'error': str(exc)})


def interactive_translate(text, lang, display_punctuation=False, mixed_mode=False):
    from IPython.display import JSON
    try:
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Enter non-empty source text.')
        return JSON({'translation': translate_document(text, lang, mixed_mode=mixed_mode,
                     display_punctuation=display_punctuation, keep_ocr=True)})
    except Exception as exc:
        return JSON({'error': str(exc)})


if SHOW_DEMO:
    required = ('detect_page_lines', 'load_ocr', 'transcribe', 'model_sources',
                'checkpoint_status', 'translate_document', 'release_routed_mt', 'release_ocr')
    missing = [name for name in required if not callable(globals().get(name))]
    if missing:
        raise RuntimeError('Run the setup cells first. Missing: ' + ', '.join(missing))
    from google.colab import output
    from IPython.display import JSON, Javascript, display

    output.register_callback('simple_ocr.translate',
                             lambda data, choice: JSON(simple_ocr_translate(data, choice)))
    display(Javascript(r"""
    (() => {
      if (window.__simpleOCRDemo) window.__simpleOCRDemo.dispose();
      const root = document.createElement('div');
      root.style.cssText = 'max-width:720px;padding:18px;border:1px solid #ddd;border-radius:10px;font:15px/1.5 Arial,sans-serif';
      root.innerHTML = `
        <h3 style="margin-top:0">English–Chinese OCR translator</h3>
        <label>Model <select class="model"><option value="baseline">Baseline</option><option value="trained">Trained</option></select></label>
        <p>English → Chinese · Chinese → English.<br>Mixed text → literal transcription + English translation + Chinese translation.</p>
        <p><label><b>1. Upload an image</b><br><input class="upload" type="file" accept="image/*"></label></p>
        <p><b>2. Live OCR translation</b><br><button class="start">Start live</button> <button class="stop" disabled>Stop</button></p>
        <video class="video" autoplay muted playsinline style="display:none;max-width:100%;max-height:320px"></video>
        <img class="preview" alt="Uploaded image" style="display:none;max-width:100%;max-height:240px">
        <p class="status" role="status" aria-live="polite">Upload a page or start the camera.</p>
        <p class="details"></p>
        <figure class="regions" hidden style="margin:12px 0">
          <b>Detected line regions · OCR order</b>
          <img class="boxes-image" alt="Numbered line boxes on the exact processed OCR frame" style="display:block;max-width:100%;height:auto">
          <figcaption class="boxes-caption" style="font-size:13px"></figcaption>
        </figure>
        <label><b>Literal transcription (raw OCR)</b><textarea class="text" readonly rows="5" style="display:block;box-sizing:border-box;width:100%"></textarea></label>
        <label class="english-field" hidden><b>English translation</b><textarea class="english" readonly rows="5" style="display:block;box-sizing:border-box;width:100%"></textarea></label>
        <label class="chinese-field" hidden><b>Chinese translation</b><textarea class="chinese" readonly rows="5" style="display:block;box-sizing:border-box;width:100%"></textarea></label>
        <p class="warnings" style="white-space:pre-wrap;color:#805500"></p>
        <small>Automatic detection is a best-effort English/Chinese script estimate, not calibrated confidence. Use clear, horizontal, single-column pages on light paper (up to 24 detected lines). Live mode reads stable changed frames, not every video frame. First use may download model weights. Mixed handwriting and translation need review. A demo does not prove training improved accuracy; use the held-out evaluation.</small>`;
      element.appendChild(root);
      const get = name => root.querySelector('.' + name);
      let stream = null, running = false, busy = false, disposed = false, epoch = 0;
      let lastImage = null, previous = null, accepted = null, steady = 0, timer = null;
      const video = get('video'), canvas = document.createElement('canvas');
      const probeCanvas = document.createElement('canvas');
      probeCanvas.width = 128; probeCanvas.height = 96;
      const probeContext = probeCanvas.getContext('2d', {willReadFrequently:true});
      const context = canvas.getContext('2d');
      function controls() {
        get('model').disabled = busy;
        get('upload').disabled = busy || running;
        get('start').disabled = busy || running;
        get('stop').disabled = !running;
      }
      function clearResults() {
        get('regions').hidden = true; get('boxes-image').removeAttribute('src');
        get('boxes-caption').textContent = '';
        for (const name of ['text','english','chinese']) get(name).value = '';
        get('details').textContent = ''; get('warnings').textContent = '';
        get('english-field').hidden = true; get('chinese-field').hidden = true;
      }
      function stop(message = 'Live camera stopped.') {
        epoch++; running = false; clearTimeout(timer); timer = null;
        if (stream) stream.getTracks().forEach(track => track.stop());
        stream = null; video.srcObject = null; video.style.display = 'none';
        previous = null; accepted = null; steady = 0;
        if (!disposed) { get('status').textContent = message; controls(); }
      }
      async function request(data, ticket) {
        if (busy || disposed) return;
        busy = true; controls(); clearResults();
        get('status').textContent = 'Reading and translating… First use can take longer.';
        try {
          const response = await google.colab.kernel.invokeFunction('simple_ocr.translate', [data, get('model').value], {});
          if (disposed || ticket !== epoch) return;
          const result = response.data['application/json'];
          if (!result || typeof result !== 'object') throw new Error('No response from the notebook. Check that the runtime is connected.');
          // This is a frozen processed frame, NOT boxes on the moving camera feed.
          const regions = result.regions;
          if (regions && regions.image && Array.isArray(regions.boxes)) {
            get('boxes-image').src = regions.image;
            get('regions').hidden = false;
            get('boxes-caption').textContent = `${regions.boxes.length} detected line region(s). Numbers match raw OCR line order. ` +
              `Exact processed snapshot (${regions.width} × ${regions.height}), after orientation, resizing and deskew. ` +
              'Red boxes are the crops sent to OCR, not confidence scores. ' +
              'Detection uses projection-based segmentation, not a learned line detector. ' +
              (regions.boxes.length ? 'A box spanning two lines indicates merged lines; several boxes on one line indicate splitting.' : 'No line regions were detected.');
          }
          get('text').value = result.text || '';
          const translations = result.translations || {};
          get('english-field').hidden = !Object.hasOwn(translations, 'en');
          get('chinese-field').hidden = !Object.hasOwn(translations, 'zh');
          const displayTranslation = (target, source) => {
          if (!result.mixed || !Array.isArray(regions?.lines)) {
              return translations[target] || '';
          }

          return regions.lines
            .filter(line => line.script === source || line.script === 'mixed')
            .map(line => {
                const text = (line.translations[target] || '').replace(/[\r\n]+/g, ' ').trim();
                return `[${line.region_id}] ${text || '(empty translation)'}`;
              })
              .join('\n');
          };
          get('english').value = displayTranslation('en', 'zh');
          get('chinese').value = displayTranslation('zh', 'en');
          if (result.error) {
            if (running) stop('Live mode paused after an error.');
            get('status').textContent = result.error;
            return;
          }
          get('details').textContent = [result.language, result.model].filter(Boolean).join(' · ');
          get('warnings').textContent = (result.warnings || []).join('\n');
          get('status').textContent = `Done in ${result.seconds}s.` + (running ? ' Move to new text and hold still.' : '');
        } catch (error) {
          if (!disposed && ticket === epoch) {
            if (running) stop('Live mode paused after an error.');
            get('status').textContent = error.message || String(error);
          }
        } finally { busy = false; if (!disposed) controls(); }
      }
      function difference(a, b) {
        if (!a || !b) return Infinity;
        // Tile differences notice a newly written word that a page-wide mean can miss.
        let total = 0; const tiles = new Float32Array(48);
        for (let i = 0; i < a.length; i += 4) {
          const delta = (Math.abs(a[i]-b[i]) + Math.abs(a[i+1]-b[i+1]) + Math.abs(a[i+2]-b[i+2])) / 3;
          const pixel = i / 4, tile = Math.floor((pixel % 128) / 16) + 8 * Math.floor(pixel / 128 / 16);
          total += delta; tiles[tile] += delta;
        }
        return Math.max(total / (a.length / 4), Math.max(...tiles) / 256 * 0.75);
      }
      async function tick(ticket) {
        if (disposed || !running || ticket !== epoch) return;
        try {
          if (!busy && video.readyState >= 2 && video.videoWidth) {
            probeContext.drawImage(video, 0, 0, 128, 96);
            const pixels = probeContext.getImageData(0, 0, 128, 96).data.slice();
            steady = difference(pixels, previous) < 4 ? steady + 1 : 0;
            previous = pixels;
            if (steady >= 2 && difference(pixels, accepted) > 2.5) {
              const scale = Math.min(1, 1400 / Math.max(video.videoWidth, video.videoHeight));
              canvas.width = Math.max(2, Math.round(video.videoWidth * scale));
              canvas.height = Math.max(2, Math.round(video.videoHeight * scale));
              context.drawImage(video, 0, 0, canvas.width, canvas.height);
              lastImage = canvas.toDataURL('image/jpeg', 0.90); accepted = pixels; steady = 0;
              await request(lastImage, ticket);
            }
          }
        } catch (error) { if (ticket === epoch) stop(error.message || String(error)); }
        if (!disposed && running && ticket === epoch) timer = setTimeout(() => tick(ticket), 600);
      }
      get('upload').onchange = async event => {
        const file = event.target.files[0]; if (!file || busy) return;
        stop('Loading image…'); clearResults(); lastImage = null; get('preview').style.display = 'none';
        if (file.size > 18_000_000) { get('status').textContent = 'Use an image smaller than 18 MB.'; return; }
        const ticket = epoch;
        const reader = new FileReader();
        reader.onerror = () => { if (ticket === epoch) get('status').textContent = 'Could not read this image.'; };
        reader.onload = async () => {
          if (ticket !== epoch || disposed) return;
          lastImage = reader.result; get('preview').src = lastImage; get('preview').style.display = 'block';
          await request(lastImage, ticket);
        };
        reader.readAsDataURL(file);
      };
      get('model').onchange = async () => {
        if (busy) return;
        clearResults();
        // Use the SAME captured image when comparing models, even in live mode.
        if (lastImage) await request(lastImage, epoch);
      };
      get('start').onclick = async () => {
        if (busy || running) return;
        running = true; const ticket = ++epoch; controls(); clearResults();
        get('preview').style.display = 'none'; get('status').textContent = 'Allow camera access, then hold the page still.';
        lastImage = null; previous = null; accepted = null; steady = 0;
        try {
          const acquired = await navigator.mediaDevices.getUserMedia({video:{facingMode:{ideal:'environment'}}, audio:false});
          if (disposed || ticket !== epoch) { acquired.getTracks().forEach(track => track.stop()); return; }
          stream = acquired; video.srcObject = stream; video.style.display = 'block';
          await video.play();
          if (ticket !== epoch || disposed) return;
          timer = setTimeout(() => tick(ticket), 600);
        } catch (error) { if (ticket === epoch) stop('Camera could not start: ' + (error.message || error)); }
      };
      get('stop').onclick = () => stop();
      const dispose = () => { disposed = true; stop(); };
      window.__simpleOCRDemo = {dispose};
      window.addEventListener('pagehide', dispose, {once:true});
      const observer = new MutationObserver(() => {
        if (!root.isConnected) { dispose(); observer.disconnect(); }
      });
      observer.observe(document.body, {childList:true, subtree:true});
      controls();
    })();
    """))


<IPython.core.display.Javascript object>

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/544 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

... (33 outputs left out)

Ready on cuda; loading took 4.07s


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/544 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [53]:
# Cell 54 · Mixed-page OCR translation
# Run immediately after Cell 53.

import re


def _simple_translate_regions(lines, source, target, warnings):
    """Build one translation column, keeping one displayed row per OCR box."""
    display_lines = []
    target_name = 'English' if target == 'en' else 'Chinese'

    for line in lines:
        text = line['raw_text']
        script = line['script']
        prediction = text
        flags = []

        if script in (source, 'mixed'):
            # Give the existing translation checks identifiers such as 482Z.
            identifiers = [
                word for word in re.findall(r'[A-Za-z0-9]+', text)
                if not word.isalpha() and not word.isdigit()
            ]

            result = translate_document(
                text,
                source,
                mixed_mode=(script == 'mixed'),
                keep_ocr=False,
                protected_terms=identifiers,
            )
            prediction = result['translation']
            flags = list(result['checks']['flags'])

            if script == 'mixed' and 'mixed_translation_requires_review' not in flags:
                flags.append('mixed_translation_requires_review')

            if flags:
                reasons = ', '.join(
                    'experimental mixed-line translation'
                    if flag == 'mixed_translation_requires_review'
                    else flag.replace('_', ' ')
                    for flag in flags
                )
                warnings.append(f"Box {line['region_id']} → {target_name} — {reasons}.")

            # A model-generated newline still belongs to this one box.
            # The exact decoder output is retained in line['translations'] below.
            display_lines.append(' '.join(prediction.splitlines()))

        else:
            # Already in the target language, or contains only numbers/punctuation.
            # An empty OCR box also keeps its original position.
            display_lines.append(text)
            if not text.strip():
                flags = ['empty_ocr']

        line['translations'][target] = prediction
        line['flags'][target] = flags

    return '\n'.join(display_lines)


def simple_ocr_translate(data_url, choice):
    """Process an uploaded image or a stable camera frame."""
    if choice not in ('baseline', 'trained'):
        return {'error': 'Choose Baseline or Trained.'}

    if not _SIMPLE_LOCK.acquire(blocking=False):
        return {'error': 'A request is still finishing. Try again shortly.'}

    transcript = ''
    regions = {'lines': []}

    def capture_regions(page):
        regions.update(_simple_regions(page))

    try:
        started = time.perf_counter()

        if not isinstance(data_url, str) or not data_url.startswith('data:image/') or ',' not in data_url:
            raise ValueError('Upload an image or start the camera.')
        if len(data_url) > 24_000_000:
            raise ValueError('Use an image smaller than 18 MB.')

        raw = base64.b64decode(data_url.split(',', 1)[1], validate=True)
        with Image.open(io.BytesIO(raw)) as source_image:
            if source_image.width * source_image.height > 30_000_000:
                raise ValueError('Resize the image below 30 megapixels.')
            image = source_image.copy()

        release_routed_mt()
        cached = _simple_probe(image, hashlib.sha256(raw).hexdigest(), capture_regions)
        page, crops = cached['page'], cached['crops']

        warnings = [
            message.replace(
                'Review the boxes or provide manual boxes.',
                'Check the numbered boxes below and retake with separated lines if needed.',
            ).replace(
                'or review manual boxes.', 'with the text upright.'
            ).replace(
                'review all boxes.', 'review the numbered boxes below.'
            )
            for message in page['warnings']
        ]

        if not crops:
            return dict(
                text='', translations={}, language='No readable text detected.',
                model=choice.title(), regions=regions, warnings=warnings,
                seconds=round(time.perf_counter() - started, 1),
            )

        # Baseline and Trained use the same boxes and the same OCR routing probe.
        versions = {
            lang: _simple_checkpoint(lang, choice)
            for lang in set(cached['routes'])
        }
        texts = list(cached['probe'])

        for language in ('en', 'zh'):
            indices = [i for i, route in enumerate(cached['routes']) if route == language]
            if not indices or (language == 'zh' and versions[language] == 'baseline'):
                continue

            decoded = _simple_read([crops[i] for i in indices], language, versions[language])
            for index, text in zip(indices, decoded):
                texts[index] = text

        transcript = '\n'.join(texts)
        regions['lines'] = [
            dict(
                region_id=number, box=box, raw_text=text,
                script=_simple_script(text)[0], translations={}, flags={},
            )
            for number, (box, text) in enumerate(zip(regions['boxes'], texts), start=1)
        ]

        probe_kind, counts = _simple_script('\n'.join(cached['probe']))
        text_kind = _simple_script(transcript)[0]
        kind = 'mixed' if text_kind == 'mixed' and probe_kind in ('en', 'zh') else probe_kind

        labels = {
            'en': 'English', 'zh': 'Chinese', 'mixed': 'Mixed English/Chinese',
            'unknown': 'Uncertain / no letters', 'unsupported': 'Unsupported script',
        }
        language_label = labels[kind] + ' (automatic script estimate)'

        if 'unknown' in cached['kinds']:
            warnings.append('Some boxes have no identifiable letters in the baseline OCR probe.')
        if counts['en'] + counts['zh'] < 4:
            warnings.append('Very short text. The automatic language estimate is uncertain.')
        if choice == 'trained' and 'zh' in versions and 'mixed' in cached['kinds']:
            warnings.append('Chinese-only fine-tuning may reduce English accuracy on mixed lines.')
        if text_kind != probe_kind:
            warnings.append('The selected OCR text differs in script from the shared baseline probe.')

        for line in regions['lines']:
            if not line['raw_text'].strip():
                warnings.append(
                    f"Box {line['region_id']} produced no OCR text. Check its crop in the numbered image."
                )

        model_label = '; '.join(
            f'{labels[lang]}: {version} ({checkpoint_status(lang, version)})'
            for lang, version in sorted(versions.items())
        )

        translations = {}
        if not transcript.strip():
            warnings.append('No text was recognized.')
        elif kind in ('unknown', 'unsupported'):
            warnings.append('Translation withheld because the source language could not be identified.')
        elif kind in ('en', 'zh') and text_kind != kind:
            warnings.append('Translation withheld because the OCR script disagrees with the probe.')
        elif text_kind == 'unsupported':
            warnings.append('Translation withheld because the selected OCR contains an unsupported script.')
        else:
            release_ocr()

            if kind == 'mixed':
                for source, target in (('zh', 'en'), ('en', 'zh')):
                    translations[target] = _simple_translate_regions(
                        regions['lines'], source, target, warnings
                    )
                language_label += ' → English + Chinese'

            else:
                # A single-language page keeps the existing paragraph translation.
                target = 'zh' if kind == 'en' else 'en'
                translated = translate_document(
                    transcript, kind, mixed_mode=False,
                    keep_ocr=False, display_punctuation=False,
                )
                translations[target] = translated['translation']
                warnings.extend(translated.get('warnings', []))
                if translated.get('needs_review'):
                    warnings.append(
                        labels[target] + ': ' +
                        translated.get('review_status', 'Translation needs review.')
                    )
                language_label += ' → ' + labels[target]

        return dict(
            text=transcript, translations=translations, language=language_label,
            model=model_label, mixed=(kind == 'mixed'), regions=regions,
            warnings=list(dict.fromkeys(warnings)),
            seconds=round(time.perf_counter() - started, 1),
        )

    except Exception as exc:
        return {
            'error': f'{type(exc).__name__}: {exc}',
            'text': transcript, 'translations': {}, 'regions': regions,
        }
    finally:
        _SIMPLE_LOCK.release()

In [54]:
# Cell 55· Page workflow smoke tests

# These checks use synthetic images and mocks, never model weights or the camera.
from unittest.mock import patch
from PIL import Image, ImageDraw
import io, base64


def test_page_workflow():
    image = Image.new('RGB', (320, 180), 'white')
    drawing = ImageDraw.Draw(image)
    for y in (35, 105):
        for x in range(25, 270, 25): drawing.rectangle((x, y, x+12, y+15), fill='black')
    detected = detect_page_lines(image, deskew=False)
    assert len(detected['boxes']) == 2
    raw_calls=[]
    def loader(lang,version): raw_calls.append((lang,version)); return object(),object()
    def reader(crop,lang,model,processor): return 'Hello 中文' if crop.width else ''
    page = recognize_page(image, 'en', 'public', recognition_mode='mixed', deskew=False,
        load_ocr_fn=loader, transcribe_fn=reader)
    assert raw_calls == [('zh','baseline')]
    assert page['text'] == 'Hello 中文\nHello 中文'
    reverse = list(reversed(detected['boxes']))
    manual = detect_page_lines(detected['image'], boxes_override=reverse, deskew=False)
    assert manual['boxes'] == reverse
    blank = recognize_page(Image.new('RGB',(200,100),'white'),'en','baseline',
        load_ocr_fn=lambda *a: (_ for _ in ()).throw(AssertionError('Blank image loaded model')),
        transcribe_fn=reader)
    assert not blank['lines']
    text='teh sentence'
    suggestions=suggest_transcript_edits(text,user_glossary={'teh':['the']})
    assert text=='teh sentence' and suggestions[0]['applied'] is False
    if SHOW_DEMO:
        buffer=io.BytesIO();image.save(buffer,format='PNG')
        data='data:image/png;base64,'+base64.b64encode(buffer.getvalue()).decode()
        def translation(text,selected_language,**options):
            return dict(source_text=text,source_language=selected_language,
                target_language='zh-Hans' if selected_language=='en' else 'en',
                translation='测试',prediction='测试',display_prediction='测试',
                review_status='Mock only',needs_review=False)
        with patch.dict(globals(),{'load_ocr':loader,'transcribe':reader,'translate_document':translation}):
            result=interactive_ocr(data,'en','baseline',False,'mixed','projection',None,False).data
            assert len(result['lines'])==2 and result['actual_version']=='baseline'
            assert result['coordinate_frame']=='working_image_pixels'
            translated=interactive_translate('Hello 中文','en',False,True).data['translation']
            assert translated['target_language']=='zh-Hans'
            assert 'error' in interactive_translate('', 'en').data
    print('PASS: page order, mixed OCR routing, blank-page guard, manual boxes, reviewed suggestions and callback targets. No real models/camera used.')


test_page_workflow()


PASS: page order, mixed OCR routing, blank-page guard, manual boxes, reviewed suggestions and callback targets. No real models/camera used.


In [55]:
# Cell 56 · Mixed-page translation regression
# Run after Cell 54.

def test_mixed_page_translation():
    import base64
    import io
    from unittest.mock import patch
    from PIL import Image

    zh_order = '我的订单号号码是4827'
    en_order = 'My order number is 482Z'
    zh_thanks = '谢谢您的帮助祝您今天愉快'
    en_thanks = 'Thank you for your help. Have a nice day.'
    mixed_line = '请说 hello'

    raw = [zh_order, en_order, zh_thanks, '', zh_thanks, mixed_line]

    expected = {
        'en': ['My order number is 4827', en_order, en_thanks, '', en_thanks, 'Please say hello.'],
        'zh': [zh_order, '我的订单号是482Z', zh_thanks, '', zh_thanks, '请说你好。'],
    }
    answers = {
        ('en', zh_order): expected['en'][0],
        ('en', zh_thanks): en_thanks,
        ('zh', en_order): expected['zh'][1],
        ('en', mixed_line): 'Please say hello.',
        ('zh', mixed_line): '请说你好。',
    }
    decoder_calls = []

    def decode(text, target):
        decoder_calls.append((target, text))
        return {'prediction': answers.get((target, text), '[unexpected decoder input]')}

    # The first pixel identifies each synthetic crop to the mock OCR reader.
    width, row_height = 96, 20
    image = Image.new('RGB', (width, row_height * len(raw)), 'white')
    boxes = [(0, i * row_height, width, (i + 1) * row_height) for i in range(len(raw))]

    for i, box in enumerate(boxes):
        image.putpixel((0, box[1]), (i, 0, 0))

    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    data_url = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()

    with patch.dict(globals(), {
        'detect_page_lines': lambda img, **kw: {'image': img, 'boxes': boxes, 'warnings': []},
        '_simple_read': lambda crops, *args: [raw[c.getpixel((0, 0))[0]] for c in crops],
        '_SIMPLE_PROBE_CACHE': None,
        'checkpoint_status': lambda *args: 'fixture',
        'release_ocr': lambda: None,
        'release_routed_mt': lambda: None,
        '_DOCUMENT_TRANSLATION_BACKEND': None,
        '_load_route': lambda *args, **kw: {'model_id': 'fixture'},
        '_document_input_limit': lambda *args: 512,
        '_document_token_count': lambda text, *args: len(text) + 2,
        '_document_decode_nllb': lambda text, hint, target, *args:
            decode(text, 'zh' if target == 'zh-Hans' else target),
        'mt_decode_loaded': lambda text, *args: decode(text, 'zh'),
        'mt_check_decoded': None,
    }):
        result = simple_ocr_translate(data_url, 'baseline')
        clean_calls = decoder_calls.copy()

        # Deliberately introduce errors to check the box-specific warnings.
        answers.update({
            ('en', zh_order): '订单 4829',       # Chinese remains, number changed
            ('zh', en_order): '我的订单号是482X',  # Identifier changed
            ('en', zh_thanks): '',              # Translation disappeared
        })
        reviewed = simple_ocr_translate(data_url, 'baseline')

    assert 'error' not in result, result.get('error')
    assert 'error' not in reviewed, reviewed.get('error')
    assert result['text'].split('\n') == raw, 'Raw OCR text or box order changed.'
    assert len(result['regions']['boxes']) == len(raw)

    for target, text in clean_calls:
        print(f'{target} decoder received {text!r}')
    assert all(call in answers for call in clean_calls), 'Unexpected text reached a decoder.'

    records = result['regions']['lines']
    for target, expected_rows in expected.items():
        actual_rows = result['translations'][target].split('\n')
        assert actual_rows == expected_rows, (target, actual_rows)
        assert 'empty_ocr' in records[3]['flags'][target]
        assert 'mixed_translation_requires_review' in records[5]['flags'][target]

    for i, line in enumerate(records):
        assert (line['region_id'], tuple(line['box']), line['raw_text']) == (i + 1, boxes[i], raw[i])
        for target in expected:
            assert line['translations'][target] == expected[target][i]

    flagged = reviewed['regions']['lines']
    for box_number, target, expected_flag in [
        (1, 'en', 'numeric_tokens_changed'),
        (1, 'en', 'residual_source_script'),
        (2, 'zh', 'protected_term_changed'),
        (3, 'en', 'empty_output'),
        (5, 'en', 'empty_output'),
    ]:
        assert expected_flag in flagged[box_number - 1]['flags'][target], (
            box_number, target, expected_flag
        )

    print('PASS — both targets retain all six box positions, both order IDs, and the repeated closing.')
    print('PASS — the affected boxes flag empty text, mixed scripts, and translation errors.')


test_mixed_page_translation()

en decoder received '我的订单号号码是4827'
en decoder received '谢谢您的帮助祝您今天愉快'
en decoder received '谢谢您的帮助祝您今天愉快'
en decoder received '请说 hello'
zh decoder received 'My order number is 482Z'
zh decoder received '请说 hello'
PASS — both targets retain all six box positions, both order IDs, and the repeated closing.
PASS — the affected boxes flag empty text, mixed scripts, and translation errors.
